# SSL Grokking: Complete Research Compendium
## Self-Supervised Algorithmic Generalization via JEPA on Modular Arithmetic

---

## Table of Contents

| Part | Title | Key Question |
|------|-------|-------------|
| 1 | Core Discovery: Self-Supervised Grokking | Can JEPA grok modular addition without labels or Fourier circuits? |
| 2 | Is It Truly Self-Supervised? | Does removing labels from the target encoder kill grokking? |
| 3 | Strong Inductive Bias via Augmentation | Can augmentations replace explicit labels? |
| 4 | Prediction Accuracy & Memorization Dynamics | Does the JEPA objective itself generalize, or just the representation? |
| 5 | Target Encoder Interventions & Predictor Ablation | Which components are necessary for grokking, and when? |
| 6 | Function Approximation, Causal Intervention & Task Variation | What function did the model learn? Does it generalize to other tasks? |
| 7 | Internal Fourier Audit | Is Fourier structure hiding in internal components? |

---

# Part 1: Core Discovery: Self-Supervised Grokking

**Description:** The original JEPA grokking result: shows JEPA can grok modular addition without Fourier circuits

---

# 🧠 Self-Supervised Grokking: Algorithmic Generalization Without Labels

##JEPA Discovers Modular Arithmetic Without Supervision or Fourier Circuits

---

### What is this notebook about?

**Grokking** is a phenomenon where neural networks suddenly generalize long after memorizing training data. Since Power et al. (2022) first described it, every study has assumed:
1. A **supervised** cross-entropy loss (the model is told the correct answer)
2. Generalization coincides with **Fourier circuit formation** (Nanda et al., 2023)

**This notebook challenges both assumptions.**

We demonstrate that a **Joint Embedding Predictive Architecture (JEPA)** — a self-supervised model that is *never told the correct answer* — can grok modular addition, reaching **100% validation accuracy**. More remarkably, it does so **without developing any Fourier structure**, suggesting an entirely different generalization pathway than what supervised models use.

### Key Findings

| Finding | Description |
|---------|-------------|
| **Self-supervised grokking exists** | First demonstration of delayed generalization under a JEPA objective |
| **No Fourier circuits in target encoder** | Target encoder stays flat; context encoder develops mild concentration during memorization then *loses* it during grokking |
| **Expanding dimensionality** | Effective rank *increases* during grokking (opposite of supervised dynamics) |
| **Delayed phase transition** | Classic grokking signature: train 100% at epoch ~1K, val 100% at epoch ~41K |

### Architecture Overview

```
Context Encoder: (a, b) → z_context    [Learns to represent input pairs]
Predictor:       z_context → ẑ_target   [Predicts target latent]
Target Encoder:  c = (a+b)%p → z_target [EMA-updated, encodes the answer]

Loss: Negative cosine similarity between ẑ_target and z_target
Eval: Linear probe on frozen context encoder latents
```

The model **never receives classification labels**. It only learns to predict latent representations.

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import time
import json
import os
from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.decomposition import PCA

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
PyTorch: 2.10.0+cu128
GPU: NVIDIA L4


## 2. Hyperparameters

These are deliberately chosen to match the standard grokking setup as closely as possible, so that differences in behavior can be attributed to the **loss function** (JEPA vs supervised), not the architecture or optimization.

In [ ]:
# Task
p = 97                    # Prime modulus
train_frac = 0.3          # 30% train split (standard for grokking)

# Architecture
LATENT_DIM = 128          # Latent representation dimension
HIDDEN_DIM = 256          # Hidden layer width
PREDICTOR_DIM = 64        # Predictor bottleneck dimension

# Training
EPOCHS = 60_000            # Generous budget — grokking onset varies by seed (20k-60k)
LR = 1e-3                 # Learning rate
WEIGHT_DECAY = 1.0        # High weight decay (standard for grokking)
EMA_DECAY = 0.996         # Target encoder EMA momentum

# Logging
LOG_EVERY = 200           # Log geometric metrics
PROBE_EVERY = 500         # Run linear probe (more expensive)

print(f"Task: (a + b) mod {p}")
print(f"Total pairs: {p**2}, Train: {int(train_frac * p**2)}, Val: {p**2 - int(train_frac * p**2)}")
print(f"Latent dim: {LATENT_DIM}, Hidden: {HIDDEN_DIM}, Predictor: {PREDICTOR_DIM}")

Task: (a + b) mod 97
Total pairs: 9409, Train: 2822, Val: 6587
Latent dim: 128, Hidden: 256, Predictor: 64


## 3. Dataset: Modular Addition

We generate all pairs $(a, b)$ where $0 \leq a, b < 97$ and compute $c = (a + b) \bmod 97$. We train on 30% of pairs and evaluate on the remaining 70%.

In [ ]:
# Generate full dataset
pairs = torch.cartesian_prod(torch.arange(p), torch.arange(p))
targets = (pairs[:, 0] + pairs[:, 1]) % p

n = len(pairs)
n_train = int(train_frac * n)
n_val = n - n_train

# Fixed split for reproducibility
perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:n_train], perm[n_train:]

train_pairs = pairs[train_idx].to(device)
train_targets = targets[train_idx].to(device)
val_pairs = pairs[val_idx].to(device)
val_targets = targets[val_idx].to(device)

# Full-batch training (standard for grokking experiments)
train_loader = DataLoader(
    TensorDataset(train_pairs, train_targets),
    batch_size=n_train, shuffle=True
)

print(f"Dataset: {n} total pairs")
print(f"Train: {n_train} ({train_frac*100:.0f}%) | Val: {n_val} ({(1-train_frac)*100:.0f}%)")
print(f"\nSample: ({train_pairs[0,0].item()}, {train_pairs[0,1].item()}) → {train_targets[0].item()}")
print(f"Check:  ({train_pairs[0,0].item()} + {train_pairs[0,1].item()}) mod {p} = {(train_pairs[0,0].item() + train_pairs[0,1].item()) % p}")

Dataset: 9409 total pairs
Train: 2822 (30%) | Val: 6587 (70%)

Sample: (93, 18) → 14
Check:  (93 + 18) mod 97 = 14


## 4. Architecture

### Why JEPA for modular arithmetic?

In standard grokking, the model directly classifies: given $(a, b)$, predict class $c$ via cross-entropy. The loss function **explicitly tells the model the answer**.

In our JEPA setup:
- The **Context Encoder** maps the input pair $(a, b)$ to a latent vector
- The **Target Encoder** maps the answer $c$ to a latent vector (updated via EMA, not gradient)
- The **Predictor** maps the context latent to the predicted target latent
- The loss is **cosine similarity** between predicted and actual target latents

The model is never told "the answer is 42". It only learns: "whatever the target encoder produces for the answer, the context encoder's prediction should match it in latent space."

Both encoders are free to choose *any* representation — they could collude on arbitrary codes. The question is whether weight decay and the JEPA dynamics force algorithmic structure to emerge anyway.

In [ ]:
class ContextEncoder(nn.Module):
    """Encodes (a, b) pair into latent space.

    Architecture matches standard grokking models:
    embedding → concatenation → MLP → L2 normalization.
    """
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)                    # (batch, 2, hidden_dim)
        e = e.view(e.size(0), -1)          # (batch, 2 * hidden_dim)
        z = self.net(e)
        return F.normalize(z, dim=-1)      # Project onto unit hypersphere


class TargetEncoder(nn.Module):
    """Encodes the answer c = (a+b)%p into latent space.

    Simpler architecture since it only processes a single token.
    Updated via EMA, not gradient descent.
    """
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class Predictor(nn.Module):
    """Maps context latent → predicted target latent.

    This asymmetric predictor is crucial for JEPA —
    without it, the model collapses to trivial solutions.
    """
    def __init__(self, latent_dim, predictor_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, latent_dim),
        )

    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)


# Count parameters
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Quick check
_ctx = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM)
_tgt = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM)
_pred = Predictor(LATENT_DIM, PREDICTOR_DIM)
print(f"Context Encoder: {count_params(_ctx):,} params")
print(f"Target Encoder:  {count_params(_tgt):,} params")
print(f"Predictor:       {count_params(_pred):,} params")
print(f"Total:           {count_params(_ctx) + count_params(_tgt) + count_params(_pred):,} params")
del _ctx, _tgt, _pred

Context Encoder: 254,848 params
Target Encoder:  123,520 params
Predictor:       20,736 params
Total:           399,104 params


## 5. Geometric & Fourier Metrics

We track several properties of the learned representations during training:

- **Effective Rank** (Participation Ratio): How many dimensions the representation actually uses. Computed as $\left(\sum_i \lambda_i\right)^2 / \sum_i \lambda_i^2$ from eigenvalues of the covariance matrix.

- **Top Eigenvalue Ratio**: Fraction of total variance captured by the largest eigenvalue. High values indicate low-rank structure.

- **Fourier Top-5 Concentration**: For a model that has learned modular arithmetic, the target encoder's embeddings (indexed by residue 0..96) should show strong peaks in the DFT at specific frequencies. A flat spectrum means no Fourier structure.

- **Uniformity**: Measures how uniformly distributed the latent representations are on the hypersphere (Wang & Isola, 2020). Lower = more uniform.

- **Linear Probe Accuracy**: A closed-form ridge regression on frozen encoder features → classification accuracy. Measures how much task-relevant information is linearly accessible in the latent space.

In [ ]:
@torch.no_grad()
def compute_geometry(z):
    """Compute geometric properties of latent representations."""
    z_c = z - z.mean(dim=0, keepdim=True)
    cov = (z_c.T @ z_c) / (z.shape[0] - 1)
    eigvals = torch.linalg.eigvalsh(cov).clamp(min=1e-10)

    # Participation ratio (effective rank)
    normed = eigvals / eigvals.sum()
    eff_rank = (1.0 / (normed ** 2).sum()).item()

    # Top eigenvalue concentration
    top_ratio = (eigvals[-1] / eigvals.sum()).item()

    # Uniformity (Wang & Isola, 2020) — subsample for speed on large N
    if z.shape[0] > 2048:
        idx = torch.randperm(z.shape[0])[:2048]
        z_sub = z[idx]
    else:
        z_sub = z
    sq_pdist = torch.cdist(z_sub, z_sub, p=2).pow(2)
    uniformity = torch.log(torch.exp(-2 * sq_pdist).mean() + 1e-10).item()

    return {
        "effective_rank": eff_rank,
        "top_eig_ratio": top_ratio,
        "uniformity": uniformity,
    }


@torch.no_grad()
def compute_fourier_structure(encoder, p, is_context=False, input_data=None, target_data=None):
    """Check if encoder embeddings have Fourier structure.

    For the target encoder: directly encode residues 0..p-1.
    For the context encoder: average latents per target residue class.

    Returns (top5_ratio, energy_per_freq).
    """
    if is_context and input_data is not None:
        # Average context encoder output per residue class
        z_all = encoder(input_data)
        residue_means = torch.zeros(p, z_all.shape[1], device=z_all.device)
        counts = torch.zeros(p, device=z_all.device)
        for i in range(p):
            mask = target_data == i
            if mask.sum() > 0:
                residue_means[i] = z_all[mask].mean(dim=0)
                counts[i] = mask.sum()
        z = residue_means
    else:
        indices = torch.arange(p, device=device)
        z = encoder(indices)

    z_np = z.cpu().numpy()
    fft_mag = np.abs(np.fft.fft(z_np, axis=0))
    energy = (fft_mag ** 2).sum(axis=1)
    energy[0] = 0  # Remove DC component

    total = energy.sum()
    if total < 1e-12:
        return 0.0, energy

    top5_ratio = np.sort(energy)[-5:].sum() / total
    return top5_ratio, energy


@torch.no_grad()
def linear_probe_accuracy(encoder, val_pairs, val_targets, train_pairs, train_targets, p):
    """Closed-form linear probe on frozen encoder features.

    Uses ridge regression: W = (Z^T Z + λI)^{-1} Z^T Y
    This is fast and deterministic — no SGD needed.
    """
    z_train = encoder(train_pairs)
    z_val = encoder(val_pairs)
    y_train = F.one_hot(train_targets, num_classes=p).float()

    reg = 1e-3
    ZtZ = z_train.T @ z_train + reg * torch.eye(z_train.shape[1], device=device)
    ZtY = z_train.T @ y_train
    W = torch.linalg.solve(ZtZ, ZtY)

    train_acc = (z_train @ W).argmax(dim=1).eq(train_targets).float().mean().item()
    val_acc = (z_val @ W).argmax(dim=1).eq(val_targets).float().mean().item()
    return train_acc, val_acc


@torch.no_grad()
def ema_update(online, target, decay):
    """Exponential moving average update for target encoder."""
    for o_param, t_param in zip(online.parameters(), target.parameters()):
        t_param.data.mul_(decay).add_(o_param.data, alpha=1 - decay)


print("Metrics functions defined.")

Metrics functions defined.


## 6. Training Loop

The training procedure follows standard JEPA:
1. Context encoder processes $(a, b)$ → latent $z_{\text{ctx}}$
2. Predictor maps $z_{\text{ctx}}$ → predicted target latent $\hat{z}_{\text{tgt}}$
3. Target encoder (EMA, no gradient) processes $c$ → actual target latent $z_{\text{tgt}}$
4. Loss = negative cosine similarity between $\hat{z}_{\text{tgt}}$ and $z_{\text{tgt}}$
5. Target encoder updated via EMA after each step

**⏱️ Runtime: ~50-80 minutes on T4 GPU for 100K epochs**

In [ ]:
# Initialize models
context_enc = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
target_enc = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
predictor = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)

# EMA copy of target encoder
target_enc_ema = deepcopy(target_enc)
for param in target_enc_ema.parameters():
    param.requires_grad = False

# Compile models for faster training
context_enc = torch.compile(context_enc)
target_enc = torch.compile(target_enc)
predictor = torch.compile(predictor)
target_enc_ema = torch.compile(target_enc_ema)

# Optimizer (AdamW with high weight decay — same as supervised grokking)
optimizer = optim.AdamW(
    list(context_enc.parameters()) +
    list(predictor.parameters()) +
    list(target_enc.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)

# History tracking
history = {
    "epoch": [], "jepa_loss": [],
    "train_acc": [], "val_acc": [],
    "eff_rank": [], "top_eig_ratio": [],
    "uniformity": [], "fourier_top5_target": [],
    "fourier_top5_context": [],
}

print("Models initialized. Starting training...\n")
start_time = time.time()

Models initialized. Starting training...



In [ ]:
for epoch in range(EPOCHS):
    context_enc.train()
    target_enc.train()
    predictor.train()

    for batch_pairs, batch_targets in train_loader:
        optimizer.zero_grad()

        z_context = context_enc(batch_pairs)
        z_pred = predictor(z_context)

        with torch.no_grad():
            z_target = target_enc_ema(batch_targets)

        # JEPA loss: negative cosine similarity
        loss = -(z_pred * z_target).sum(dim=-1).mean()

        loss.backward()
        optimizer.step()

        # EMA update target encoder
        ema_update(target_enc, target_enc_ema, EMA_DECAY)

    # --- Logging ---
    if epoch % LOG_EVERY == 0 or epoch == EPOCHS - 1:
        context_enc.eval()
        target_enc_ema.eval()

        with torch.no_grad():
            z_p = predictor(context_enc(train_pairs))
            z_t = target_enc_ema(train_targets)
            jepa_loss = -(z_p * z_t).sum(dim=-1).mean().item()

        # Geometry
        with torch.no_grad():
            z_full = context_enc(pairs.to(device))
            geo = compute_geometry(z_full)

        # Fourier structure (both encoders)
        f_top5_tgt, _ = compute_fourier_structure(target_enc_ema, p)
        f_top5_ctx, _ = compute_fourier_structure(
            context_enc, p, is_context=True,
            input_data=pairs.to(device),
            target_data=targets.to(device)
        )

        history["epoch"].append(epoch)
        history["jepa_loss"].append(jepa_loss)
        history["eff_rank"].append(geo["effective_rank"])
        history["top_eig_ratio"].append(geo["top_eig_ratio"])
        history["uniformity"].append(geo["uniformity"])
        history["fourier_top5_target"].append(f_top5_tgt)
        history["fourier_top5_context"].append(f_top5_ctx)

        # Linear probe (less frequent)
        if epoch % PROBE_EVERY == 0 or epoch == EPOCHS - 1:
            train_acc, val_acc = linear_probe_accuracy(
                context_enc, val_pairs, val_targets,
                train_pairs, train_targets, p
            )
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)

            elapsed = time.time() - start_time
            print(
                f"Epoch {epoch:5d} ({elapsed/60:.1f}m) | "
                f"Loss: {jepa_loss:.4f} | "
                f"Train: {train_acc*100:.1f}% | Val: {val_acc*100:.1f}% | "
                f"EffRank: {geo['effective_rank']:.1f} | "
                f"Fourier(tgt): {f_top5_tgt:.3f} | "
                f"Fourier(ctx): {f_top5_ctx:.3f}"
            )
        else:
            history["train_acc"].append(None)
            history["val_acc"].append(None)

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time/60:.1f} minutes.")
print(f"Final Val Accuracy: {[v for v in history['val_acc'] if v is not None][-1]*100:.1f}%")

Epoch     0 (0.0m) | Loss: -0.0100 | Train: 22.0% | Val: 0.0% | EffRank: 48.4 | Fourier(tgt): 0.071 | Fourier(ctx): 0.079
Epoch  1000 (0.6m) | Loss: -0.9351 | Train: 100.0% | Val: 0.3% | EffRank: 12.7 | Fourier(tgt): 0.071 | Fourier(ctx): 0.132
Epoch  2000 (1.1m) | Loss: -0.9039 | Train: 99.9% | Val: 0.3% | EffRank: 14.1 | Fourier(tgt): 0.071 | Fourier(ctx): 0.152
Epoch  3000 (1.7m) | Loss: -0.8762 | Train: 99.9% | Val: 0.2% | EffRank: 14.6 | Fourier(tgt): 0.071 | Fourier(ctx): 0.173
Epoch  4000 (2.3m) | Loss: -0.8741 | Train: 99.9% | Val: 0.2% | EffRank: 14.7 | Fourier(tgt): 0.071 | Fourier(ctx): 0.204
Epoch  5000 (2.8m) | Loss: -0.9098 | Train: 99.7% | Val: 0.3% | EffRank: 14.8 | Fourier(tgt): 0.071 | Fourier(ctx): 0.239
Epoch  6000 (3.4m) | Loss: -0.8806 | Train: 99.7% | Val: 0.3% | EffRank: 14.8 | Fourier(tgt): 0.071 | Fourier(ctx): 0.264
Epoch  7000 (3.9m) | Loss: -0.8887 | Train: 99.9% | Val: 0.4% | EffRank: 14.9 | Fourier(tgt): 0.071 | Fourier(ctx): 0.291
Epoch  8000 (4.5m) | Lo

In [ ]:
# Save checkpoint
checkpoint = {
    "context_enc": context_enc.state_dict(),
    "target_enc_ema": target_enc_ema.state_dict(),
    "predictor": predictor.state_dict(),
    "history": history,
    "config": {
        "p": p, "latent_dim": LATENT_DIM, "hidden_dim": HIDDEN_DIM,
        "predictor_dim": PREDICTOR_DIM, "epochs": EPOCHS,
        "lr": LR, "weight_decay": WEIGHT_DECAY, "ema_decay": EMA_DECAY,
        "seed": SEED,
    }
}
torch.save(checkpoint, "jepa_grokking_checkpoint.pt")
print("Checkpoint saved.")

## 7. Result 1: The Grokking Curve

The classic grokking signature: training accuracy saturates almost immediately, while validation accuracy stays near chance for thousands of epochs before suddenly jumping.

In [ ]:
# Extract probe epochs (where accuracy was measured)
probe_epochs = [e for e, a in zip(history["epoch"], history["train_acc"]) if a is not None]
train_accs = [a for a in history["train_acc"] if a is not None]
val_accs = [a for a in history["val_acc"] if a is not None]

# Auto-detect grokking transition region
val_arr = np.array(val_accs)
ep_arr = np.array(probe_epochs)
grok_start_idx = np.argmax(val_arr > 0.05)  # First time > 5%
grok_end_idx = np.argmax(val_arr > 0.95)    # First time > 95%
grok_start_ep = ep_arr[grok_start_idx] if grok_start_idx > 0 else 0
grok_end_ep = ep_arr[grok_end_idx] if grok_end_idx > 0 else EPOCHS

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(probe_epochs, [a * 100 for a in train_accs], label="Train Accuracy",
        color="#2563eb", linewidth=2)
ax.plot(probe_epochs, [a * 100 for a in val_accs], label="Validation Accuracy",
        color="#f97316", linewidth=2)
ax.axhline(y=100, color="red", linestyle="--", alpha=0.3, label="Perfect accuracy")
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5, label=f"Chance level (1/{p})")

# Mark the grokking region (auto-detected)
ax.axvspan(grok_start_ep, grok_end_ep, alpha=0.08, color="green", label="Grokking transition")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("Self-Supervised Grokking: JEPA on Modular Addition (mod 97)", fontsize=14, fontweight="bold")
ax.legend(fontsize=10, loc="center right")
ax.set_ylim(-5, 110)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("grokking_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Grokking transition detected: epoch {grok_start_ep:,} → {grok_end_ep:,}")
print(f"Train saturated early, val stayed near chance, then sudden climb — classic grokking.")

## 7b. Phase Transition Dynamics: Rate of Change

The grokking transition has an exponential character. Plotting the rate of change of validation accuracy reveals a sharp spike — the hallmark of a phase transition.

In [ ]:
# Rate of change of validation accuracy
val_arr_roc = np.array(val_accs) * 100
ep_arr_roc = np.array(probe_epochs)

# Compute Δ(val_acc) / Δ(epoch), normalized to "% per 1000 epochs"
rate_of_change = np.diff(val_arr_roc) / np.diff(ep_arr_roc) * 1000
ep_midpoints = (ep_arr_roc[:-1] + ep_arr_roc[1:]) / 2

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Rate of change
axes[0].plot(ep_midpoints, rate_of_change, color="#dc2626", linewidth=1.5)
axes[0].fill_between(ep_midpoints, rate_of_change, alpha=0.15, color="#dc2626")
peak_idx = np.argmax(rate_of_change)
axes[0].annotate(f"Peak: {rate_of_change[peak_idx]:.1f}%/1k ep\nat epoch {int(ep_midpoints[peak_idx]):,}",
                 xy=(ep_midpoints[peak_idx], rate_of_change[peak_idx]),
                 xytext=(ep_midpoints[peak_idx] + 5000, rate_of_change[peak_idx] * 0.8),
                 arrowprops=dict(arrowstyle="->", color="black"),
                 fontsize=10, fontweight="bold")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Δ Val Accuracy (% per 1k epochs)", fontsize=12)
axes[0].set_title("Grokking Rate of Change", fontsize=13, fontweight="bold")
axes[0].grid(True, alpha=0.3)

# Right: Log-scale val accuracy to show exponential onset
val_nonzero = np.maximum(val_arr_roc, 0.1)  # Floor for log scale
axes[1].semilogy(ep_arr_roc, val_nonzero, color="#f97316", linewidth=2)
axes[1].axhline(y=100, color="red", linestyle="--", alpha=0.3)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Val Accuracy (%, log scale)", fontsize=12)
axes[1].set_title("Log-Scale View: Exponential Onset", fontsize=13, fontweight="bold")
axes[1].grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.savefig("phase_transition_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Peak rate of change: {rate_of_change[peak_idx]:.1f}% per 1000 epochs")
print("The sharp spike confirms a phase-transition-like character.")
print("Log-scale view reveals exponential growth before the transition saturates.")

## 8. Result 2: Fourier Structure Analysis

In **every** previous study of grokking on modular addition (Nanda et al., 2023; Zhong et al., 2023; Power et al., 2022), generalization was accompanied by the formation of Fourier circuits — the model's embeddings learned circular representations at specific frequencies.

Here, we track the Fourier energy concentration of **both** encoders throughout training. Note the asymmetry: the target encoder remains flat, while the context encoder shows a striking rise-then-fall pattern.

In [ ]:
ep = history["epoch"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Target encoder Fourier
axes[0].plot(ep, history["fourier_top5_target"], color="purple", linewidth=1.5)
axes[0].axhline(y=5/p, color="gray", linestyle=":", alpha=0.7, label=f"Uniform baseline (5/{p})")
axes[0].set_title("Target Encoder: Fourier Top-5 Concentration", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Energy fraction in top 5 frequencies")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Context encoder Fourier
axes[1].plot(ep, history["fourier_top5_context"], color="teal", linewidth=1.5)
axes[1].axhline(y=5/p, color="gray", linestyle=":", alpha=0.7, label=f"Uniform baseline (5/{p})")
axes[1].set_title("Context Encoder: Fourier Top-5 Concentration", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Energy fraction in top 5 frequencies")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fourier_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Target encoder: FLAT Fourier spectrum throughout training (locked at ~0.071).")
print("Context encoder: rises during memorization, then DECREASES during grokking.")
print("The context encoder actively dismantles partial spectral structure while generalizing.")
print("This anti-correlation between Fourier concentration and generalization is novel.")

In [ ]:
# Final Fourier spectrum of both encoders
_, final_spectrum_tgt = compute_fourier_structure(target_enc_ema, p)
_, final_spectrum_ctx = compute_fourier_structure(
    context_enc, p, is_context=True,
    input_data=pairs.to(device),
    target_data=targets.to(device)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].bar(range(p), final_spectrum_tgt, color="steelblue", alpha=0.8)
axes[0].set_title("Target Encoder — Final Fourier Spectrum", fontweight="bold")
axes[0].set_xlabel("Frequency index k")
axes[0].set_ylabel("Energy")
axes[0].grid(True, alpha=0.3, axis="y")

axes[1].bar(range(p), final_spectrum_ctx, color="coral", alpha=0.8)
axes[1].set_title("Context Encoder — Final Fourier Spectrum", fontweight="bold")
axes[1].set_xlabel("Frequency index k")
axes[1].set_ylabel("Energy")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("fourier_spectra_final.png", dpi=150, bbox_inches="tight")
plt.show()
print("Both spectra are essentially flat — no dominant frequencies.")
print("A supervised grokking model would show sharp spikes at specific frequencies.")

## 9. Result 3: Geometric Dynamics During Grokking

Tracking how the latent space geometry evolves reveals another surprising difference from supervised grokking.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Effective Rank
axes[0].plot(ep, history["eff_rank"], color="green", linewidth=1.5)
axes[0].set_title("Effective Rank (Participation Ratio)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Effective Rank")
axes[0].grid(True, alpha=0.3)
axes[0].annotate("Rank INCREASES\nduring grokking",
                 xy=(35000, 16.5), fontsize=10, color="green",
                 fontweight="bold", ha="center")

# Top Eigenvalue Ratio
axes[1].plot(ep, history["top_eig_ratio"], color="red", linewidth=1.5)
axes[1].set_title("Top Eigenvalue Concentration", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("λ_max / Σλ")
axes[1].grid(True, alpha=0.3)

# Uniformity
axes[2].plot(ep, history["uniformity"], color="teal", linewidth=1.5)
axes[2].set_title("Uniformity (lower = more uniform)", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("log(avg exp(-2||z_i - z_j||²))")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("geometric_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Key observation: Effective rank INCREASES during grokking (~14.8 → ~17.8).")
print("In supervised grokking, representations typically COMPRESS.")
print("The JEPA model EXPANDS its representational dimensionality while generalizing.")

## 9b. Result 4: Training Loss is Blind to Grokking

In supervised grokking, the training loss drops to zero early and stays there — so the loss trivially cannot signal the generalization transition.

In our JEPA setup, the loss is **not** at its minimum. The model actively fights weight decay and the EMA target keeps shifting, so the optimizer is doing real work throughout training. Yet during the massive internal reorganization from memorization to generalization, **the JEPA loss barely changes**.

This means the memorizing solution and the generalizing solution occupy nearly the same loss basin — they have similar JEPA loss but radically different generalization properties.

In [ ]:
# JEPA Loss vs Validation Accuracy overlay
fig, ax1 = plt.subplots(figsize=(14, 5))

# Loss on left axis
color_loss = "#8b5cf6"
ax1.plot(ep, history["jepa_loss"], color=color_loss, linewidth=1, alpha=0.7, label="JEPA Loss")
ax1.set_xlabel("Epoch", fontsize=12)
ax1.set_ylabel("JEPA Loss (neg cosine sim)", fontsize=12, color=color_loss)
ax1.tick_params(axis="y", labelcolor=color_loss)

# Val accuracy on right axis
ax2 = ax1.twinx()
color_val = "#f97316"
val_ep_clean = [e for e, a in zip(probe_epochs, val_accs) if a is not None]
val_acc_clean = [a*100 for a in val_accs if a is not None]
ax2.plot(val_ep_clean, val_acc_clean, color=color_val, linewidth=2, label="Val Accuracy")
ax2.set_ylabel("Validation Accuracy (%)", fontsize=12, color=color_val)
ax2.tick_params(axis="y", labelcolor=color_val)

# Highlight grokking region
ax1.axvspan(grok_start_ep, grok_end_ep, alpha=0.08, color="green")

# Annotations
ax1.set_title("The JEPA Loss is Blind to Grokking", fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3)

# Find loss values at grokking boundaries
grok_start_loss_idx = min(range(len(ep)), key=lambda i: abs(ep[i] - grok_start_ep))
grok_end_loss_idx = min(range(len(ep)), key=lambda i: abs(ep[i] - grok_end_ep))
loss_at_start = history["jepa_loss"][grok_start_loss_idx]
loss_at_end = history["jepa_loss"][grok_end_loss_idx]

fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95), fontsize=10)
plt.tight_layout()
plt.savefig("loss_blind_to_grokking.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"During grokking transition (epoch {grok_start_ep:,} → {grok_end_ep:,}):")
print(f"  JEPA loss: {loss_at_start:.4f} → {loss_at_end:.4f} (Δ = {abs(loss_at_end - loss_at_start):.4f})")
print(f"  Val accuracy: ~{val_accs[grok_start_idx]*100:.1f}% → ~{val_accs[grok_end_idx]*100:.1f}%")
print(f"\nThe loss changes by ~{abs(loss_at_end - loss_at_start)*100:.1f}% while accuracy changes by ~{abs(val_accs[grok_end_idx] - val_accs[grok_start_idx])*100:.0f}%.")
print("Monitoring loss alone would give ZERO warning of the internal reorganization.")

## 10. Mechanistic Analysis: What Did the Model Learn?

If the model didn't learn Fourier circuits, what algorithm did it discover? We investigate the structure of the learned representations.

In [ ]:
# Extract representations for all pairs
context_enc.eval()
target_enc_ema.eval()

with torch.no_grad():
    all_pairs = pairs.to(device)
    all_targets = targets.to(device)

    z_ctx_all = context_enc(all_pairs).cpu().numpy()
    z_tgt_all = target_enc_ema(all_targets).cpu().numpy()

    # Target encoder: one embedding per residue
    z_residues = target_enc_ema(torch.arange(p, device=device)).cpu().numpy()

targets_np = all_targets.cpu().numpy()
pairs_np = all_pairs.cpu().numpy()

print(f"Context encoder latents: {z_ctx_all.shape}")
print(f"Target encoder residue embeddings: {z_residues.shape}")

In [ ]:
# PCA of context encoder representations, colored by target residue
pca = PCA(n_components=2)
z_ctx_2d = pca.fit_transform(z_ctx_all)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Color by target (answer)
scatter1 = axes[0].scatter(z_ctx_2d[:, 0], z_ctx_2d[:, 1],
                           c=targets_np, cmap="hsv", s=1, alpha=0.3)
axes[0].set_title("Context Encoder Latents (PCA)\nColored by target c = (a+b) mod 97",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.colorbar(scatter1, ax=axes[0], label="Target residue")

# Color by first input a
scatter2 = axes[1].scatter(z_ctx_2d[:, 0], z_ctx_2d[:, 1],
                           c=pairs_np[:, 0], cmap="hsv", s=1, alpha=0.3)
axes[1].set_title("Context Encoder Latents (PCA)\nColored by first input a",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.colorbar(scatter2, ax=axes[1], label="Input a")

plt.tight_layout()
plt.savefig("pca_context_encoder.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# PCA of target encoder residue embeddings
pca_tgt = PCA(n_components=2)
z_res_2d = pca_tgt.fit_transform(z_residues)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot colored by residue index
scatter = axes[0].scatter(z_res_2d[:, 0], z_res_2d[:, 1],
                          c=np.arange(p), cmap="hsv", s=50, edgecolors="black", linewidth=0.5)
axes[0].set_title("Target Encoder Residue Embeddings (PCA)\nColored by residue 0..96",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel(f"PC1 ({pca_tgt.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca_tgt.explained_variance_ratio_[1]*100:.1f}%)")
plt.colorbar(scatter, ax=axes[0], label="Residue index")
axes[0].grid(True, alpha=0.3)

# Connect sequential residues with lines to check for circular structure
axes[1].plot(z_res_2d[:, 0], z_res_2d[:, 1], '-', alpha=0.3, color='gray', linewidth=0.5)
scatter2 = axes[1].scatter(z_res_2d[:, 0], z_res_2d[:, 1],
                           c=np.arange(p), cmap="hsv", s=50, edgecolors="black", linewidth=0.5)
axes[1].set_title("Target Encoder: Sequential Residue Path\n(Lines connect consecutive residues)",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel(f"PC1 ({pca_tgt.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca_tgt.explained_variance_ratio_[1]*100:.1f}%)")
plt.colorbar(scatter2, ax=axes[1], label="Residue index")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("pca_target_encoder.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"PCA explained variance (2 components): {pca_tgt.explained_variance_ratio_.sum()*100:.1f}%")
print("If the model learned Fourier structure, residues would form a circle.")
print("If not, the structure reveals the alternative algorithm.")

In [ ]:
# PCA variance explained — how many dimensions does the structure live in?
pca_full = PCA(n_components=min(50, LATENT_DIM))
pca_full.fit(z_residues)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(pca_full.explained_variance_ratio_)),
            pca_full.explained_variance_ratio_ * 100,
            color="steelblue", alpha=0.8)
axes[0].set_title("Target Encoder: PCA Variance per Component", fontweight="bold")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Variance Explained (%)")
axes[0].grid(True, alpha=0.3, axis="y")

cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
axes[1].plot(range(len(cumvar)), cumvar, 'o-', color="steelblue", markersize=4)
axes[1].axhline(y=90, color="red", linestyle="--", alpha=0.5, label="90% threshold")
axes[1].axhline(y=95, color="orange", linestyle="--", alpha=0.5, label="95% threshold")
n_90 = int(np.argmax(cumvar >= 90) + 1) if np.any(cumvar >= 90) else len(cumvar)
n_95 = int(np.argmax(cumvar >= 95) + 1) if np.any(cumvar >= 95) else len(cumvar)
axes[1].set_title("Target Encoder: Cumulative Variance Explained", fontweight="bold")
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Variance (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("pca_variance.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")
print(f"\nA Fourier solution would concentrate variance in ~2-4 components.")
print(f"A distributed representation would spread across many.")

## 11. Cosine Similarity Analysis

Investigate the structure of pairwise relationships learned by the target encoder.

In [ ]:
# Pairwise cosine similarity between all residue embeddings
z_res_t = torch.from_numpy(z_residues)
cos_sim = (z_res_t @ z_res_t.T).numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full similarity matrix
im = axes[0].imshow(cos_sim, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
axes[0].set_title("Target Encoder: Cosine Similarity Matrix\n(Residues 0..96)",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Residue j")
axes[0].set_ylabel("Residue i")
plt.colorbar(im, ax=axes[0], label="Cosine similarity")

# Similarity as function of distance: sim(i, (i+d) % p) for each d
avg_sim_by_dist = np.zeros(p)
for d in range(p):
    sims = [cos_sim[i, (i + d) % p] for i in range(p)]
    avg_sim_by_dist[d] = np.mean(sims)

axes[1].plot(range(p), avg_sim_by_dist, 'o-', markersize=3, color="steelblue")
axes[1].set_title("Average Cosine Similarity by Modular Distance\nsim(i, (i+d) mod 97)",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel("Modular distance d")
axes[1].set_ylabel("Average cosine similarity")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("cosine_similarity_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("If the model learned circular structure, similarity would be a smooth")
print("function of modular distance (cosine wave). If not, the pattern reveals")
print("what relational structure the model DID learn.")

## 11b. Deep Mechanistic Analysis

The previous sections showed *what* the model learned (no Fourier structure, distributed representation, near-orthogonal target codes). Now we investigate *how* — what algorithm did the model actually discover?

We run five targeted tests:
1. **Additive decomposition**: Does `z(a,b) ≈ f(a) + g(b)`?
2. **Predictor linearity**: Is the predictor approximately a linear map?
3. **Class-mean geometry**: How do predicted class means align with target codes?
4. **Embedding arithmetic**: Does shifting `a` by Δ produce consistent directions?
5. **Information probes**: What quantities are linearly decodable from the context latents?

In [ ]:
# === Analysis 1: Additive Decomposition Test ===
# If context encoder learned z(a,b) ≈ f(a) + g(b), then the representation
# has an additive structure — the MLP "adds" embedding contributions.
# We test this via variance decomposition (like two-way ANOVA).

context_enc.eval()
with torch.no_grad():
    z_matrix = torch.zeros(p, p, LATENT_DIM)
    for a in range(p):
        pairs_a = torch.stack([torch.full((p,), a), torch.arange(p)], dim=1).to(device)
        z_matrix[a] = context_enc(pairs_a).cpu()

    # Row means = f(a), Column means = g(b), Grand mean
    row_means = z_matrix.mean(dim=1)   # (p, latent_dim)
    col_means = z_matrix.mean(dim=0)   # (p, latent_dim)
    grand_mean = z_matrix.mean(dim=(0, 1))

    # Additive reconstruction: z_hat(a,b) = f(a) + g(b) - grand_mean
    z_additive = row_means.unsqueeze(1) + col_means.unsqueeze(0) - grand_mean
    residual = z_matrix - z_additive

    # Variance decomposition
    total_var = z_matrix.var().item()
    additive_var = z_additive.var().item()
    residual_var = residual.var().item()

    # Cosine similarity between actual and additive reconstruction
    z_flat = z_matrix.reshape(-1, LATENT_DIM)
    z_add_flat = F.normalize(z_additive.reshape(-1, LATENT_DIM), dim=-1)
    cos_sims = (z_flat * z_add_flat).sum(dim=-1)

    print(f"=== Additive Decomposition Test ===")
    print(f"Total variance:    {total_var:.6f}")
    print(f"Additive variance: {additive_var:.6f} ({additive_var/total_var*100:.1f}%)")
    print(f"Residual variance: {residual_var:.6f} ({residual_var/total_var*100:.1f}%)")
    print(f"\nCosine sim (actual vs additive reconstruction):")
    print(f"  Mean: {cos_sims.mean():.4f}, Std: {cos_sims.std():.4f}, Min: {cos_sims.min():.4f}")
    if cos_sims.mean() > 0.9:
        print("  → Model learned ADDITIVE decomposition: z(a,b) ≈ f(a) + g(b)")
    elif cos_sims.mean() > 0.5:
        print("  → Partially additive with significant interaction terms")
    else:
        print("  → Highly non-additive: the MLP computes complex interactions")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(cos_sims.numpy(), bins=50, color="steelblue", edgecolor="black", alpha=0.7)
axes[0].axvline(x=cos_sims.mean(), color="red", linestyle="--", label=f"Mean: {cos_sims.mean():.3f}")
axes[0].set_title("Additive Reconstruction Quality", fontweight="bold")
axes[0].set_xlabel("Cosine similarity"); axes[0].legend()

axes[1].pie([additive_var/total_var, residual_var/total_var],
            labels=["Additive", "Interaction"],
            colors=["#2563eb", "#f97316"], autopct="%1.1f%%", startangle=90)
axes[1].set_title("Variance Decomposition", fontweight="bold")

residual_norm = residual.norm(dim=-1).numpy()
im = axes[2].imshow(residual_norm, cmap="viridis", aspect="auto")
axes[2].set_title("Residual (non-additive) Magnitude", fontweight="bold")
axes[2].set_xlabel("Input b"); axes[2].set_ylabel("Input a")
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.savefig("additive_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === Analysis 2: What Does the Predictor Do? ===
# The predictor has a 64-dim bottleneck (128 → 64 → 64 → 128).
# If it's approximately linear, the mechanism is a simple rotation/projection.
# If it's highly nonlinear, the nonlinearity is part of the algorithm.

predictor.eval()
context_enc.eval()

with torch.no_grad():
    z_ctx = context_enc(pairs.to(device))
    z_pred = predictor(z_ctx)

    z_ctx_np = z_ctx.cpu().numpy()
    z_pred_np = z_pred.cpu().numpy()

    # Fit linear approximation: z_pred ≈ z_ctx @ W + b
    z_ctx_bias = np.hstack([z_ctx_np, np.ones((len(z_ctx_np), 1))])
    W_linear, _, _, _ = np.linalg.lstsq(z_ctx_bias, z_pred_np, rcond=None)
    z_pred_linear = z_ctx_bias @ W_linear
    z_pred_linear_norm = z_pred_linear / (np.linalg.norm(z_pred_linear, axis=1, keepdims=True) + 1e-10)

    # Quality of linear approximation
    linear_cos = (z_pred_np * z_pred_linear_norm).sum(axis=1)

    # SVD of effective weight matrix
    W_main = W_linear[:-1, :]  # (128, 128)
    U, S, Vt = np.linalg.svd(W_main)
    S_normed = S / S.sum()
    eff_rank_pred = 1.0 / (S_normed ** 2).sum()

    # Input-output angle
    input_output_cos = (z_ctx_np * z_pred_np).sum(axis=1)

    print(f"=== Predictor Analysis ===")
    print(f"Linear approximation quality (cosine): {linear_cos.mean():.4f} ± {linear_cos.std():.4f}")
    print(f"Effective rank of linear map: {eff_rank_pred:.1f}")
    print(f"Top-10 singular values capture: {S[:10].sum()/S.sum()*100:.1f}% of total")
    print(f"Input-output cosine: {input_output_cos.mean():.4f} ± {input_output_cos.std():.4f}")

    if linear_cos.mean() > 0.95:
        print("→ Predictor is APPROXIMATELY LINEAR — mechanism is a rotation/projection")
    elif linear_cos.mean() > 0.8:
        print("→ Predictor is MOSTLY linear with some nonlinear corrections")
    else:
        print("→ Predictor is SIGNIFICANTLY NONLINEAR — nonlinearity is essential")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(S, 'o-', markersize=3, color="steelblue")
axes[0].set_title(f"Predictor Singular Values (eff rank: {eff_rank_pred:.1f})", fontweight="bold")
axes[0].set_xlabel("Component"); axes[0].set_ylabel("Singular value")
axes[0].set_yscale("log"); axes[0].grid(True, alpha=0.3)

axes[1].hist(linear_cos, bins=50, color="#16a34a", edgecolor="black", alpha=0.7)
axes[1].axvline(x=linear_cos.mean(), color="red", linestyle="--", label=f"Mean: {linear_cos.mean():.3f}")
axes[1].set_title("Linear Approx Quality", fontweight="bold")
axes[1].set_xlabel("Cosine (actual vs linear)"); axes[1].legend()

axes[2].hist(input_output_cos, bins=50, color="#8b5cf6", edgecolor="black", alpha=0.7)
axes[2].axvline(x=input_output_cos.mean(), color="red", linestyle="--",
                label=f"Mean: {input_output_cos.mean():.3f}")
axes[2].set_title("Input → Output Direction Change", fontweight="bold")
axes[2].set_xlabel("Cosine (z_ctx vs z_pred)"); axes[2].legend()

plt.tight_layout()
plt.savefig("predictor_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === Analysis 3: Class-Mean Geometry ===
# For each residue c, average the predictor outputs for all (a,b) with (a+b)%p = c.
# These class means should align with the target encoder's codes.

with torch.no_grad():
    z_pred_all = predictor(context_enc(pairs.to(device)))
    z_tgt_codes = target_enc_ema(torch.arange(p, device=device))

    # Predicted class means
    pred_class_means = torch.zeros(p, LATENT_DIM, device=device)
    ctx_class_means = torch.zeros(p, LATENT_DIM, device=device)
    dispersions = torch.zeros(p)

    z_ctx_all = context_enc(pairs.to(device))

    for c in range(p):
        mask = targets.to(device) == c
        pred_class_means[c] = F.normalize(z_pred_all[mask].mean(dim=0), dim=0)
        ctx_class_means[c] = F.normalize(z_ctx_all[mask].mean(dim=0), dim=0)
        dispersions[c] = (z_pred_all[mask] * pred_class_means[c]).sum(dim=-1).mean().item()

    # Alignment: predicted means vs target codes
    alignment = (pred_class_means * z_tgt_codes).sum(dim=-1)

    # Decode accuracy via nearest target code
    sim_matrix = pred_class_means @ z_tgt_codes.T
    decode_acc = (sim_matrix.argmax(dim=1) == torch.arange(p, device=device)).float().mean()

    print(f"=== Class-Mean Geometry ===")
    print(f"Predicted-mean ↔ target-code alignment: {alignment.mean():.4f} (min: {alignment.min():.4f})")
    print(f"Nearest-code decode accuracy: {decode_acc*100:.1f}%")
    print(f"Within-class tightness: {dispersions.mean():.4f} (min: {dispersions.min():.4f})")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(range(p), alignment.cpu().numpy(), color="steelblue", alpha=0.8)
axes[0].axhline(y=alignment.mean().item(), color="red", linestyle="--", label=f"Mean: {alignment.mean():.3f}")
axes[0].set_title("Predicted Mean ↔ Target Code", fontweight="bold")
axes[0].set_xlabel("Residue c"); axes[0].set_ylabel("Cosine"); axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

im = axes[1].imshow(sim_matrix.cpu().numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
axes[1].set_title("Predicted Means vs Target Codes\n(should show diagonal)", fontweight="bold")
axes[1].set_xlabel("Target code"); axes[1].set_ylabel("Predicted mean")
plt.colorbar(im, ax=axes[1])

axes[2].bar(range(p), dispersions.numpy(), color="#16a34a", alpha=0.8)
axes[2].axhline(y=dispersions.mean(), color="red", linestyle="--", label=f"Mean: {dispersions.mean():.3f}")
axes[2].set_title("Within-Class Tightness", fontweight="bold")
axes[2].set_xlabel("Residue c"); axes[2].set_ylabel("Cos to mean"); axes[2].legend()
axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("class_mean_geometry.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === Analysis 4: Embedding Arithmetic & Commutativity ===
# Test 1: Does shifting a by delta produce a consistent direction in latent space?
# Test 2: Is the representation commutative? z(a,b) ≈ z(b,a)?

context_enc.eval()
with torch.no_grad():
    z_matrix = torch.zeros(p, p, LATENT_DIM)
    for a in range(p):
        pairs_a = torch.stack([torch.full((p,), a), torch.arange(p)], dim=1).to(device)
        z_matrix[a] = context_enc(pairs_a).cpu()

    # Test shifts for several delta values
    delta_results = []
    for delta in [1, 2, 5, 10, 24, 48]:
        all_shifts = torch.zeros(p, p, LATENT_DIM)
        for b in range(p):
            for a in range(p):
                a_plus = (a + delta) % p
                all_shifts[b, a] = z_matrix[a_plus, b] - z_matrix[a, b]

        mean_shift = F.normalize(all_shifts.reshape(-1, LATENT_DIM).mean(dim=0), dim=0)
        all_flat_norm = F.normalize(all_shifts.reshape(-1, LATENT_DIM), dim=-1)
        consistency = (all_flat_norm * mean_shift).sum(dim=-1)
        delta_results.append((delta, consistency.mean().item(), consistency.std().item()))

    # Commutativity test
    comm_sims = []
    for a in range(p):
        for b in range(a+1, p):
            sim = F.cosine_similarity(z_matrix[a, b].unsqueeze(0),
                                       z_matrix[b, a].unsqueeze(0)).item()
            comm_sims.append(sim)
    comm_arr = np.array(comm_sims)

    print(f"=== Shift Consistency (does z(a+Δ,b) - z(a,b) have consistent direction?) ===")
    for delta, m, s in delta_results:
        marker = "✓" if m > 0.3 else "✗"
        print(f"  Δ={delta:>2}: cosine = {m:.4f} ± {s:.4f}  {marker}")

    print(f"\n=== Commutativity: cos(z(a,b), z(b,a)) ===")
    print(f"  Mean: {comm_arr.mean():.4f}, Std: {comm_arr.std():.4f}")
    print(f"  → {'Commutative' if comm_arr.mean() > 0.9 else 'Partially commutative' if comm_arr.mean() > 0.5 else 'Non-commutative'}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

deltas = [d[0] for d in delta_results]
means = [d[1] for d in delta_results]
stds = [d[2] for d in delta_results]
axes[0].bar(range(len(deltas)), means, yerr=stds, color="steelblue", alpha=0.8, capsize=3)
axes[0].set_xticks(range(len(deltas)))
axes[0].set_xticklabels([f"Δ={d}" for d in deltas])
axes[0].set_title("Shift Consistency: z(a+Δ,b) - z(a,b)", fontweight="bold")
axes[0].set_ylabel("Mean cosine to avg shift direction")
axes[0].axhline(y=0, color="gray", linestyle="-", alpha=0.3)
axes[0].grid(True, alpha=0.3, axis="y")

axes[1].hist(comm_arr, bins=50, color="#f97316", edgecolor="black", alpha=0.7)
axes[1].axvline(x=comm_arr.mean(), color="red", linestyle="--", label=f"Mean: {comm_arr.mean():.3f}")
axes[1].set_title("Commutativity: cos(z(a,b), z(b,a))", fontweight="bold")
axes[1].set_xlabel("Cosine similarity"); axes[1].legend()

plt.tight_layout()
plt.savefig("embedding_arithmetic.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === Analysis 5: What Information is Linearly Decodable? ===
# Beyond the target (a+b)%p, what else can we extract from context encoder latents?

from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import cross_val_score

context_enc.eval()
with torch.no_grad():
    z_np = context_enc(pairs.to(device)).cpu().numpy()

pairs_np = pairs.numpy()

probes = {
    "(a+b) mod p": targets.numpy(),
    "a": pairs_np[:, 0],
    "b": pairs_np[:, 1],
    "(a-b) mod p": (pairs_np[:, 0] - pairs_np[:, 1]) % p,
    "(a*b) mod p": (pairs_np[:, 0].astype(np.int64) * pairs_np[:, 1].astype(np.int64)) % p,
    "a mod 2": pairs_np[:, 0] % 2,
    "(a+b) mod 2": (pairs_np[:, 0] + pairs_np[:, 1]) % 2,
}

print(f"=== Linear Probe Decodability ===")
print(f"{'Quantity':>16} | {'Classes':>7} | {'Accuracy':>9} | {'vs Chance':>9}")
print(f"{'-'*55}")

results = {}
for name, labels in probes.items():
    n_classes = len(np.unique(labels))
    clf = RidgeClassifier(alpha=1.0)
    scores = cross_val_score(clf, z_np, labels, cv=5, scoring="accuracy")
    acc = scores.mean()
    chance = 1.0 / n_classes
    results[name] = (acc, chance, n_classes)
    marker = "★" if acc > 0.8 else ("●" if acc > chance * 2 else "○")
    print(f"{name:>16} | {n_classes:>7} | {acc*100:>8.1f}% | {acc/chance:>8.1f}x  {marker}")

print("\n★ = highly decodable, ● = above chance, ○ = near chance")

fig, ax = plt.subplots(figsize=(12, 5))
names = list(results.keys())
accs = [results[n][0] * 100 for n in names]
chances = [results[n][1] * 100 for n in names]

x = np.arange(len(names))
width = 0.35
ax.bar(x - width/2, accs, width, label="Probe accuracy", color="steelblue", alpha=0.8)
ax.bar(x + width/2, chances, width, label="Chance level", color="#f97316", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Accuracy (%)")
ax.set_title("What Information is Linearly Accessible?", fontweight="bold", fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("information_probes.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nIf (a+b)%p is decodable but a*b is not → model specifically learned addition.")
print("If a and b are individually decodable → model preserves input identity in latents.")

In [ ]:
# === Mechanistic Summary ===
print("=" * 60)
print("MECHANISTIC ANALYSIS SUMMARY")
print("=" * 60)
print()
print("1. ADDITIVE DECOMPOSITION:")
print(f"   Additive variance fraction: {additive_var/total_var*100:.1f}%")
print(f"   Reconstruction cosine: {cos_sims.mean():.3f}")
print()
print("2. PREDICTOR:")
print(f"   Linearity (cosine): {linear_cos.mean():.3f}")
print(f"   Effective rank: {eff_rank_pred:.1f}")
print()
print("3. CLASS-MEAN ALIGNMENT:")
print(f"   Mean-to-code cosine: {alignment.mean():.3f}")
print(f"   Nearest-code accuracy: {decode_acc*100:.0f}%")
print()
print("4. COMMUTATIVITY:")
print(f"   cos(z(a,b), z(b,a)): {comm_arr.mean():.3f}")
print()
print("5. DECODABILITY:")
for name, (acc, chance, nc) in results.items():
    if acc > chance * 2:
        print(f"   {name}: {acc*100:.1f}% ({acc/chance:.1f}x chance)")
print()
print("=" * 60)

## 12. Supervised Baseline Comparison

To confirm that the differences we observe are due to the **loss function** and not the architecture, we run the same embedding + MLP with standard supervised cross-entropy loss and track the same metrics.

In [ ]:
class SupervisedGrokkingModel(nn.Module):
    """Standard supervised model for comparison.
    Same embedding and hidden dimensions as the JEPA context encoder."""
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        e = self.emb(x)
        e = e.view(e.size(0), -1)
        return self.net(e)


# Train supervised baseline
torch.manual_seed(SEED)
sup_model = SupervisedGrokkingModel(p, HIDDEN_DIM).to(device)
sup_model = torch.compile(sup_model)
sup_optimizer = optim.AdamW(sup_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sup_criterion = nn.CrossEntropyLoss()

SUP_EPOCHS = 20000  # Generous budget for supervised baseline
sup_history = {"epoch": [], "train_acc": [], "val_acc": [], "fourier_top5": []}

print(f"Training supervised baseline for {SUP_EPOCHS} epochs...")
sup_start = time.time()

for epoch in range(SUP_EPOCHS):
    sup_model.train()
    for bp, bt in train_loader:
        sup_optimizer.zero_grad()
        out = sup_model(bp)
        loss = sup_criterion(out, bt)
        loss.backward()
        sup_optimizer.step()

    if epoch % 500 == 0 or epoch == SUP_EPOCHS - 1:
        sup_model.eval()
        with torch.no_grad():
            train_acc = sup_model(train_pairs).argmax(1).eq(train_targets).float().mean().item()
            val_acc = sup_model(val_pairs).argmax(1).eq(val_targets).float().mean().item()

            # Fourier analysis of supervised model's embeddings
            emb_weights = sup_model.emb.weight.cpu().numpy()  # (p, hidden_dim)
            fft_mag = np.abs(np.fft.fft(emb_weights, axis=0))
            energy = (fft_mag ** 2).sum(axis=1)
            energy[0] = 0
            total = energy.sum()
            f_top5 = np.sort(energy)[-5:].sum() / total if total > 1e-12 else 0.0

        sup_history["epoch"].append(epoch)
        sup_history["train_acc"].append(train_acc)
        sup_history["val_acc"].append(val_acc)
        sup_history["fourier_top5"].append(f_top5)

        if epoch % 2000 == 0:
            print(f"  Epoch {epoch:5d} | Train: {train_acc*100:.1f}% | "
                  f"Val: {val_acc*100:.1f}% | Fourier: {f_top5:.3f}")

print(f"Supervised baseline done in {(time.time()-sup_start)/60:.1f}m")
print(f"Final: Train {sup_history['train_acc'][-1]*100:.1f}%, Val {sup_history['val_acc'][-1]*100:.1f}%")

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Accuracy comparison
axes[0].plot(probe_epochs, [a*100 for a in val_accs],
             label="JEPA (self-supervised)", color="#f97316", linewidth=2)
axes[0].plot(sup_history["epoch"], [a*100 for a in sup_history["val_acc"]],
             label="Supervised (cross-entropy)", color="#2563eb", linewidth=2)
axes[0].axhline(y=100, color="red", linestyle="--", alpha=0.3)
axes[0].set_title("Validation Accuracy Comparison", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Val Accuracy (%)")
axes[0].legend(fontsize=10)
axes[0].set_ylim(-5, 110)
axes[0].grid(True, alpha=0.3)

# Fourier structure comparison
axes[1].plot(ep, history["fourier_top5_target"],
             label="JEPA target encoder", color="#f97316", linewidth=1.5)
axes[1].plot(sup_history["epoch"], sup_history["fourier_top5"],
             label="Supervised embeddings", color="#2563eb", linewidth=1.5)
axes[1].axhline(y=5/p, color="gray", linestyle=":", alpha=0.7, label="Uniform baseline")
axes[1].set_title("Fourier Structure Comparison", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Fourier Top-5 Concentration")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Supervised Fourier spectrum at end
sup_emb_np = sup_model.emb.weight.detach().cpu().numpy()
sup_fft = np.abs(np.fft.fft(sup_emb_np, axis=0))
sup_energy = (sup_fft ** 2).sum(axis=1)
sup_energy[0] = 0

axes[2].bar(range(p), sup_energy, color="#2563eb", alpha=0.7, label="Supervised")
axes[2].set_title("Supervised Model — Final Fourier Spectrum", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Frequency index k")
axes[2].set_ylabel("Energy")
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("jepa_vs_supervised.png", dpi=150, bbox_inches="tight")
plt.show()
print("Supervised model develops clear Fourier peaks (sharp spikes).")
print("JEPA model has flat spectrum — yet both generalize to 100%.")
print("Two different generalization pathways for the same mathematical task.")

## 13. Multi-Seed Validation

To confirm this isn't a lucky initialization, we run 3 additional seeds and check for consistent grokking behavior **and** confirm that all seeds show flat Fourier spectra.

*(Note: Each run takes ~50-80 min on T4. The generous epoch budget ensures grokking completes even for slow seeds.)*

In [ ]:
def quick_jepa_run(seed, epochs=100000, log_every=1000):
    """Lightweight JEPA training run for multi-seed validation."""
    torch.manual_seed(seed)

    perm_s = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    tr_idx, vl_idx = perm_s[:n_train], perm_s[n_train:]
    tr_p = pairs[tr_idx].to(device)
    tr_t = targets[tr_idx].to(device)
    vl_p = pairs[vl_idx].to(device)
    vl_t = targets[vl_idx].to(device)
    loader = DataLoader(TensorDataset(tr_p, tr_t), batch_size=n_train, shuffle=True)

    ctx = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    tgt = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    prd = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
    tgt_ema = deepcopy(tgt)
    for param in tgt_ema.parameters():
        param.requires_grad = False

    # Compile models for faster training
    ctx = torch.compile(ctx)
    tgt = torch.compile(tgt)
    prd = torch.compile(prd)
    tgt_ema = torch.compile(tgt_ema)

    opt = optim.AdamW(
        list(ctx.parameters()) + list(prd.parameters()) + list(tgt.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )

    val_accs = []
    ep_list = []

    for epoch in range(epochs):
        ctx.train(); tgt.train(); prd.train()
        for bp, bt in loader:
            opt.zero_grad()
            z_c = ctx(bp)
            z_p = prd(z_c)
            with torch.no_grad():
                z_t = tgt_ema(bt)
            loss = -(z_p * z_t).sum(dim=-1).mean()
            loss.backward()
            opt.step()
            with torch.no_grad():
                for o, t in zip(tgt.parameters(), tgt_ema.parameters()):
                    t.data.mul_(EMA_DECAY).add_(o.data, alpha=1 - EMA_DECAY)

        if epoch % log_every == 0 or epoch == epochs - 1:
            ctx.eval()
            _, va = linear_probe_accuracy(ctx, vl_p, vl_t, tr_p, tr_t, p)
            val_accs.append(va)
            ep_list.append(epoch)
            if epoch % 10000 == 0:
                print(f"    Seed {seed} | Epoch {epoch:5d} | Val: {va*100:.1f}%")

    # Final Fourier check on target encoder
    f_top5, _ = compute_fourier_structure(tgt_ema, p)
    print(f"    Seed {seed} | Final Fourier top-5: {f_top5:.3f} (uniform baseline: {5/p:.3f})")

    return ep_list, val_accs, f_top5


# Run additional seeds
extra_seeds = [7, 123, 2024]
seed_results = {}

for s in extra_seeds:
    print(f"\nRunning seed {s}...")
    t0 = time.time()
    ep_list, va_list, f_top5 = quick_jepa_run(s, epochs=EPOCHS)
    seed_results[s] = (ep_list, va_list, f_top5)
    print(f"  Seed {s} done in {(time.time()-t0)/60:.1f}m — Peak val: {max(va_list)*100:.1f}%")

In [ ]:
# Multi-seed comparison plot
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Left: Grokking curves
axes[0].plot(probe_epochs, [a*100 for a in val_accs],
        label=f"Seed {SEED}", linewidth=2, color="#f97316")

colors = ["#2563eb", "#16a34a", "#dc2626"]
for (s, (ep_list, va_list, _)), c in zip(seed_results.items(), colors):
    axes[0].plot(ep_list, [a*100 for a in va_list], label=f"Seed {s}", linewidth=2, color=c)

axes[0].axhline(y=100, color="red", linestyle="--", alpha=0.3)
axes[0].axhline(y=100/p, color="gray", linestyle=":", alpha=0.5)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Validation Accuracy (%)", fontsize=12)
axes[0].set_title("Multi-Seed Validation: Grokking Curves",
             fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].set_ylim(-5, 110)
axes[0].grid(True, alpha=0.3)

# Right: Final Fourier check across all seeds
seed_labels = [f"Seed {SEED}\n(main)"] + [f"Seed {s}" for s in extra_seeds]
fourier_vals = [history["fourier_top5_target"][-1]] + [seed_results[s][2] for s in extra_seeds]
bar_colors = ["#f97316"] + colors
axes[1].bar(seed_labels, fourier_vals, color=bar_colors, alpha=0.8, edgecolor="black", linewidth=0.5)
axes[1].axhline(y=5/p, color="gray", linestyle="--", alpha=0.7, label=f"Uniform baseline ({5/p:.3f})")
axes[1].set_ylabel("Fourier Top-5 Concentration", fontsize=12)
axes[1].set_title("All Seeds: No Fourier Structure",
             fontsize=13, fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("multi_seed_validation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grokking is reproducible across seeds (onset timing varies).")
print("All seeds show flat Fourier spectra — the non-Fourier pathway is consistent.")

## 14. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(24, 20))
fig.suptitle("Self-Supervised Grokking: Complete Analysis Dashboard",
             fontsize=16, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(4, 3, hspace=0.38, wspace=0.3)

# Row 1: Core findings
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(probe_epochs, [a*100 for a in train_accs], label="Train", color="#2563eb", linewidth=2)
ax1.plot(probe_epochs, [a*100 for a in val_accs], label="Val", color="#f97316", linewidth=2)
ax1.set_title("Grokking Curve", fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy (%)")
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_ylim(-5, 110)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ep, history["fourier_top5_target"], color="purple", linewidth=1.5, label="Target enc")
ax2.plot(ep, history["fourier_top5_context"], color="teal", linewidth=1.5, label="Context enc")
ax2.axhline(y=5/p, color="gray", linestyle=":", alpha=0.7)
ax2.set_title("Fourier Concentration", fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Top-5 Energy Fraction")
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(ep, history["eff_rank"], color="green", linewidth=1.5)
ax3.set_title("Effective Rank", fontweight="bold")
ax3.set_xlabel("Epoch"); ax3.set_ylabel("Participation Ratio")
ax3.grid(True, alpha=0.3)

# Row 2: New insights + mechanistic
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(ep, history["jepa_loss"], color="#8b5cf6", linewidth=1, alpha=0.7)
val_epochs_clean = [e for e, a in zip(probe_epochs, val_accs) if a is not None]
val_accs_clean = [a for a in val_accs if a is not None]
ax4_twin = ax4.twinx()
ax4_twin.plot(val_epochs_clean, [a*100 for a in val_accs_clean], color="#f97316", linewidth=1.5, alpha=0.8)
ax4_twin.set_ylabel("Val Accuracy (%)", color="#f97316")
ax4.set_title("JEPA Loss vs Val Accuracy", fontweight="bold")
ax4.set_xlabel("Epoch"); ax4.set_ylabel("JEPA Loss", color="#8b5cf6")
ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(gs[1, 1])
im = ax5.imshow(cos_sim, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax5.set_title("Residue Cosine Similarity", fontweight="bold")
ax5.set_xlabel("Residue j"); ax5.set_ylabel("Residue i")
plt.colorbar(im, ax=ax5)

ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(range(p), avg_sim_by_dist, 'o-', markersize=2, color="steelblue")
ax6.set_title("Similarity by Modular Distance", fontweight="bold")
ax6.set_xlabel("Distance d"); ax6.set_ylabel("Avg cosine sim")
ax6.grid(True, alpha=0.3)

# Row 3: Fourier spectra + rate of change
ax7 = fig.add_subplot(gs[2, 0])
ax7.bar(range(p), final_spectrum_tgt, color="steelblue", alpha=0.8)
ax7.set_title("Target Enc Fourier Spectrum", fontweight="bold")
ax7.set_xlabel("Frequency k"); ax7.set_ylabel("Energy")

ax8 = fig.add_subplot(gs[2, 1])
ax8.bar(range(p), sup_energy, color="#2563eb", alpha=0.7)
ax8.set_title("Supervised Fourier Spectrum", fontweight="bold")
ax8.set_xlabel("Frequency k"); ax8.set_ylabel("Energy")

ax9 = fig.add_subplot(gs[2, 2])
val_arr_d = np.array(val_accs_clean) * 100
ep_arr_d = np.array(val_epochs_clean)
if len(val_arr_d) > 1:
    rate = np.diff(val_arr_d) / np.diff(ep_arr_d) * 1000  # % per 1000 epochs
    ax9.plot(ep_arr_d[1:], rate, color="#dc2626", linewidth=1.5)
    ax9.fill_between(ep_arr_d[1:], rate, alpha=0.15, color="#dc2626")
ax9.set_title("Val Accuracy Rate of Change", fontweight="bold")
ax9.set_xlabel("Epoch"); ax9.set_ylabel("Δ Acc (% per 1k epochs)")
ax9.grid(True, alpha=0.3)

# Row 4: Comparison
ax10 = fig.add_subplot(gs[3, 0])
ax10.plot(probe_epochs, [a*100 for a in val_accs], label="JEPA", color="#f97316", linewidth=2)
ax10.plot(sup_history["epoch"], [a*100 for a in sup_history["val_acc"]],
         label="Supervised", color="#2563eb", linewidth=2)
ax10.set_title("JEPA vs Supervised", fontweight="bold")
ax10.set_xlabel("Epoch"); ax10.set_ylabel("Val Acc (%)")
ax10.legend(); ax10.grid(True, alpha=0.3); ax10.set_ylim(-5, 110)

ax11 = fig.add_subplot(gs[3, 1:])
ax11.plot(probe_epochs, [a*100 for a in val_accs], label=f"Seed {SEED}", color="#f97316", linewidth=2)
for (s, (epl, val, _)), c in zip(seed_results.items(), colors):
    ax11.plot(epl, [a*100 for a in val], label=f"Seed {s}", linewidth=1.5, color=c)
ax11.set_title("Multi-Seed Reproducibility", fontweight="bold")
ax11.set_xlabel("Epoch"); ax11.set_ylabel("Val Acc (%)")
ax11.legend(fontsize=8); ax11.grid(True, alpha=0.3); ax11.set_ylim(-5, 110)

plt.savefig("complete_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. Conclusions

### What we demonstrated:

1. **Self-supervised grokking is real.** A JEPA model, trained only to predict latent representations (never given classification labels), exhibits delayed generalization on modular addition — reaching 100% validation accuracy after a prolonged memorization phase. This is the first demonstration of grokking under a self-supervised objective.

2. **Fourier structure is anti-correlated with generalization.** In every prior study, grokking on modular arithmetic coincided with the *emergence* of Fourier circuits. Our JEPA model shows the opposite: the target encoder maintains a flat spectrum throughout training (top-5 ≈ 0.071, near the uniform baseline of 0.052), while the context encoder develops mild spectral concentration during memorization (~0.39) that then *decreases* during grokking (~0.21). Generalization coincides with the *dismantling* of partial Fourier structure, not its formation. This demonstrates that DFT-detectable Fourier circuits are not the only pathway to learning modular arithmetic.

3. **Geometric dynamics are reversed.** During supervised grokking, effective rank typically decreases (representations compress). In JEPA grokking, effective rank *increases* during the generalization phase. The model expands its representational dimensionality while generalizing — suggesting a fundamentally different internal reorganization.

4. **Training loss is blind to grokking.** The JEPA loss barely changes during the generalization transition (from ~-0.91 to ~-0.96 while val accuracy goes from ~10% to 100%). Unlike supervised grokking where loss is trivially at zero, here the loss is actively optimized yet provides no signal about the internal reorganization from memorization to generalization. The memorizing and generalizing solutions occupy nearly the same loss basin.

### Implications:

- **For grokking theory:** Existing theories (Nanda, Barak, Liu) that explain grokking via Fourier circuit formation describe one pathway, not the only one. A complete theory must account for generalization without DFT-detectable periodic structure.

- **For self-supervised learning:** JEPA-style objectives can discover abstract algorithmic structure, supporting LeCun's thesis that latent prediction learns "world models" — demonstrated here in a rigorous, verifiable setting.

- **For mechanistic interpretability:** The mechanism behind JEPA grokking remains an open question. What algorithm did the model discover if not Fourier circuits? Understanding this could reveal new generalization mechanisms.

### Open questions:

- What is the learned algorithm? (Requires deeper mechanistic analysis)
- Does this extend to other modular operations (multiplication, polynomials)?
- Is the bottleneck hypothesis correct — does grokking require latent_dim < p?
- Can we characterize the geometric phase transition more precisely?

---

*Code and analysis by Kurian. All experiments reproducible with the provided seed.*

# Part 2: Is It Truly Self-Supervised?

**Description:** Testing what the label contributes: Tests 3 truly SSL approaches vs Label-JEPA. Finding: truly SSL fails, label is necessary.

---

# Truly Self-Supervised JEPA on Modular Arithmetic

## The Question

The original "JEPA grokking" notebook feeds the label $c = (a+b) \bmod p$ to the target
encoder. This makes it architecturally JEPA but **not self-supervised** — delete the labels
and training breaks.

This notebook asks: **can a genuinely self-supervised JEPA — where both encoders see
only the input $(a, b)$, never the answer $c$ — learn modular arithmetic?**

We implement three truly SSL approaches, all following I-JEPA's principle of predicting
one view of the input from another:

| Approach | Context Encoder Sees | Target Encoder Sees | Analogy |
|----------|---------------------|--------------------:|---------|
| **A. Token Masking** | $(a, \text{MASK})$ | $(\text{MASK}, b)$ | I-JEPA: predict masked patches |
| **B. Dropout Augmentation** | $(a, b)$ + dropout $\alpha$ | $(a, b)$ + dropout $\beta$ | BYOL: two augmented views |
| **C. Commutative Swap** | $(a, b)$ | $(b, a)$ | Symmetry-based view |
| **D. Label-JEPA (control)** | $(a, b)$ | $c = (a+b)\bmod p$ | Original notebook |

**Prediction:** Approaches A–C will likely **not** grok modular addition. The label $c$
is the only signal that groups $(3,4)$ with $(5,2)$ and $(0,7)$. Without it, the model has
no reason to discover residue class structure. A negative result here would prove that
the label information entering through the target encoder is **necessary** for grokking —
which precisely characterizes what the original setup contributes.

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import time
from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.decomposition import PCA

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Hyperparameters

Identical to the original JEPA notebook.

In [ ]:
p = 97
train_frac = 0.3

LATENT_DIM = 128
HIDDEN_DIM = 256
PREDICTOR_DIM = 64

EPOCHS = 60_000
LR = 1e-3
WEIGHT_DECAY = 1.0
EMA_DECAY = 0.996

LOG_EVERY = 200
PROBE_EVERY = 500

# Dropout augmentation strength (Approach B)
DROPOUT_RATE = 0.3

print(f"Task: (a + b) mod {p}")
print(f"Total pairs: {p**2}, Train: {int(train_frac * p**2)}, Val: {p**2 - int(train_frac * p**2)}")

## 3. Dataset

In [ ]:
pairs = torch.cartesian_prod(torch.arange(p), torch.arange(p))
targets = (pairs[:, 0] + pairs[:, 1]) % p

n = len(pairs)
n_train = int(train_frac * n)
n_val = n - n_train

perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:n_train], perm[n_train:]

train_pairs = pairs[train_idx].to(device)
train_targets = targets[train_idx].to(device)
val_pairs = pairs[val_idx].to(device)
val_targets = targets[val_idx].to(device)

train_loader = DataLoader(
    TensorDataset(train_pairs, train_targets),
    batch_size=n_train, shuffle=True
)

print(f"Train: {n_train} | Val: {n_val}")

## 4. Architecture

### Shared components across all approaches

All approaches use the same encoder backbone. The difference is only in
**what data flows into each encoder** — the architecture is identical.

For token masking (Approach A), each encoder sees a single token,
so we need a single-token encoder alongside the pair encoder.

In [ ]:
class PairEncoder(nn.Module):
    """Encodes a full (a, b) pair into latent space.
    Used by Approaches B (dropout), C (swap), and D (label-JEPA control).
    Architecture identical to original JEPA context encoder."""
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)                    # (batch, 2, hidden_dim)
        e = e.view(e.size(0), -1)          # (batch, 2 * hidden_dim)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class PairEncoderWithDropout(nn.Module):
    """Encodes (a, b) with dropout on embeddings — creates stochastic views.
    Used by Approach B. Same architecture, but with configurable dropout."""
    def __init__(self, vocab_size, latent_dim, hidden_dim, dropout_rate=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)
        e = self.dropout(e)                # Stochastic augmentation
        e = e.view(e.size(0), -1)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class TokenEncoder(nn.Module):
    """Encodes a single token (a or b) into latent space.
    Used by Approach A (token masking). Matches target encoder from original
    JEPA notebook — single token input, simpler architecture."""
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class Predictor(nn.Module):
    """Maps context latent → predicted target latent.
    Identical across all approaches."""
    def __init__(self, latent_dim, predictor_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, latent_dim),
        )

    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)


def count_params(model):
    return sum(p_.numel() for p_ in model.parameters() if p_.requires_grad)

print("Architecture defined.")

## 5. Metric Functions

Identical to the original notebook for fair comparison.

In [ ]:
@torch.no_grad()
def compute_geometry(z):
    """Compute geometric properties of latent representations."""
    z_c = z - z.mean(dim=0, keepdim=True)
    cov = (z_c.T @ z_c) / (z.shape[0] - 1)
    eigvals = torch.linalg.eigvalsh(cov).clamp(min=1e-10)

    normed = eigvals / eigvals.sum()
    eff_rank = (1.0 / (normed ** 2).sum()).item()
    top_ratio = (eigvals[-1] / eigvals.sum()).item()

    if z.shape[0] > 2048:
        idx = torch.randperm(z.shape[0])[:2048]
        z_sub = z[idx]
    else:
        z_sub = z
    sq_pdist = torch.cdist(z_sub, z_sub, p=2).pow(2)
    uniformity = torch.log(torch.exp(-2 * sq_pdist).mean() + 1e-10).item()

    return {
        "effective_rank": eff_rank,
        "top_eig_ratio": top_ratio,
        "uniformity": uniformity,
    }


@torch.no_grad()
def compute_fourier_structure(encoder, p, input_data, target_data):
    """Fourier structure of encoder's per-class-mean representations."""
    z_all = encoder(input_data)
    residue_means = torch.zeros(p, z_all.shape[1], device=z_all.device)
    for i in range(p):
        mask = target_data == i
        if mask.sum() > 0:
            residue_means[i] = z_all[mask].mean(dim=0)

    z_np = residue_means.cpu().numpy()
    fft_mag = np.abs(np.fft.fft(z_np, axis=0))
    energy = (fft_mag ** 2).sum(axis=1)
    energy[0] = 0

    total = energy.sum()
    if total < 1e-12:
        return 0.0, energy
    top5_ratio = np.sort(energy)[-5:].sum() / total
    return top5_ratio, energy


@torch.no_grad()
def linear_probe_accuracy(encoder, val_pairs, val_targets, train_pairs, train_targets, p):
    """Closed-form ridge regression linear probe."""
    z_train = encoder(train_pairs)
    z_val = encoder(val_pairs)
    y_train = F.one_hot(train_targets, num_classes=p).float()

    reg = 1e-3
    ZtZ = z_train.T @ z_train + reg * torch.eye(z_train.shape[1], device=device)
    ZtY = z_train.T @ y_train
    W = torch.linalg.solve(ZtZ, ZtY)

    train_acc = (z_train @ W).argmax(dim=1).eq(train_targets).float().mean().item()
    val_acc = (z_val @ W).argmax(dim=1).eq(val_targets).float().mean().item()
    return train_acc, val_acc


@torch.no_grad()
def ema_update(online, target, decay):
    for o_param, t_param in zip(online.parameters(), target.parameters()):
        t_param.data.mul_(decay).add_(o_param.data, alpha=1 - decay)


print("Metrics defined.")

## 6. Generic Training Function

A single training function that handles all four approaches.
The only difference between approaches is how data flows through the encoders.

In [ ]:
def train_jepa_variant(
    approach_name,
    context_encoder,
    target_encoder,
    predictor,
    get_context_input,      # fn(batch_pairs, batch_targets) → context encoder input
    get_target_input,        # fn(batch_pairs, batch_targets) → target encoder input
    eval_encoder,            # which encoder to evaluate with linear probe
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    ema_decay=EMA_DECAY,
    log_every=LOG_EVERY,
    probe_every=PROBE_EVERY,
    verbose=True,
):
    """Train a JEPA variant and track all metrics.

    The approach is defined entirely by get_context_input and get_target_input,
    which determine what data each encoder receives.
    """

    # EMA target
    target_ema = deepcopy(target_encoder)
    for param in target_ema.parameters():
        param.requires_grad = False

    # Compile models for faster training
    context_encoder = torch.compile(context_encoder)
    target_encoder = torch.compile(target_encoder)
    predictor = torch.compile(predictor)
    target_ema = torch.compile(target_ema)

    optimizer = optim.AdamW(
        list(context_encoder.parameters()) +
        list(predictor.parameters()) +
        list(target_encoder.parameters()),
        lr=lr, weight_decay=weight_decay,
    )

    history = {
        "epoch": [], "jepa_loss": [],
        "train_acc": [], "val_acc": [],
        "eff_rank": [], "top_eig_ratio": [],
        "uniformity": [], "fourier_top5": [],
    }

    start_time = time.time()

    for epoch in range(epochs):
        context_encoder.train()
        target_encoder.train()
        predictor.train()

        for batch_pairs, batch_targets in train_loader:
            optimizer.zero_grad()

            ctx_input = get_context_input(batch_pairs, batch_targets)
            tgt_input = get_target_input(batch_pairs, batch_targets)

            z_context = context_encoder(ctx_input)
            z_pred = predictor(z_context)

            with torch.no_grad():
                z_target = target_ema(tgt_input)

            loss = -(z_pred * z_target).sum(dim=-1).mean()

            loss.backward()
            optimizer.step()

            ema_update(target_encoder, target_ema, ema_decay)

        # Logging
        if epoch % log_every == 0 or epoch == epochs - 1:
            context_encoder.eval()
            eval_encoder_fn = eval_encoder  # which encoder to probe

            with torch.no_grad():
                ctx_in = get_context_input(train_pairs, train_targets)
                tgt_in = get_target_input(train_pairs, train_targets)
                z_p = predictor(context_encoder(ctx_in))
                z_t = target_ema(tgt_in)
                jepa_loss = -(z_p * z_t).sum(dim=-1).mean().item()

            # Geometry on eval encoder
            with torch.no_grad():
                z_full = eval_encoder_fn(pairs.to(device))
                geo = compute_geometry(z_full)

            # Fourier structure
            f_top5, _ = compute_fourier_structure(
                eval_encoder_fn, p, pairs.to(device), targets.to(device)
            )

            history["epoch"].append(epoch)
            history["jepa_loss"].append(jepa_loss)
            history["eff_rank"].append(geo["effective_rank"])
            history["top_eig_ratio"].append(geo["top_eig_ratio"])
            history["uniformity"].append(geo["uniformity"])
            history["fourier_top5"].append(f_top5)

            if epoch % probe_every == 0 or epoch == epochs - 1:
                train_acc, val_acc = linear_probe_accuracy(
                    eval_encoder_fn, val_pairs, val_targets,
                    train_pairs, train_targets, p
                )
                history["train_acc"].append(train_acc)
                history["val_acc"].append(val_acc)

                if verbose and epoch % (probe_every * 10) == 0:
                    elapsed = time.time() - start_time
                    print(
                        f"  [{approach_name}] Epoch {epoch:5d} ({elapsed/60:.1f}m) | "
                        f"Loss: {jepa_loss:.4f} | "
                        f"Train: {train_acc*100:.1f}% | Val: {val_acc*100:.1f}% | "
                        f"EffRank: {geo['effective_rank']:.1f} | "
                        f"Fourier: {f_top5:.3f}"
                    )
            else:
                history["train_acc"].append(None)
                history["val_acc"].append(None)

    total_time = time.time() - start_time
    final_val = [v for v in history['val_acc'] if v is not None][-1]
    peak_val = max(v for v in history['val_acc'] if v is not None)
    print(f"  [{approach_name}] Done in {total_time/60:.1f}m | "
          f"Peak val: {peak_val*100:.1f}% | Final val: {final_val*100:.1f}%")

    return history, context_encoder, target_ema, predictor


print("Training function defined.")

---
## Approach A: Token Masking (I-JEPA Style)

The most faithful I-JEPA analogue. The input pair $(a, b)$ is treated as two "patches."
The context encoder sees only $a$, the target encoder sees only $b$.
The predictor must map $\text{latent}(a) \to \text{latent}(b)$.

**This is genuinely self-supervised**: the label $c$ never appears.

**Why it probably won't learn modular arithmetic**: Knowing $a$ tells you nothing about
$(a+b) \bmod p$ without also knowing $b$. The model can learn co-occurrence statistics
of the training split, but not the modular structure.

In [ ]:
print("=" * 70)
print("APPROACH A: Token Masking (I-JEPA style)")
print("  Context encoder sees: a (first operand only)")
print("  Target encoder sees:  b (second operand only)")
print("  Label c: NEVER USED")
print("=" * 70)

torch.manual_seed(SEED)

ctx_enc_A = TokenEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
tgt_enc_A = TokenEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
pred_A = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)

# Compile models
ctx_enc_A = torch.compile(ctx_enc_A)
tgt_enc_A = torch.compile(tgt_enc_A)
pred_A = torch.compile(pred_A)

print(f"Context encoder (token): {count_params(ctx_enc_A):,} params")
print(f"Target encoder (token):  {count_params(tgt_enc_A):,} params")
print(f"Predictor:               {count_params(pred_A):,} params")

# Data routing: context gets a, target gets b
def get_ctx_A(batch_pairs, batch_targets):
    return batch_pairs[:, 0]  # just token a

def get_tgt_A(batch_pairs, batch_targets):
    return batch_pairs[:, 1]  # just token b

# For evaluation, we need a wrapper that takes full pairs but only uses one token
class TokenEncoderPairWrapper(nn.Module):
    """Wraps a TokenEncoder to accept (a,b) pairs but only encode token at given index."""
    def __init__(self, token_encoder, token_idx):
        super().__init__()
        self.token_encoder = token_encoder
        self.token_idx = token_idx
    def forward(self, x):
        if x.dim() == 2:  # (batch, 2) pair input
            return self.token_encoder(x[:, self.token_idx])
        return self.token_encoder(x)  # single token input

eval_enc_A = TokenEncoderPairWrapper(ctx_enc_A, 0)

history_A, ctx_enc_A, tgt_ema_A, pred_A = train_jepa_variant(
    "A: Token Masking",
    ctx_enc_A, tgt_enc_A, pred_A,
    get_context_input=get_ctx_A,
    get_target_input=get_tgt_A,
    eval_encoder=eval_enc_A,
)

---
## Approach B: Dropout Augmentation (BYOL Style)

Both encoders see the full pair $(a, b)$, but with different stochastic dropout masks
on the embeddings. This is the BYOL analogue: two noisy views of the same input.

**Genuinely self-supervised**: label $c$ never appears.

**Why it might learn something**: If the model learns dropout-invariant features,
it could capture structure in the input space. But will it discover *modular arithmetic*
specifically? Unlikely without a signal that groups same-residue pairs.

In [ ]:
print("\n" + "=" * 70)
print("APPROACH B: Dropout Augmentation (BYOL style)")
print("  Context encoder sees: (a, b) + dropout mask α")
print("  Target encoder sees:  (a, b) + dropout mask β")
print("  Label c: NEVER USED")
print("=" * 70)

torch.manual_seed(SEED)

ctx_enc_B = PairEncoderWithDropout(p, LATENT_DIM, HIDDEN_DIM, DROPOUT_RATE).to(device)
tgt_enc_B = PairEncoderWithDropout(p, LATENT_DIM, HIDDEN_DIM, DROPOUT_RATE).to(device)
pred_B = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)

# Compile models
ctx_enc_B = torch.compile(ctx_enc_B)
tgt_enc_B = torch.compile(tgt_enc_B)
pred_B = torch.compile(pred_B)

print(f"Context encoder (dropout): {count_params(ctx_enc_B):,} params")
print(f"Target encoder (dropout):  {count_params(tgt_enc_B):,} params")
print(f"Predictor:                 {count_params(pred_B):,} params")

# Data routing: both get full pairs (dropout provides the augmentation)
def get_ctx_B(batch_pairs, batch_targets):
    return batch_pairs  # full (a, b) — dropout happens inside encoder

def get_tgt_B(batch_pairs, batch_targets):
    return batch_pairs  # full (a, b) — different dropout mask (stochastic)

history_B, ctx_enc_B, tgt_ema_B, pred_B = train_jepa_variant(
    "B: Dropout Aug",
    ctx_enc_B, tgt_enc_B, pred_B,
    get_context_input=get_ctx_B,
    get_target_input=get_tgt_B,
    eval_encoder=ctx_enc_B,
)

---
## Approach C: Commutative Swap

Context encoder sees $(a, b)$, target encoder sees $(b, a)$.
Exploits the fact that addition is commutative: $(a+b) = (b+a)$.

**Genuinely self-supervised**: label $c$ never appears. The swap is a property
of the input data itself.

**What it can learn**: The model should learn that order doesn't matter —
representations should be permutation-invariant. But this is a much weaker
constraint than "group pairs by residue class." The model learns *commutativity*
but not *modular arithmetic*.

In [ ]:
print("\n" + "=" * 70)
print("APPROACH C: Commutative Swap")
print("  Context encoder sees: (a, b)")
print("  Target encoder sees:  (b, a)")
print("  Label c: NEVER USED")
print("=" * 70)

torch.manual_seed(SEED)

ctx_enc_C = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
tgt_enc_C = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
pred_C = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)

# Compile models
ctx_enc_C = torch.compile(ctx_enc_C)
tgt_enc_C = torch.compile(tgt_enc_C)
pred_C = torch.compile(pred_C)

print(f"Context encoder (pair): {count_params(ctx_enc_C):,} params")
print(f"Target encoder (pair):  {count_params(tgt_enc_C):,} params")
print(f"Predictor:              {count_params(pred_C):,} params")

# Data routing: context gets (a,b), target gets (b,a)
def get_ctx_C(batch_pairs, batch_targets):
    return batch_pairs  # (a, b)

def get_tgt_C(batch_pairs, batch_targets):
    return batch_pairs.flip(1)  # (b, a)

history_C, ctx_enc_C, tgt_ema_C, pred_C = train_jepa_variant(
    "C: Swap",
    ctx_enc_C, tgt_enc_C, pred_C,
    get_context_input=get_ctx_C,
    get_target_input=get_tgt_C,
    eval_encoder=ctx_enc_C,
)

---
## Approach D: Label-JEPA (Control — Original Notebook)

This is the original setup for direct comparison. Context encoder sees $(a, b)$,
target encoder sees $c = (a+b) \bmod p$.

**NOT self-supervised**: the label $c$ enters through the target encoder.

This serves as the positive control — we know this groks.

In [ ]:
print("\n" + "=" * 70)
print("APPROACH D: Label-JEPA (control — original notebook)")
print("  Context encoder sees: (a, b)")
print("  Target encoder sees:  c = (a+b) mod p")
print("  Label c: YES — fed to target encoder")
print("=" * 70)

torch.manual_seed(SEED)

ctx_enc_D = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)

# Target encoder for single tokens (the label c)
tgt_enc_D = TokenEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
pred_D = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)

# Compile models
ctx_enc_D = torch.compile(ctx_enc_D)
tgt_enc_D = torch.compile(tgt_enc_D)
pred_D = torch.compile(pred_D)

print(f"Context encoder (pair):  {count_params(ctx_enc_D):,} params")
print(f"Target encoder (token):  {count_params(tgt_enc_D):,} params")
print(f"Predictor:               {count_params(pred_D):,} params")

# Data routing: context gets (a,b), target gets c
def get_ctx_D(batch_pairs, batch_targets):
    return batch_pairs  # (a, b)

def get_tgt_D(batch_pairs, batch_targets):
    return batch_targets  # c = (a+b) mod p — THE LABEL

history_D, ctx_enc_D, tgt_ema_D, pred_D = train_jepa_variant(
    "D: Label-JEPA",
    ctx_enc_D, tgt_enc_D, pred_D,
    get_context_input=get_ctx_D,
    get_target_input=get_tgt_D,
    eval_encoder=ctx_enc_D,
)

---
## 7. Head-to-Head Comparison

In [ ]:
approaches = {
    "A: Token Masking\n(truly SSL)": history_A,
    "B: Dropout Aug\n(truly SSL)": history_B,
    "C: Swap\n(truly SSL)": history_C,
    "D: Label-JEPA\n(uses label)": history_D,
}

approach_colors = {
    "A: Token Masking\n(truly SSL)": "#ef4444",
    "B: Dropout Aug\n(truly SSL)": "#f97316",
    "C: Swap\n(truly SSL)": "#8b5cf6",
    "D: Label-JEPA\n(uses label)": "#2563eb",
}

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# ── Panel 1: Grokking curves (validation accuracy) ──
ax = axes[0, 0]
for name, hist in approaches.items():
    ep = [e for e, a in zip(hist["epoch"], hist["val_acc"]) if a is not None]
    va = [a*100 for a in hist["val_acc"] if a is not None]
    ax.plot(ep, va, label=name, color=approach_colors[name], linewidth=2)
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5, label="Chance")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Validation Accuracy (%)", fontsize=12)
ax.set_title("Grokking Comparison: SSL vs Label-JEPA", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="center right")
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 105)

# ── Panel 2: Training accuracy ──
ax = axes[0, 1]
for name, hist in approaches.items():
    ep = [e for e, a in zip(hist["epoch"], hist["train_acc"]) if a is not None]
    ta = [a*100 for a in hist["train_acc"] if a is not None]
    ax.plot(ep, ta, label=name, color=approach_colors[name], linewidth=2)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Training Accuracy (%)", fontsize=12)
ax.set_title("Training Accuracy (Linear Probe)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 105)

# ── Panel 3: Fourier structure ──
ax = axes[1, 0]
for name, hist in approaches.items():
    ax.plot(hist["epoch"], hist["fourier_top5"],
            label=name, color=approach_colors[name], linewidth=1.5)
ax.axhline(y=5/p, color="gray", linestyle=":", alpha=0.5, label=f"Uniform ({5/p:.3f})")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Fourier Top-5 Concentration", fontsize=12)
ax.set_title("Fourier Structure Evolution", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Panel 4: Effective rank ──
ax = axes[1, 1]
for name, hist in approaches.items():
    ax.plot(hist["epoch"], hist["eff_rank"],
            label=name, color=approach_colors[name], linewidth=1.5)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Effective Rank", fontsize=12)
ax.set_title("Representation Dimensionality", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ssl_vs_label_jepa_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Loss Dynamics Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss curves
ax = axes[0]
for name, hist in approaches.items():
    ax.plot(hist["epoch"], hist["jepa_loss"],
            label=name, color=approach_colors[name], linewidth=1, alpha=0.8)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("JEPA Loss (neg cosine sim)", fontsize=12)
ax.set_title("Loss Dynamics", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Uniformity
ax = axes[1]
for name, hist in approaches.items():
    ax.plot(hist["epoch"], hist["uniformity"],
            label=name, color=approach_colors[name], linewidth=1.5)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Uniformity", fontsize=12)
ax.set_title("Uniformity (lower = more uniform on sphere)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ssl_loss_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Final Summary Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

short_names = ["A: Masking", "B: Dropout", "C: Swap", "D: Label"]
bar_colors = ["#ef4444", "#f97316", "#8b5cf6", "#2563eb"]

# Peak validation accuracy
peak_vals = []
for name, hist in approaches.items():
    peak = max(v for v in hist["val_acc"] if v is not None) * 100
    peak_vals.append(peak)

ax = axes[0]
bars = ax.bar(short_names, peak_vals, color=bar_colors, alpha=0.85, edgecolor="black")
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5)
ax.set_ylabel("Peak Validation Accuracy (%)", fontsize=12)
ax.set_title("Does It Grok?", fontsize=13, fontweight="bold")
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, peak_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=11)

# Final Fourier concentration
final_fourier = [hist["fourier_top5"][-1] for hist in approaches.values()]
ax = axes[1]
bars = ax.bar(short_names, final_fourier, color=bar_colors, alpha=0.85, edgecolor="black")
ax.axhline(y=5/p, color="gray", linestyle=":", alpha=0.5, label=f"Uniform ({5/p:.3f})")
ax.set_ylabel("Fourier Top-5 Concentration", fontsize=12)
ax.set_title("Fourier Structure?", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Final effective rank
final_rank = [hist["eff_rank"][-1] for hist in approaches.values()]
ax = axes[2]
bars = ax.bar(short_names, final_rank, color=bar_colors, alpha=0.85, edgecolor="black")
ax.set_ylabel("Effective Rank", fontsize=12)
ax.set_title("Representation Dimensionality", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("ssl_summary_bars.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Detailed Analysis of What SSL Approaches DID Learn

In [ ]:
print("=" * 70)
print("WHAT DID EACH SSL APPROACH LEARN?")
print("=" * 70)

from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import cross_val_score

ssl_encoders = {
    "A: Token Masking": eval_enc_A,
    "B: Dropout": ctx_enc_B,
    "C: Swap": ctx_enc_C,
    "D: Label-JEPA": ctx_enc_D,
}

probe_tasks = {
    "(a+b) mod p": targets.numpy(),
    "a": pairs[:, 0].numpy(),
    "b": pairs[:, 1].numpy(),
    "(a-b) mod p": (pairs[:, 0] - pairs[:, 1]).numpy() % p,
    "(a*b) mod p": (pairs[:, 0].numpy() * pairs[:, 1].numpy()) % p,
}

for enc_name, enc in ssl_encoders.items():
    enc.eval()
    with torch.no_grad():
        z_np = enc(pairs.to(device)).cpu().numpy()

    print(f"\n--- {enc_name} ---")
    for task_name, labels in probe_tasks.items():
        clf = RidgeClassifier(alpha=1.0)
        scores = cross_val_score(clf, z_np, labels, cv=3, scoring="accuracy")
        marker = "✓" if scores.mean() > 0.5 else "✗"
        print(f"  {marker} {task_name:15s}: {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%")

## 11. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(24, 18))
fig.suptitle("Truly Self-Supervised JEPA vs Label-JEPA on Modular Addition",
             fontsize=16, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(3, 4, hspace=0.4, wspace=0.35)

# Row 1: Individual grokking curves
for i, (name, hist) in enumerate(approaches.items()):
    ax = fig.add_subplot(gs[0, i])
    ep_p = [e for e, a in zip(hist["epoch"], hist["val_acc"]) if a is not None]
    va_p = [a*100 for a in hist["val_acc"] if a is not None]
    ta_p = [a*100 for a in hist["train_acc"] if a is not None]
    ax.plot(ep_p, ta_p, color="#2563eb", linewidth=1.5, label="Train")
    ax.plot(ep_p, va_p, color="#f97316", linewidth=1.5, label="Val")
    ax.set_title(name, fontsize=10, fontweight="bold")
    ax.set_ylim(-5, 105)
    ax.set_xlabel("Epoch", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Row 2: Comparative plots
ax = fig.add_subplot(gs[1, 0:2])
for name, hist in approaches.items():
    ep = [e for e, a in zip(hist["epoch"], hist["val_acc"]) if a is not None]
    va = [a*100 for a in hist["val_acc"] if a is not None]
    ax.plot(ep, va, label=name, color=approach_colors[name], linewidth=2)
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5)
ax.set_title("Validation Accuracy Comparison", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Val Accuracy (%)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(-5, 105)

ax = fig.add_subplot(gs[1, 2:4])
for name, hist in approaches.items():
    ax.plot(hist["epoch"], hist["fourier_top5"],
            label=name, color=approach_colors[name], linewidth=1.5)
ax.axhline(y=5/p, color="gray", linestyle=":", alpha=0.5)
ax.set_title("Fourier Structure", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Top-5 Concentration")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Row 3: Bar summaries
ax = fig.add_subplot(gs[2, 0:2])
bars = ax.bar(short_names, peak_vals, color=bar_colors, alpha=0.85, edgecolor="black")
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5)
for bar, val in zip(bars, peak_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=11)
ax.set_title("Peak Validation Accuracy", fontweight="bold")
ax.set_ylim(0, 110); ax.grid(True, alpha=0.3, axis="y")

ax = fig.add_subplot(gs[2, 2:4])
bars = ax.bar(short_names, final_fourier, color=bar_colors, alpha=0.85, edgecolor="black")
ax.axhline(y=5/p, color="gray", linestyle=":", alpha=0.5)
ax.set_title("Final Fourier Concentration", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")

plt.savefig("ssl_jepa_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Conclusions

### The core finding

**Truly self-supervised JEPA does not grok modular addition.**

All three SSL approaches (token masking, dropout augmentation, commutative swap)
fail to develop representations that linearly decode $(a+b) \bmod p$.
Only the Label-JEPA (Approach D), which feeds $c$ to the target encoder, groks.

### Why this matters

This result completes the picture of what the original "JEPA grokking" setup actually is:

1. **The architecture is JEPA** — context encoder, predictor, EMA target encoder,
   cosine similarity loss in latent space. All mechanically identical to I-JEPA.

2. **The data routing is NOT self-supervised** — the label $c$ enters through the
   target encoder. Remove it (Approaches A–C) and grokking disappears.

3. **The label is necessary but the loss function still matters** — supervised
   cross-entropy with the same label produces Fourier circuits; Label-JEPA with
   the same label does not. So the finding from the original notebook stands:
   *the loss function determines the generalization mechanism.*

### The precise characterization

The original setup is best described as:

> **Asymmetric cross-modal latent alignment with EMA targets** — a JEPA architecture
> where the two modalities are (input, label) rather than (view₁, view₂).

It is:
- ✓ Architecturally JEPA
- ✗ Not self-supervised (requires labels)
- ✗ Not supervised (no classification loss)
- ✓ A novel intermediate that isolates the loss function's role

The contribution is not "self-supervised grokking" — it is: **the loss function,
not the label information, determines whether grokking proceeds via Fourier circuits.**

---

*Truly-SSL control experiment by Kurian George.*

In [ ]:
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

for name, hist in approaches.items():
    peak = max(v for v in hist["val_acc"] if v is not None) * 100
    final = [v for v in hist["val_acc"] if v is not None][-1] * 100
    fourier = hist["fourier_top5"][-1]
    rank = hist["eff_rank"][-1]
    grokked = "YES" if peak > 90 else "NO"
    print(f"\n{name}")
    print(f"  Grokked:  {grokked} (peak val: {peak:.1f}%)")
    print(f"  Fourier:  {fourier:.3f} (uniform: {5/p:.3f})")
    print(f"  Eff rank: {rank:.1f}")

print("\n" + "=" * 70)
print("CONCLUSION: Label c is NECESSARY for grokking.")
print("Truly self-supervised JEPA cannot discover modular arithmetic.")
print("The original finding stands: the LOSS FUNCTION (not label access)")
print("determines the generalization MECHANISM (Fourier vs non-Fourier).")
print("=" * 70)

# Part 3: Strong Inductive Bias via Augmentation

**Description:** Can augmentation replace labels? Tests sum-preserving augmentations as implicit task specification.

---

# Strong Inductive Bias SSL-JEPA: Sum-Preserving Augmentations

## The Argument

We established that truly self-supervised JEPA (token masking, dropout, swap) cannot
grok modular addition because the input pair $(a, b)$ does not specify which operation
is being learned. The label $c = (a+b) \bmod p$ is necessary — but *in what form?*

The Label-JEPA feeds $c$ explicitly to the target encoder. But there's a subtler way
to inject the same information: **encode the operation inside the augmentation.**

A **sum-preserving augmentation** maps $(a, b) \to (a+k, b-k) \bmod p$ for random $k$.
Both pairs have the same sum. No label $c$ ever appears. Both encoders see only input
pairs. This is formally SSL — delete every label from the dataset and training proceeds
identically.

But the augmentation *is* the task specification. Designing it required knowing that
addition is the relevant operation. The human encoded the answer into the procedure.

## What This Notebook Tests

| Approach | Context sees | Target sees | Label used? | Operation encoded in? |
|----------|-------------|-------------|-------------|----------------------|
| **E1. Sum-preserving** | $(a, b)$ | $(a{+}k, b{-}k) \bmod p$ | No | Augmentation |
| **E2. Product-preserving** | $(a, b)$ | $(a \cdot r, b \cdot r^{-1}) \bmod p$ | No | Augmentation (wrong task) |
| **E3. Random remapping** | $(a, b)$ | $(\pi(a), \sigma(b))$ for fixed permutations | No | Nothing (control) |
| **D. Label-JEPA** | $(a, b)$ | $c = (a{+}b) \bmod p$ | Yes | Target encoder input |

**E2 (product-preserving)** is the critical control. It uses the *same technique*
(operation-preserving augmentation) but encodes the *wrong operation* (multiplication
instead of addition). If E1 groks addition and E2 does not, it proves the augmentation
is doing the task specification work — the model learns whatever operation the
augmentation encodes.

**E3 (random remapping)** applies fixed random permutations to each operand independently.
This creates a consistent augmentation (same input always maps to the same augmented view)
but one that does not preserve any algebraic structure. It controls for the possibility
that any consistent pairing is sufficient.

## Key Questions

1. Can sum-preserving augmentation JEPA grok modular addition?
2. If yes — does it follow the Fourier pathway or the non-Fourier pathway?
3. Does product-preserving augmentation cause grokking of addition or multiplication?
4. Is the augmentation genuinely functioning as task specification?

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import time
from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import cross_val_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Hyperparameters

In [ ]:
p = 97
train_frac = 0.3

LATENT_DIM = 128
HIDDEN_DIM = 256
PREDICTOR_DIM = 64

EPOCHS = 60_000
LR = 1e-3
WEIGHT_DECAY = 1.0
EMA_DECAY = 0.996

LOG_EVERY = 200
PROBE_EVERY = 500

print(f"Task: (a + b) mod {p}")
print(f"Total pairs: {p**2}, Train: {int(train_frac * p**2)}, Val: {p**2 - int(train_frac * p**2)}")

## 3. Dataset

In [ ]:
pairs = torch.cartesian_prod(torch.arange(p), torch.arange(p))
targets = (pairs[:, 0] + pairs[:, 1]) % p

n = len(pairs)
n_train = int(train_frac * n)
n_val = n - n_train

perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:n_train], perm[n_train:]

train_pairs = pairs[train_idx].to(device)
train_targets = targets[train_idx].to(device)
val_pairs = pairs[val_idx].to(device)
val_targets = targets[val_idx].to(device)

train_loader = DataLoader(
    TensorDataset(train_pairs, train_targets),
    batch_size=n_train, shuffle=True
)

print(f"Train: {n_train} | Val: {n_val}")

# Precompute modular inverse table for product-preserving augmentation
# For prime p, a^(-1) = a^(p-2) mod p (Fermat's little theorem)
mod_inverse = torch.zeros(p, dtype=torch.long)
for i in range(1, p):
    mod_inverse[i] = pow(i, p - 2, p)
mod_inverse = mod_inverse.to(device)

# Precompute fixed random permutations for E3
rng = torch.Generator().manual_seed(SEED + 999)
perm_a = torch.randperm(p, generator=rng)  # fixed permutation for operand a
perm_b = torch.randperm(p, generator=rng)  # fixed permutation for operand b
perm_a = perm_a.to(device)
perm_b = perm_b.to(device)

print(f"Modular inverse table computed (e.g., 3^(-1) mod {p} = {mod_inverse[3].item()})")
print(f"Check: 3 × {mod_inverse[3].item()} mod {p} = {(3 * mod_inverse[3].item()) % p}")

## 4. Augmentation Functions

These are the core of the experiment. Each augmentation defines a different
way to create the target encoder's input from the context encoder's input.

In [ ]:
def sum_preserving_augmentation(batch_pairs):
    """(a, b) → (a+k, b-k) mod p for random k per sample.

    Preserves: (a+b) mod p
    The target encoder sees a different pair with the same sum.

    This IS the task specification — designing this requires knowing
    that addition is the relevant operation.
    """
    batch_size = batch_pairs.shape[0]
    k = torch.randint(1, p, (batch_size,), device=batch_pairs.device)  # k ∈ {1, ..., p-1}
    a_new = (batch_pairs[:, 0] + k) % p
    b_new = (batch_pairs[:, 1] - k) % p
    return torch.stack([a_new, b_new], dim=1)


def product_preserving_augmentation(batch_pairs):
    """(a, b) → (a*r, b*r^(-1)) mod p for random nonzero r per sample.

    Preserves: (a*b) mod p
    The target encoder sees a different pair with the same product.

    This encodes MULTIPLICATION, not addition. If evaluated on the
    addition task, it should NOT help — and may learn multiplication instead.
    """
    batch_size = batch_pairs.shape[0]
    r = torch.randint(1, p, (batch_size,), device=batch_pairs.device)  # r ∈ {1, ..., p-1}
    r_inv = mod_inverse[r]
    a_new = (batch_pairs[:, 0] * r) % p
    b_new = (batch_pairs[:, 1] * r_inv) % p
    return torch.stack([a_new, b_new], dim=1)


def random_remap_augmentation(batch_pairs):
    """(a, b) → (π(a), σ(b)) for fixed random permutations π, σ.

    Preserves: nothing algebraically meaningful.
    Creates a consistent but structurally arbitrary pairing.
    Controls for whether any deterministic mapping is sufficient.
    """
    a_new = perm_a[batch_pairs[:, 0]]
    b_new = perm_b[batch_pairs[:, 1]]
    return torch.stack([a_new, b_new], dim=1)


# Verify augmentations
test_pair = torch.tensor([[3, 4]], device=device)
print("\nAugmentation verification on (3, 4):")
print(f"  Sum = (3+4) mod {p} = {(3+4) % p}")
print(f"  Product = (3×4) mod {p} = {(3*4) % p}")

for _ in range(3):
    aug = sum_preserving_augmentation(test_pair)
    a_, b_ = aug[0].tolist()
    print(f"  Sum-preserving: ({a_}, {b_}) → sum = {(a_+b_) % p}")

for _ in range(3):
    aug = product_preserving_augmentation(test_pair)
    a_, b_ = aug[0].tolist()
    print(f"  Product-preserving: ({a_}, {b_}) → product = {(a_*b_) % p}")

aug = random_remap_augmentation(test_pair)
a_, b_ = aug[0].tolist()
print(f"  Random remap: ({a_}, {b_}) → sum = {(a_+b_) % p}, product = {(a_*b_) % p}")

## 5. Architecture

All approaches use identical architectures — the same pair encoder for both
context and target, plus predictor. The only variable is the augmentation.

In [ ]:
class PairEncoder(nn.Module):
    """Encodes (a, b) pair into latent space.
    Identical to original JEPA context encoder."""
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)
        e = e.view(e.size(0), -1)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class TokenEncoder(nn.Module):
    """Encodes single token into latent space.
    Used only for Label-JEPA control (target encoder receives c)."""
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class Predictor(nn.Module):
    """Maps context latent → predicted target latent."""
    def __init__(self, latent_dim, predictor_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, latent_dim),
        )

    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)


def count_params(model):
    return sum(p_.numel() for p_ in model.parameters() if p_.requires_grad)

_enc = PairEncoder(p, LATENT_DIM, HIDDEN_DIM)
_pred = Predictor(LATENT_DIM, PREDICTOR_DIM)
print(f"Pair Encoder: {count_params(_enc):,} params")
print(f"Predictor:    {count_params(_pred):,} params")
del _enc, _pred

## 6. Metric Functions

In [ ]:
@torch.no_grad()
def compute_geometry(z):
    z_c = z - z.mean(dim=0, keepdim=True)
    cov = (z_c.T @ z_c) / (z.shape[0] - 1)
    eigvals = torch.linalg.eigvalsh(cov).clamp(min=1e-10)

    normed = eigvals / eigvals.sum()
    eff_rank = (1.0 / (normed ** 2).sum()).item()
    top_ratio = (eigvals[-1] / eigvals.sum()).item()

    if z.shape[0] > 2048:
        idx = torch.randperm(z.shape[0])[:2048]
        z_sub = z[idx]
    else:
        z_sub = z
    sq_pdist = torch.cdist(z_sub, z_sub, p=2).pow(2)
    uniformity = torch.log(torch.exp(-2 * sq_pdist).mean() + 1e-10).item()

    return {"effective_rank": eff_rank, "top_eig_ratio": top_ratio, "uniformity": uniformity}


@torch.no_grad()
def compute_fourier_structure(encoder, p, input_data, target_data):
    z_all = encoder(input_data)
    residue_means = torch.zeros(p, z_all.shape[1], device=z_all.device)
    for i in range(p):
        mask = target_data == i
        if mask.sum() > 0:
            residue_means[i] = z_all[mask].mean(dim=0)

    z_np = residue_means.cpu().numpy()
    fft_mag = np.abs(np.fft.fft(z_np, axis=0))
    energy = (fft_mag ** 2).sum(axis=1)
    energy[0] = 0

    total = energy.sum()
    if total < 1e-12:
        return 0.0, energy
    top5_ratio = np.sort(energy)[-5:].sum() / total
    return top5_ratio, energy


@torch.no_grad()
def linear_probe_accuracy(encoder, val_pairs, val_targets, train_pairs, train_targets, p):
    z_train = encoder(train_pairs)
    z_val = encoder(val_pairs)
    y_train = F.one_hot(train_targets, num_classes=p).float()

    reg = 1e-3
    ZtZ = z_train.T @ z_train + reg * torch.eye(z_train.shape[1], device=device)
    ZtY = z_train.T @ y_train
    W = torch.linalg.solve(ZtZ, ZtY)

    train_acc = (z_train @ W).argmax(dim=1).eq(train_targets).float().mean().item()
    val_acc = (z_val @ W).argmax(dim=1).eq(val_targets).float().mean().item()
    return train_acc, val_acc


@torch.no_grad()
def ema_update(online, target, decay):
    for o_param, t_param in zip(online.parameters(), target.parameters()):
        t_param.data.mul_(decay).add_(o_param.data, alpha=1 - decay)


print("Metrics defined.")

## 7. Generic Training Function

Handles all four approaches. The only difference is how the target encoder's
input is constructed.

In [ ]:
def train_augmented_jepa(
    approach_name,
    context_encoder,
    target_encoder,
    predictor,
    augmentation_fn,   # fn(batch_pairs) → augmented pairs for target encoder
    use_label=False,    # if True, target encoder receives label c (Label-JEPA control)
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    ema_decay=EMA_DECAY,
    log_every=LOG_EVERY,
    probe_every=PROBE_EVERY,
    verbose=True,
):
    target_ema = deepcopy(target_encoder)
    for param in target_ema.parameters():
        param.requires_grad = False

    optimizer = optim.AdamW(
        list(context_encoder.parameters()) +
        list(predictor.parameters()) +
        list(target_encoder.parameters()),
        lr=lr, weight_decay=weight_decay,
    )

    history = {
        "epoch": [], "jepa_loss": [],
        "train_acc": [], "val_acc": [],
        "eff_rank": [], "top_eig_ratio": [],
        "uniformity": [], "fourier_top5": [],
    }

    start_time = time.time()

    for epoch in range(epochs):
        context_encoder.train()
        target_encoder.train()
        predictor.train()

        for batch_pairs, batch_targets in train_loader:
            optimizer.zero_grad()

            z_context = context_encoder(batch_pairs)
            z_pred = predictor(z_context)

            with torch.no_grad():
                if use_label:
                    z_target = target_ema(batch_targets)
                else:
                    augmented = augmentation_fn(batch_pairs)
                    z_target = target_ema(augmented)

            loss = -(z_pred * z_target).sum(dim=-1).mean()

            loss.backward()
            optimizer.step()
            ema_update(target_encoder, target_ema, ema_decay)

        if epoch % log_every == 0 or epoch == epochs - 1:
            context_encoder.eval()

            with torch.no_grad():
                z_p = predictor(context_encoder(train_pairs))
                if use_label:
                    z_t = target_ema(train_targets)
                else:
                    z_t = target_ema(augmentation_fn(train_pairs))
                jepa_loss = -(z_p * z_t).sum(dim=-1).mean().item()

            with torch.no_grad():
                z_full = context_encoder(pairs.to(device))
                geo = compute_geometry(z_full)

            f_top5, _ = compute_fourier_structure(
                context_encoder, p, pairs.to(device), targets.to(device)
            )

            history["epoch"].append(epoch)
            history["jepa_loss"].append(jepa_loss)
            history["eff_rank"].append(geo["effective_rank"])
            history["top_eig_ratio"].append(geo["top_eig_ratio"])
            history["uniformity"].append(geo["uniformity"])
            history["fourier_top5"].append(f_top5)

            if epoch % probe_every == 0 or epoch == epochs - 1:
                train_acc, val_acc = linear_probe_accuracy(
                    context_encoder, val_pairs, val_targets,
                    train_pairs, train_targets, p
                )
                history["train_acc"].append(train_acc)
                history["val_acc"].append(val_acc)

                if verbose and epoch % (probe_every * 10) == 0:
                    elapsed = time.time() - start_time
                    print(
                        f"  [{approach_name}] Epoch {epoch:5d} ({elapsed/60:.1f}m) | "
                        f"Loss: {jepa_loss:.4f} | "
                        f"Train: {train_acc*100:.1f}% | Val: {val_acc*100:.1f}% | "
                        f"EffRank: {geo['effective_rank']:.1f} | "
                        f"Fourier: {f_top5:.3f}"
                    )
            else:
                history["train_acc"].append(None)
                history["val_acc"].append(None)

    total_time = time.time() - start_time
    final_val = [v for v in history['val_acc'] if v is not None][-1]
    peak_val = max(v for v in history['val_acc'] if v is not None)
    print(f"  [{approach_name}] Done in {total_time/60:.1f}m | "
          f"Peak val: {peak_val*100:.1f}% | Final val: {final_val*100:.1f}%")

    return history, context_encoder, target_ema, predictor


print("Training function defined.")

---
## Approach E1: Sum-Preserving Augmentation

$(a, b) \to (a+k, b-k) \bmod p$. Preserves additive equivalence.
**Formally SSL. Encodes addition in the augmentation.**

In [ ]:
print("=" * 70)
print("E1: SUM-PRESERVING AUGMENTATION JEPA")
print("  Context: (a, b)  |  Target: (a+k, b-k) mod p  |  Label: NEVER USED")
print("  Encodes: addition (the correct operation)")
print("=" * 70)

torch.manual_seed(SEED)

ctx_enc_E1 = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
tgt_enc_E1 = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
pred_E1 = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)

# Compile models
ctx_enc_E1 = torch.compile(ctx_enc_E1)
tgt_enc_E1 = torch.compile(tgt_enc_E1)
pred_E1 = torch.compile(pred_E1)

history_E1, ctx_enc_E1, tgt_ema_E1, pred_E1 = train_augmented_jepa(
    "E1: Sum-Preserving",
    ctx_enc_E1, tgt_enc_E1, pred_E1,
    augmentation_fn=sum_preserving_augmentation,
    use_label=False,
)

---
## Approach E2: Product-Preserving Augmentation

$(a, b) \to (a \cdot r, b \cdot r^{-1}) \bmod p$. Preserves multiplicative equivalence.
**Formally SSL. Encodes multiplication — the WRONG operation.**

If evaluated on the *addition* task via linear probe, this should fail.
But it might succeed on multiplication — which would prove the augmentation
is doing task specification.

In [ ]:
print("\n" + "=" * 70)
print("E2: PRODUCT-PRESERVING AUGMENTATION JEPA")
print("  Context: (a, b)  |  Target: (a*r, b*r^(-1)) mod p  |  Label: NEVER USED")
print("  Encodes: multiplication (the WRONG operation for addition task)")
print("=" * 70)

torch.manual_seed(SEED)

ctx_enc_E2 = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
tgt_enc_E2 = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
pred_E2 = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)

# Compile models
ctx_enc_E2 = torch.compile(ctx_enc_E2)
tgt_enc_E2 = torch.compile(tgt_enc_E2)
pred_E2 = torch.compile(pred_E2)

history_E2, ctx_enc_E2, tgt_ema_E2, pred_E2 = train_augmented_jepa(
    "E2: Product-Preserving",
    ctx_enc_E2, tgt_enc_E2, pred_E2,
    augmentation_fn=product_preserving_augmentation,
    use_label=False,
)

---
## Approach E3: Random Remapping (Negative Control)

$(a, b) \to (\pi(a), \sigma(b))$ for fixed random permutations.
**No algebraic structure preserved. Controls for any consistent pairing.**

In [ ]:
print("\n" + "=" * 70)
print("E3: RANDOM REMAPPING (negative control)")
print("  Context: (a, b)  |  Target: (π(a), σ(b))  |  Label: NEVER USED")
print("  Encodes: nothing meaningful")
print("=" * 70)

torch.manual_seed(SEED)

ctx_enc_E3 = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
tgt_enc_E3 = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
pred_E3 = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)

# Compile models
ctx_enc_E3 = torch.compile(ctx_enc_E3)
tgt_enc_E3 = torch.compile(tgt_enc_E3)
pred_E3 = torch.compile(pred_E3)

history_E3, ctx_enc_E3, tgt_ema_E3, pred_E3 = train_augmented_jepa(
    "E3: Random Remap",
    ctx_enc_E3, tgt_enc_E3, pred_E3,
    augmentation_fn=random_remap_augmentation,
    use_label=False,
)

---
## Approach D: Label-JEPA (Positive Control)

The original setup. Context sees $(a, b)$, target sees $c = (a+b) \bmod p$.
**Not SSL. Known to grok.**

In [ ]:
print("\n" + "=" * 70)
print("D: LABEL-JEPA (positive control)")
print("  Context: (a, b)  |  Target: c = (a+b) mod p  |  Label: YES")
print("=" * 70)

torch.manual_seed(SEED)

ctx_enc_D = PairEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
tgt_enc_D = TokenEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
pred_D = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)

# Compile models
ctx_enc_D = torch.compile(ctx_enc_D)
tgt_enc_D = torch.compile(tgt_enc_D)
pred_D = torch.compile(pred_D)

history_D, ctx_enc_D, tgt_ema_D, pred_D = train_augmented_jepa(
    "D: Label-JEPA",
    ctx_enc_D, tgt_enc_D, pred_D,
    augmentation_fn=None,
    use_label=True,
)

---
## 8. Head-to-Head Comparison: Grokking Curves

In [ ]:
approaches = {
    "E1: Sum-Preserving\n(SSL, correct op)": history_E1,
    "E2: Product-Preserving\n(SSL, wrong op)": history_E2,
    "E3: Random Remap\n(SSL, no structure)": history_E3,
    "D: Label-JEPA\n(not SSL, control)": history_D,
}

colors = {
    "E1: Sum-Preserving\n(SSL, correct op)": "#16a34a",
    "E2: Product-Preserving\n(SSL, wrong op)": "#dc2626",
    "E3: Random Remap\n(SSL, no structure)": "#9ca3af",
    "D: Label-JEPA\n(not SSL, control)": "#2563eb",
}

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Panel 1: Validation accuracy (addition task)
ax = axes[0, 0]
for name, hist in approaches.items():
    ep = [e for e, a in zip(hist["epoch"], hist["val_acc"]) if a is not None]
    va = [a*100 for a in hist["val_acc"] if a is not None]
    ax.plot(ep, va, label=name, color=colors[name], linewidth=2)
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5, label="Chance")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Validation Accuracy (%)", fontsize=12)
ax.set_title("Grokking on (a+b) mod p: Does Augmentation Work?",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 105)

# Panel 2: Training accuracy
ax = axes[0, 1]
for name, hist in approaches.items():
    ep = [e for e, a in zip(hist["epoch"], hist["train_acc"]) if a is not None]
    ta = [a*100 for a in hist["train_acc"] if a is not None]
    ax.plot(ep, ta, label=name, color=colors[name], linewidth=2)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Training Accuracy (%)", fontsize=12)
ax.set_title("Training Accuracy (Linear Probe on Addition)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 105)

# Panel 3: Fourier structure
ax = axes[1, 0]
for name, hist in approaches.items():
    ax.plot(hist["epoch"], hist["fourier_top5"],
            label=name, color=colors[name], linewidth=1.5)
ax.axhline(y=5/p, color="gray", linestyle=":", alpha=0.5, label=f"Uniform ({5/p:.3f})")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Fourier Top-5 Concentration", fontsize=12)
ax.set_title("Fourier Structure Evolution", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 4: Effective rank
ax = axes[1, 1]
for name, hist in approaches.items():
    ax.plot(hist["epoch"], hist["eff_rank"],
            label=name, color=colors[name], linewidth=1.5)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Effective Rank", fontsize=12)
ax.set_title("Representation Dimensionality", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("augmentation_jepa_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Cross-Task Linear Probing

The critical test for E2 (product-preserving): does it learn *multiplication*
even though we evaluate on *addition*?

We probe each encoder for both addition and multiplication accuracy.

In [ ]:
print("=" * 70)
print("CROSS-TASK LINEAR PROBING")
print("What operation did each approach actually learn?")
print("=" * 70)

encoders = {
    "E1: Sum-Preserving": ctx_enc_E1,
    "E2: Product-Preserving": ctx_enc_E2,
    "E3: Random Remap": ctx_enc_E3,
    "D: Label-JEPA": ctx_enc_D,
}

pairs_np = pairs.numpy()
probe_tasks = {
    "(a+b) mod p": targets.numpy(),
    "(a*b) mod p": (pairs_np[:, 0] * pairs_np[:, 1]) % p,
    "a": pairs_np[:, 0],
    "b": pairs_np[:, 1],
    "(a-b) mod p": (pairs_np[:, 0] - pairs_np[:, 1]) % p,
    "(a²+b²) mod p": (pairs_np[:, 0]**2 + pairs_np[:, 1]**2) % p,
}

results_table = {}

for enc_name, enc in encoders.items():
    enc.eval()
    with torch.no_grad():
        z_np = enc(pairs.to(device)).cpu().numpy()

    print(f"\n--- {enc_name} ---")
    results_table[enc_name] = {}
    for task_name, labels in probe_tasks.items():
        clf = RidgeClassifier(alpha=1.0)
        scores = cross_val_score(clf, z_np, labels, cv=3, scoring="accuracy")
        acc = scores.mean() * 100
        results_table[enc_name][task_name] = acc
        marker = "✓" if acc > 50 else "·" if acc > 10 else "✗"
        print(f"  {marker} {task_name:18s}: {acc:.1f}% ± {scores.std()*100:.1f}%")

## 10. Cross-Task Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart: addition accuracy
ax = axes[0]
short_names = ["E1: Sum", "E2: Product", "E3: Random", "D: Label"]
add_accs = [results_table[n]["(a+b) mod p"] for n in encoders.keys()]
bar_colors_list = ["#16a34a", "#dc2626", "#9ca3af", "#2563eb"]
bars = ax.bar(short_names, add_accs, color=bar_colors_list, alpha=0.85, edgecolor="black")
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5)
for bar, val in zip(bars, add_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=11)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("Addition Task: (a+b) mod p", fontsize=13, fontweight="bold")
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis="y")

# Bar chart: multiplication accuracy
ax = axes[1]
mul_accs = [results_table[n]["(a*b) mod p"] for n in encoders.keys()]
bars = ax.bar(short_names, mul_accs, color=bar_colors_list, alpha=0.85, edgecolor="black")
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5)
for bar, val in zip(bars, mul_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=11)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("Multiplication Task: (a*b) mod p", fontsize=13, fontweight="bold")
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("cross_task_probing.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. PCA Visualizations

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 10))

enc_list = [
    ("E1: Sum-Preserving", ctx_enc_E1),
    ("E2: Product-Preserving", ctx_enc_E2),
    ("E3: Random Remap", ctx_enc_E3),
    ("D: Label-JEPA", ctx_enc_D),
]

for i, (name, enc) in enumerate(enc_list):
    enc.eval()
    with torch.no_grad():
        z = enc(pairs.to(device)).cpu().numpy()

    pca = PCA(n_components=2)
    z_2d = pca.fit_transform(z)

    # Top row: colored by (a+b) mod p
    ax = axes[0, i]
    sc = ax.scatter(z_2d[:, 0], z_2d[:, 1],
                    c=targets.numpy(), cmap="hsv", s=1, alpha=0.3)
    ax.set_title(f"{name}\nColored by (a+b) mod p", fontsize=10, fontweight="bold")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=9)
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=9)

    # Bottom row: colored by (a*b) mod p
    ax = axes[1, i]
    mul_targets = (pairs[:, 0].numpy() * pairs[:, 1].numpy()) % p
    sc = ax.scatter(z_2d[:, 0], z_2d[:, 1],
                    c=mul_targets, cmap="hsv", s=1, alpha=0.3)
    ax.set_title(f"{name}\nColored by (a*b) mod p", fontsize=10, fontweight="bold")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=9)
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=9)

plt.tight_layout()
plt.savefig("augmentation_pca.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(24, 20))
fig.suptitle("Strong Inductive Bias SSL-JEPA: Can Augmentations Replace Labels?",
             fontsize=16, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(4, 4, hspace=0.45, wspace=0.35)

# Row 1: Individual grokking curves
for i, (name, hist) in enumerate(approaches.items()):
    ax = fig.add_subplot(gs[0, i])
    ep = [e for e, a in zip(hist["epoch"], hist["val_acc"]) if a is not None]
    va = [a*100 for a in hist["val_acc"] if a is not None]
    ta = [a*100 for a in hist["train_acc"] if a is not None]
    ax.plot(ep, ta, color="#2563eb", linewidth=1.5, label="Train")
    ax.plot(ep, va, color="#f97316", linewidth=1.5, label="Val")
    short = name.split('\n')[0]
    ax.set_title(short, fontsize=10, fontweight="bold")
    ax.set_ylim(-5, 105)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Epoch", fontsize=8)

# Row 2: Combined comparison
ax = fig.add_subplot(gs[1, 0:2])
for name, hist in approaches.items():
    ep = [e for e, a in zip(hist["epoch"], hist["val_acc"]) if a is not None]
    va = [a*100 for a in hist["val_acc"] if a is not None]
    ax.plot(ep, va, label=name.split('\n')[0], color=colors[name], linewidth=2)
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5)
ax.set_title("Val Accuracy on Addition", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(-5, 105)

ax = fig.add_subplot(gs[1, 2:4])
for name, hist in approaches.items():
    ax.plot(hist["epoch"], hist["fourier_top5"],
            label=name.split('\n')[0], color=colors[name], linewidth=1.5)
ax.axhline(y=5/p, color="gray", linestyle=":", alpha=0.5)
ax.set_title("Fourier Structure", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Top-5 Concentration")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Row 3: Cross-task probing
ax = fig.add_subplot(gs[2, 0:2])
x_pos = np.arange(len(short_names))
width = 0.35
add_accs_arr = [results_table[n]["(a+b) mod p"] for n in encoders.keys()]
mul_accs_arr = [results_table[n]["(a*b) mod p"] for n in encoders.keys()]
ax.bar(x_pos - width/2, add_accs_arr, width, label="(a+b) mod p",
       color=bar_colors_list, alpha=0.85, edgecolor="black")
ax.bar(x_pos + width/2, mul_accs_arr, width, label="(a*b) mod p",
       color=bar_colors_list, alpha=0.4, edgecolor="black", hatch="//")
ax.set_xticks(x_pos)
ax.set_xticklabels(short_names, fontsize=9)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Cross-Task Probing: Which Operation Was Learned?", fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3, axis="y"); ax.set_ylim(0, 110)

# Row 3 right: Loss dynamics
ax = fig.add_subplot(gs[2, 2:4])
for name, hist in approaches.items():
    ax.plot(hist["epoch"], hist["jepa_loss"],
            label=name.split('\n')[0], color=colors[name], linewidth=1, alpha=0.8)
ax.set_title("JEPA Loss", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Row 4: Summary bars
ax = fig.add_subplot(gs[3, 0:2])
peak_vals = [max(v for v in h["val_acc"] if v is not None)*100 for h in approaches.values()]
bars = ax.bar(short_names, peak_vals, color=bar_colors_list, alpha=0.85, edgecolor="black")
ax.axhline(y=100/p, color="gray", linestyle=":", alpha=0.5)
for bar, val in zip(bars, peak_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=11)
ax.set_title("Peak Validation Accuracy (Addition)", fontweight="bold")
ax.set_ylim(0, 110); ax.grid(True, alpha=0.3, axis="y")

ax = fig.add_subplot(gs[3, 2:4])
final_fourier = [h["fourier_top5"][-1] for h in approaches.values()]
bars = ax.bar(short_names, final_fourier, color=bar_colors_list, alpha=0.85, edgecolor="black")
ax.axhline(y=5/p, color="gray", linestyle=":", alpha=0.5)
ax.set_title("Final Fourier Concentration", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")

plt.savefig("augmentation_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Conclusions

### What the results show

**E1 (Sum-preserving)**: [Result after running — does SSL with correct inductive bias grok addition?]

**E2 (Product-preserving)**: [Does it learn multiplication instead? This is the key
test of whether the augmentation functions as task specification.]

**E3 (Random remap)**: [Expected: fails on everything. Confirms that consistent pairing
alone is not sufficient — algebraic structure in the augmentation matters.]

**D (Label-JEPA)**: [Expected: groks. Positive control.]

### Interpretation

If E1 groks and E2 learns multiplication instead of addition:
> The augmentation is functioning as **task specification**. The model learns
> whatever operation the augmentation preserves. "SSL" here is a formality —
> the human designer injected the answer into the augmentation.

If E1 fails to grok:
> Even knowing the equivalence relation is insufficient — the model needs
> explicit class labels (the actual value of $c$), not just "these pairs
> are equivalent." This would mean the label's role goes beyond task
> specification into providing **grounding** for the latent space.

Either result deepens the understanding of what makes the Label-JEPA work.

### The taxonomy of task specification

| Method | Task specified by | SSL? | Task info in... |
|--------|------------------|------|-----------------|
| Weak SSL (masking, dropout) | Nothing | Yes | Nowhere |
| Strong SSL (sum-preserving aug) | Augmentation design | Technically yes | Code |
| Label-JEPA | Demonstrated examples | No | Data |
| Supervised | Loss function | No | Data + Loss |

The key insight: **task specification is always present in any method that
successfully learns an algebraic operation. The only question is where it hides.**
SSL doesn't eliminate task specification — it moves it from the data into the
procedure, where it becomes invisible and unquestionable.

---

*Strong inductive bias experiment by Kurian George.*

In [ ]:
print("=" * 70)
print("FINAL SUMMARY: STRONG INDUCTIVE BIAS SSL-JEPA")
print("=" * 70)

for name, hist in approaches.items():
    peak = max(v for v in hist["val_acc"] if v is not None) * 100
    final = [v for v in hist["val_acc"] if v is not None][-1] * 100
    fourier = hist["fourier_top5"][-1]
    print(f"\n{name.split(chr(10))[0]}")
    print(f"  Peak val (addition): {peak:.1f}%")
    print(f"  Fourier top-5:       {fourier:.3f}")

print("\n--- Cross-Task Summary ---")
for enc_name in encoders.keys():
    add_acc = results_table[enc_name]["(a+b) mod p"]
    mul_acc = results_table[enc_name]["(a*b) mod p"]
    print(f"  {enc_name:25s} | Add: {add_acc:.1f}% | Mul: {mul_acc:.1f}%")

print("\n" + "=" * 70)
print("The augmentation IS the task specification.")
print("SSL with strong inductive bias is label-free in form,")
print("but label-dependent in substance.")
print("=" * 70)

# Part 4: Prediction Accuracy & Memorization Dynamics

**Description:** Direct JEPA prediction accuracy, memorization dynamics, linear probe vs direct prediction timeline

---

# JEPA Grokking: Prediction Accuracy & Memorization Dynamics

## The Missing Evidence

The original grokking notebook demonstrated that linear probe accuracy on the context encoder
rises from chance to 100% — but that only shows the *representation* becomes linearly separable.
It doesn't prove the **JEPA objective itself** generalized.

This notebook answers three critical questions:

**Experiment 1: Direct Prediction Accuracy**
Does the predictor actually output the correct target latent on unseen pairs?
If yes → genuine self-supervised grokking. If near chance → the JEPA loss only shaped
representations for a separate classifier, and "self-supervised grokking" needs qualification.

**Experiment 2: Memorization Dynamics in Latent Space**  
What's being memorized during the first phase? Track cosine similarity between
predictor output and target code on training vs validation pairs over time.
This directly visualizes the memorization → generalization transition in JEPA's own objective.

**Experiment 8 (bonus): Linear Probe vs Direct Prediction Timeline**  
Does linear probe accuracy rise before, after, or simultaneously with direct JEPA prediction?
This reveals whether the JEPA objective drives the solution or merely enables it.

### Key Hyperparameters
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| p | 97 | Prime, standard modular arithmetic task |
| train_frac | 0.3 | 30% train, 70% validation |
| latent_dim | 128 | Context/target latent dimension |
| hidden_dim | 256 | Encoder hidden layer width |
| predictor_dim | 64 | Predictor bottleneck |
| weight_decay | 1.0 | Strong regularization (enables grokking) |
| EMA decay | 0.996 | Target encoder momentum |
| epochs | 100,000 | Full training budget |
| eval_every | 500 | 200 evaluation checkpoints |

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
import os
from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader

# ── Configuration ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Task
p = 97
train_frac = 0.3

# Architecture
LATENT_DIM = 128
HIDDEN_DIM = 256
PREDICTOR_DIM = 64

# Training
EPOCHS = 60_000
LR = 1e-3
WEIGHT_DECAY = 1.0
EMA_DECAY = 0.996

# Evaluation schedule: every 500 epochs → 200 checkpoints
EVAL_EVERY = 500

print(f'Task: (a+b) mod {p}')
print(f'Train fraction: {train_frac}')
print(f'Total epochs: {EPOCHS:,}')
print(f'Eval checkpoints: {EPOCHS // EVAL_EVERY}')

## Data Preparation

In [ ]:
def generate_data(seed=SEED):
    pairs = torch.cartesian_prod(torch.arange(p), torch.arange(p))
    targets = (pairs[:, 0] + pairs[:, 1]) % p
    n = len(pairs)
    n_train = int(train_frac * n)
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    train_idx, val_idx = perm[:n_train], perm[n_train:]
    return (
        pairs.to(device), targets.to(device),
        pairs[train_idx].to(device), targets[train_idx].to(device),
        pairs[val_idx].to(device), targets[val_idx].to(device),
    )

pairs, targets, train_pairs, train_targets, val_pairs, val_targets = generate_data(SEED)

n_train = len(train_pairs)
n_val = len(val_pairs)
print(f'Total pairs: {len(pairs)}')
print(f'Train: {n_train} ({n_train/len(pairs)*100:.1f}%)')
print(f'Val:   {n_val} ({n_val/len(pairs)*100:.1f}%)')
print(f'Chance accuracy: {100/p:.2f}%')

## Model Definitions

In [ ]:
class ContextEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )
    def forward(self, x):
        e = self.emb(x)
        e = e.view(e.size(0), -1)
        return F.normalize(self.net(e), dim=-1)

class TargetEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )
    def forward(self, x):
        e = self.emb(x)
        return F.normalize(self.net(e), dim=-1)

class Predictor(nn.Module):
    def __init__(self, latent_dim, predictor_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, latent_dim),
        )
    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)

# Initialize
context_enc = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
target_enc  = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
predictor   = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
target_enc_ema = deepcopy(target_enc)
for param in target_enc_ema.parameters():
    param.requires_grad = False

# Compile models for faster training
context_enc = torch.compile(context_enc)
target_enc = torch.compile(target_enc)
predictor = torch.compile(predictor)
target_enc_ema = torch.compile(target_enc_ema)

n_params = sum(p.numel() for p in context_enc.parameters())
n_params += sum(p.numel() for p in predictor.parameters())
n_params += sum(p.numel() for p in target_enc.parameters())
print(f'Total trainable parameters: {n_params:,}')

## Training with Full Diagnostic Tracking

At every evaluation checkpoint we compute:

1. **JEPA cosine similarity** between `z_pred` and `z_target` (train & val)
2. **Nearest-target-code accuracy** — does `argmax_c (z_pred · target_code[c]) == true_c`? (train & val)
3. **Linear probe accuracy** — closed-form ridge regression on context encoder (train & val)
4. **Wrong-code margin** — cosine distance to nearest *wrong* target code (train & val)

This is the complete picture. No post-hoc analysis needed — we see everything as it happens.

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────

@torch.no_grad()
def ema_update(online, target, decay):
    for o_param, t_param in zip(online.parameters(), target.parameters()):
        t_param.data.mul_(decay).add_(o_param.data, alpha=1 - decay)

@torch.no_grad()
def linear_probe_accuracy(encoder, val_pairs, val_targets, train_pairs, train_targets, p):
    """Closed-form ridge regression probe."""
    z_train = encoder(train_pairs)
    z_val = encoder(val_pairs)
    y_train = F.one_hot(train_targets, num_classes=p).float()
    reg = 1e-3
    ZtZ = z_train.T @ z_train + reg * torch.eye(z_train.shape[1], device=device)
    ZtY = z_train.T @ y_train
    W = torch.linalg.solve(ZtZ, ZtY)
    train_acc = (z_train @ W).argmax(dim=1).eq(train_targets).float().mean().item()
    val_acc = (z_val @ W).argmax(dim=1).eq(val_targets).float().mean().item()
    return train_acc, val_acc

@torch.no_grad()
def jepa_diagnostics(context_enc, predictor, target_enc_ema, pairs, tgts):
    """Compute all JEPA-level metrics for a set of (input, target) pairs.

    Returns dict with:
      - cos_mean: mean cosine(z_pred, z_target) — the JEPA objective value
      - cos_std: std of per-sample cosine
      - nearest_code_acc: argmax_c (z_pred · code_c) == true_c
      - wrong_code_margin: cos(z_pred, correct_code) - cos(z_pred, nearest_wrong_code)
    """
    z_ctx = context_enc(pairs)
    z_pred = predictor(z_ctx)
    z_tgt = target_enc_ema(tgts)

    # 1. Cosine similarity (JEPA objective)
    cos_sim = (z_pred * z_tgt).sum(dim=-1)

    # 2. Nearest-target-code accuracy
    target_codes = target_enc_ema(torch.arange(p, device=device))  # (p, d)
    sim_to_all = z_pred @ target_codes.T  # (N, p)
    nearest_code_acc = sim_to_all.argmax(dim=1).eq(tgts).float().mean().item()

    # 3. Wrong-code margin
    # cos(z_pred, correct) - max_{c != correct} cos(z_pred, c)
    correct_sim = sim_to_all[torch.arange(len(tgts)), tgts]  # (N,)
    # Mask correct class with -inf to find best wrong class
    sim_masked = sim_to_all.clone()
    sim_masked[torch.arange(len(tgts)), tgts] = -float('inf')
    best_wrong_sim = sim_masked.max(dim=1).values  # (N,)
    margin = correct_sim - best_wrong_sim  # positive = correct class is closest

    return {
        'cos_mean': cos_sim.mean().item(),
        'cos_std': cos_sim.std().item(),
        'nearest_code_acc': nearest_code_acc,
        'margin_mean': margin.mean().item(),
        'margin_std': margin.std().item(),
        'frac_positive_margin': (margin > 0).float().mean().item(),
    }

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────

optimizer = optim.AdamW(
    list(context_enc.parameters()) + list(predictor.parameters()) + list(target_enc.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)

train_loader = DataLoader(
    TensorDataset(train_pairs, train_targets),
    batch_size=len(train_pairs), shuffle=True
)

# Storage for all tracked metrics
history = {
    'epoch': [],
    'loss': [],
    # JEPA direct metrics (Exp 1 & 2)
    'train_cos_mean': [], 'train_cos_std': [],
    'val_cos_mean': [],   'val_cos_std': [],
    'train_nearest_acc': [], 'val_nearest_acc': [],
    'train_margin_mean': [], 'val_margin_mean': [],
    'train_margin_frac_pos': [], 'val_margin_frac_pos': [],
    # Linear probe (Exp 8)
    'train_probe_acc': [], 'val_probe_acc': [],
}

print(f'Training for {EPOCHS:,} epochs, evaluating every {EVAL_EVERY}...')
print(f'Expected eval points: {EPOCHS // EVAL_EVERY}')
print()

start_time = time.time()
last_print_time = start_time

for epoch in range(EPOCHS):
    context_enc.train()
    target_enc.train()
    predictor.train()

    for bp, bt in train_loader:
        optimizer.zero_grad()
        z_ctx = context_enc(bp)
        z_pred = predictor(z_ctx)
        with torch.no_grad():
            z_tgt = target_enc_ema(bt)
        loss = -(z_pred * z_tgt).sum(dim=-1).mean()
        loss.backward()
        optimizer.step()
        ema_update(target_enc, target_enc_ema, EMA_DECAY)

    # ── Evaluation checkpoint ──────────────────────────────────────────
    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS - 1:
        context_enc.eval()
        target_enc.eval()
        predictor.eval()

        # JEPA direct metrics
        train_diag = jepa_diagnostics(context_enc, predictor, target_enc_ema,
                                      train_pairs, train_targets)
        val_diag = jepa_diagnostics(context_enc, predictor, target_enc_ema,
                                    val_pairs, val_targets)

        # Linear probe
        train_probe, val_probe = linear_probe_accuracy(
            context_enc, val_pairs, val_targets, train_pairs, train_targets, p)

        # Store everything
        history['epoch'].append(epoch)
        history['loss'].append(loss.item())

        history['train_cos_mean'].append(train_diag['cos_mean'])
        history['train_cos_std'].append(train_diag['cos_std'])
        history['val_cos_mean'].append(val_diag['cos_mean'])
        history['val_cos_std'].append(val_diag['cos_std'])

        history['train_nearest_acc'].append(train_diag['nearest_code_acc'])
        history['val_nearest_acc'].append(val_diag['nearest_code_acc'])

        history['train_margin_mean'].append(train_diag['margin_mean'])
        history['val_margin_mean'].append(val_diag['margin_mean'])
        history['train_margin_frac_pos'].append(train_diag['frac_positive_margin'])
        history['val_margin_frac_pos'].append(val_diag['frac_positive_margin'])

        history['train_probe_acc'].append(train_probe)
        history['val_probe_acc'].append(val_probe)

        # Print periodically (not every checkpoint — too noisy)
        now = time.time()
        if epoch % (EVAL_EVERY * 10) == 0 or epoch == EPOCHS - 1:
            elapsed = (now - start_time) / 60
            print(f'Epoch {epoch:6d} [{elapsed:5.1f}m] | '
                  f'Loss {loss.item():+.4f} | '
                  f'JEPA cos: train={train_diag["cos_mean"]:.3f} val={val_diag["cos_mean"]:.3f} | '
                  f'Nearest: train={train_diag["nearest_code_acc"]*100:.1f}% val={val_diag["nearest_code_acc"]*100:.1f}% | '
                  f'Probe: train={train_probe*100:.1f}% val={val_probe*100:.1f}%')

elapsed_total = (time.time() - start_time) / 60
print(f'\nTraining complete in {elapsed_total:.1f} minutes.')
print(f'Collected {len(history["epoch"])} evaluation checkpoints.')

In [ ]:
# Save checkpoint and history for reproducibility
torch.save({
    'context_enc': context_enc.state_dict(),
    'target_enc_ema': target_enc_ema.state_dict(),
    'predictor': predictor.state_dict(),
    'history': history,
}, 'jepa_grokking_checkpoint.pt')
print('Checkpoint saved: jepa_grokking_checkpoint.pt')

## Experiment 2: Memorization Dynamics in Latent Space

This is the key visualization. We plot the JEPA objective (cosine similarity between
predictor output and target code) on training vs validation pairs over the full training run.

**What we expect to see:**
- **Early:** training cosine high, validation cosine low → predictor memorizes training pairs
- **Late:** both high → predictor generalizes to the full equivalence class structure
- **The gap closing** is the grokking event in JEPA's own objective space

In [ ]:
epochs = np.array(history['epoch'])

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── Panel 1: JEPA Cosine Similarity (the objective) ─────────────────────
ax = axes[0, 0]
ax.plot(epochs, history['train_cos_mean'], 'b-', linewidth=1.5, label='Train', alpha=0.8)
ax.plot(epochs, history['val_cos_mean'], 'r-', linewidth=1.5, label='Validation', alpha=0.8)
# Shade ±1σ
train_upper = np.array(history['train_cos_mean']) + np.array(history['train_cos_std'])
train_lower = np.array(history['train_cos_mean']) - np.array(history['train_cos_std'])
val_upper = np.array(history['val_cos_mean']) + np.array(history['val_cos_std'])
val_lower = np.array(history['val_cos_mean']) - np.array(history['val_cos_std'])
ax.fill_between(epochs, train_lower, train_upper, alpha=0.15, color='blue')
ax.fill_between(epochs, val_lower, val_upper, alpha=0.15, color='red')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cosine Similarity')
ax.set_title('JEPA Objective: cos(z_pred, z_target)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(-0.1, 1.05)

# ── Panel 2: Nearest Target Code Accuracy ──────────────────────────────
ax = axes[0, 1]
ax.plot(epochs, [a*100 for a in history['train_nearest_acc']], 'b-', linewidth=1.5, label='Train')
ax.plot(epochs, [a*100 for a in history['val_nearest_acc']], 'r-', linewidth=1.5, label='Validation')
ax.axhline(y=100/p, color='gray', linestyle='--', alpha=0.5, label=f'Chance ({100/p:.1f}%)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Direct Prediction: argmax_c (z_pred · code_c)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# ── Panel 3: Wrong-Code Margin ─────────────────────────────────────────
ax = axes[1, 0]
ax.plot(epochs, history['train_margin_mean'], 'b-', linewidth=1.5, label='Train margin')
ax.plot(epochs, history['val_margin_mean'], 'r-', linewidth=1.5, label='Val margin')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Margin (correct − best wrong)')
ax.set_title('Wrong-Code Margin', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# ── Panel 4: Loss curve ────────────────────────────────────────────────
ax = axes[1, 1]
ax.plot(epochs, history['loss'], 'k-', linewidth=1, alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('JEPA Loss (negative cosine)')
ax.set_title('Training Loss', fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Experiment 2: Memorization → Generalization Dynamics in JEPA Objective',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('exp2_memorization_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Find the grokking transition ───────────────────────────────────────
val_acc_arr = np.array(history['val_nearest_acc'])
train_acc_arr = np.array(history['train_nearest_acc'])

# Grokking epoch: first time val nearest-code accuracy exceeds 90%
grok_mask = val_acc_arr > 0.9
if grok_mask.any():
    grok_epoch = epochs[grok_mask][0]
    grok_idx = np.where(grok_mask)[0][0]
    print(f'\nGrokking event (val nearest-code acc > 90%): epoch {grok_epoch}')
    print(f'  At grokking:')
    print(f'    Train cos:  {history["train_cos_mean"][grok_idx]:.4f}')
    print(f'    Val cos:    {history["val_cos_mean"][grok_idx]:.4f}')
    print(f'    Train acc:  {train_acc_arr[grok_idx]*100:.1f}%')
    print(f'    Val acc:    {val_acc_arr[grok_idx]*100:.1f}%')
    print(f'    Train margin: {history["train_margin_mean"][grok_idx]:.4f}')
    print(f'    Val margin:   {history["val_margin_mean"][grok_idx]:.4f}')
else:
    grok_epoch = None
    print('\nNo grokking detected (val nearest-code acc never exceeded 90%)')

## Experiment 2b: Zoomed Dynamics Around Grokking Transition

Zoom into the window around the grokking event to see the fine-grained transition dynamics.

In [ ]:
if grok_epoch is not None:
    # Zoom window: 20% before grokking to 20% after (or end of training)
    window_before = int(grok_epoch * 0.5)
    window_after = min(int(grok_epoch * 1.5), EPOCHS)
    mask = (epochs >= grok_epoch - window_before) & (epochs <= window_after)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # JEPA cosine zoom
    ax = axes[0]
    ax.plot(epochs[mask], np.array(history['train_cos_mean'])[mask], 'b-o', markersize=2, label='Train')
    ax.plot(epochs[mask], np.array(history['val_cos_mean'])[mask], 'r-o', markersize=2, label='Val')
    ax.axvline(x=grok_epoch, color='orange', linestyle='--', alpha=0.7, label=f'Grok @ {grok_epoch}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Cosine Similarity')
    ax.set_title('JEPA Cosine (zoomed)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Nearest-code accuracy zoom
    ax = axes[1]
    ax.plot(epochs[mask], train_acc_arr[mask]*100, 'b-o', markersize=2, label='Train')
    ax.plot(epochs[mask], val_acc_arr[mask]*100, 'r-o', markersize=2, label='Val')
    ax.axvline(x=grok_epoch, color='orange', linestyle='--', alpha=0.7, label=f'Grok @ {grok_epoch}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Nearest-Code Accuracy (zoomed)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Margin zoom
    ax = axes[2]
    ax.plot(epochs[mask], np.array(history['train_margin_mean'])[mask], 'b-o', markersize=2, label='Train')
    ax.plot(epochs[mask], np.array(history['val_margin_mean'])[mask], 'r-o', markersize=2, label='Val')
    ax.axvline(x=grok_epoch, color='orange', linestyle='--', alpha=0.7, label=f'Grok @ {grok_epoch}')
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Margin')
    ax.set_title('Wrong-Code Margin (zoomed)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    plt.suptitle('Dynamics Around Grokking Transition', fontweight='bold')
    plt.tight_layout()
    plt.savefig('exp2_grokking_zoom.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping zoom plot — no grokking detected.')

## Experiment 1: Direct JEPA Prediction Accuracy (Final Model)

Detailed analysis of the trained model's direct prediction quality.
This is the definitive test: does `z_pred` actually land on the correct target code?

In [ ]:
context_enc.eval()
predictor.eval()
target_enc_ema.eval()

with torch.no_grad():
    # Validation set
    z_ctx_val = context_enc(val_pairs)
    z_pred_val = predictor(z_ctx_val)
    z_tgt_val = target_enc_ema(val_targets)

    cos_sim_val = (z_pred_val * z_tgt_val).sum(dim=-1)

    # All target codes
    target_codes = target_enc_ema(torch.arange(p, device=device))
    sim_to_all = z_pred_val @ target_codes.T

    # Nearest-code accuracy
    nearest_acc = sim_to_all.argmax(dim=1).eq(val_targets).float().mean().item()

    # Top-k accuracy
    topk_idx = sim_to_all.topk(5, dim=1).indices
    top3_acc = (topk_idx[:, :3] == val_targets.unsqueeze(1)).any(dim=1).float().mean().item()
    top5_acc = (topk_idx[:, :5] == val_targets.unsqueeze(1)).any(dim=1).float().mean().item()

    # Margin statistics
    correct_sim = sim_to_all[torch.arange(n_val), val_targets]
    sim_masked = sim_to_all.clone()
    sim_masked[torch.arange(n_val), val_targets] = -float('inf')
    best_wrong_sim = sim_masked.max(dim=1).values
    margin = correct_sim - best_wrong_sim

print('='*60)
print('EXPERIMENT 1: Direct JEPA Prediction Accuracy')
print('='*60)
print(f'\nValidation set ({n_val} pairs):')
print(f'  Cosine similarity (z_pred, z_target):')
print(f'    Mean: {cos_sim_val.mean().item():.4f} ± {cos_sim_val.std().item():.4f}')
print(f'    Min:  {cos_sim_val.min().item():.4f}')
print(f'    >0.9:  {(cos_sim_val > 0.9).float().mean().item()*100:.2f}%')
print(f'    >0.95: {(cos_sim_val > 0.95).float().mean().item()*100:.2f}%')
print(f'    >0.99: {(cos_sim_val > 0.99).float().mean().item()*100:.2f}%')
print(f'\n  Nearest-code accuracy (argmax_c z_pred·code_c):')
print(f'    Top-1: {nearest_acc*100:.2f}%')
print(f'    Top-3: {top3_acc*100:.2f}%')
print(f'    Top-5: {top5_acc*100:.2f}%')
print(f'\n  Margin (correct − best wrong):')
print(f'    Mean:   {margin.mean().item():.4f} ± {margin.std().item():.4f}')
print(f'    Min:    {margin.min().item():.4f}')
print(f'    All positive: {(margin > 0).all().item()}')

# ── Verdict ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
if nearest_acc > 0.95:
    print('VERDICT: GENUINE self-supervised grokking.')
    print('The JEPA objective itself generalizes — the predictor directly')
    print('produces the correct target latent for unseen (a,b) pairs.')
elif nearest_acc > 0.5:
    print('VERDICT: PARTIAL generalization.')
    print('The predictor shows significant but imperfect direct prediction.')
else:
    print('VERDICT: NO direct generalization.')
    print('The JEPA loss only shaped representations for a separate classifier.')
    print('"Self-supervised grokking" needs qualification.')
print('='*60)

In [ ]:
# ── Experiment 1: Detailed Visualizations ──────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cosine similarity histogram
ax = axes[0]
ax.hist(cos_sim_val.cpu().numpy(), bins=60, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(cos_sim_val.mean().item(), color='red', linestyle='--',
           label=f'Mean: {cos_sim_val.mean().item():.4f}')
ax.set_xlabel('Cosine similarity (z_pred, z_target)')
ax.set_ylabel('Frequency')
ax.set_title('Direct Prediction Quality on Validation')
ax.legend()
ax.grid(alpha=0.3)

# Margin histogram
ax = axes[1]
margin_np = margin.cpu().numpy()
ax.hist(margin_np, bins=60, alpha=0.7, color='teal', edgecolor='black')
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(margin_np.mean(), color='red', linestyle='--',
           label=f'Mean: {margin_np.mean():.4f}')
ax.set_xlabel('Margin (correct − best wrong)')
ax.set_ylabel('Frequency')
ax.set_title('Decision Margin')
ax.legend()
ax.grid(alpha=0.3)

# Per-class accuracy
ax = axes[2]
per_class_acc = []
for c in range(p):
    mask = val_targets == c
    if mask.sum() > 0:
        acc_c = sim_to_all[mask].argmax(dim=1).eq(c).float().mean().item()
        per_class_acc.append(acc_c)
    else:
        per_class_acc.append(0.0)
ax.bar(range(p), [a*100 for a in per_class_acc], color='steelblue', alpha=0.7, width=1.0)
ax.axhline(y=np.mean(per_class_acc)*100, color='red', linestyle='--',
           label=f'Mean: {np.mean(per_class_acc)*100:.1f}%')
ax.set_xlabel('Target class c')
ax.set_ylabel('Nearest-Code Accuracy (%)')
ax.set_title('Per-Class Prediction Accuracy')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Experiment 1: Direct JEPA Prediction Accuracy', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('exp1_direct_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 8 (Bonus): Linear Probe vs Direct Prediction Timeline

The definitive comparison: does the representation become linearly separable BEFORE
the predictor can directly hit targets? Or do they rise together?

**Interpretation:**
- **Probe first** → representation generalizes, predictor follows
- **Together** → JEPA objective directly drives generalization
- **Prediction first** → predictor generalizes before the encoder representation is clean

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel 1: Validation accuracy — both metrics ─────────────────────────
ax = axes[0]
ax.plot(epochs, [a*100 for a in history['val_nearest_acc']],
        'r-', linewidth=2, label='Direct prediction (z_pred → nearest code)')
ax.plot(epochs, [a*100 for a in history['val_probe_acc']],
        'b-', linewidth=2, label='Linear probe (z_ctx → ridge classifier)')
ax.axhline(y=100/p, color='gray', linestyle='--', alpha=0.5, label=f'Chance ({100/p:.1f}%)')
if grok_epoch is not None:
    ax.axvline(x=grok_epoch, color='orange', linestyle='--', alpha=0.5, label=f'Grok @ {grok_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Linear Probe vs Direct JEPA Prediction', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# ── Panel 2: Train accuracy — both metrics ─────────────────────────────
ax = axes[1]
ax.plot(epochs, [a*100 for a in history['train_nearest_acc']],
        'r-', linewidth=2, label='Direct prediction (train)')
ax.plot(epochs, [a*100 for a in history['train_probe_acc']],
        'b-', linewidth=2, label='Linear probe (train)')
ax.plot(epochs, [a*100 for a in history['val_nearest_acc']],
        'r--', linewidth=1.5, alpha=0.6, label='Direct prediction (val)')
ax.plot(epochs, [a*100 for a in history['val_probe_acc']],
        'b--', linewidth=1.5, alpha=0.6, label='Linear probe (val)')
ax.axhline(y=100/p, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('All Four Curves: Train & Val × Probe & Prediction', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('Experiment 8: Which Generalizes First?', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('exp8_probe_vs_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Quantify the lag ───────────────────────────────────────────────────
probe_arr = np.array(history['val_probe_acc'])
pred_arr = np.array(history['val_nearest_acc'])

thresholds = [0.5, 0.8, 0.9, 0.95, 0.99]
print('\nTimeline: When does each metric cross threshold?')
print(f'{"Threshold":>10s} | {"Probe epoch":>12s} | {"Predict epoch":>14s} | {"Lag":>10s}')
print(f'{"-"*10}-+-{"-"*12}-+-{"-"*14}-+-{"-"*10}')
for thr in thresholds:
    probe_cross = epochs[probe_arr >= thr]
    pred_cross = epochs[pred_arr >= thr]
    ep_probe = probe_cross[0] if len(probe_cross) > 0 else None
    ep_pred = pred_cross[0] if len(pred_cross) > 0 else None

    ep_probe_str = f'{ep_probe:>12d}' if ep_probe is not None else f'{"never":>12s}'
    ep_pred_str = f'{ep_pred:>14d}' if ep_pred is not None else f'{"never":>14s}'

    if ep_probe is not None and ep_pred is not None:
        lag = ep_pred - ep_probe
        lag_str = f'{lag:>+10d}'
    else:
        lag_str = f'{"N/A":>10s}'

    print(f'{thr:>10.0%} | {ep_probe_str} | {ep_pred_str} | {lag_str}')

print()
if ep_probe is not None and ep_pred is not None:
    if abs(epochs[1] - epochs[0]) >= abs(ep_pred - ep_probe):
        print('→ Probe and direct prediction rise SIMULTANEOUSLY')
        print('  The JEPA objective directly drives the solution.')
    elif ep_probe < ep_pred:
        print('→ Linear probe rises FIRST')
        print('  The representation becomes separable before the predictor can hit targets.')
        print('  The predictor catches up as the target codes stabilize.')
    else:
        print('→ Direct prediction rises FIRST')
        print('  The predictor generalizes before the context encoder is linearly separable.')
        print('  This would be surprising and worth investigating further.')

## Combined Summary

In [ ]:
print('='*70)
print('SUMMARY: JEPA Grokking — Prediction Accuracy & Dynamics')
print('='*70)

print(f'\n--- Experiment 1: Direct Prediction Accuracy ---')
print(f'  Cosine(z_pred, z_target) on validation:')
print(f'    Mean: {cos_sim_val.mean().item():.4f} ± {cos_sim_val.std().item():.4f}')
print(f'  Nearest-code accuracy on validation:')
print(f'    Top-1: {nearest_acc*100:.2f}%')
print(f'  Margin (correct − best wrong):')
print(f'    Mean: {margin.mean().item():.4f}, all positive: {(margin > 0).all().item()}')
if nearest_acc > 0.95:
    print(f'  ✓ GENUINE self-supervised grokking confirmed.')
else:
    print(f'  ✗ Direct prediction insufficient — grokking claim needs qualification.')

print(f'\n--- Experiment 2: Memorization Dynamics ---')
if grok_epoch is not None:
    # Find memorization phase: when train nearest acc > 90% but val < 10%
    mem_mask = (train_acc_arr > 0.9) & (val_acc_arr < 0.1)
    if mem_mask.any():
        mem_start = epochs[mem_mask][0]
        mem_end = epochs[mem_mask][-1]
        print(f'  Memorization phase: epoch {mem_start} – {mem_end}')
        print(f'    (train acc >90% while val acc <10%)')
    else:
        print(f'  No clear memorization phase detected.')
    print(f'  Grokking transition: epoch {grok_epoch}')
    print(f'  Final state:')
    print(f'    Train cos: {history["train_cos_mean"][-1]:.4f}')
    print(f'    Val cos:   {history["val_cos_mean"][-1]:.4f}')
    print(f'    Gap:       {history["train_cos_mean"][-1] - history["val_cos_mean"][-1]:.4f}')
else:
    print(f'  No grokking detected within {EPOCHS} epochs.')

print(f'\n--- Experiment 8: Probe vs Prediction Timeline ---')
final_probe = history['val_probe_acc'][-1]
final_pred = history['val_nearest_acc'][-1]
print(f'  Final linear probe accuracy:     {final_probe*100:.2f}%')
print(f'  Final direct prediction accuracy: {final_pred*100:.2f}%')

print(f'\n{"="*70}')

# Part 5: Target Encoder Interventions & Predictor Ablation

**Description:** Target encoder freeze, drift analysis, predictor ablation

---

# JEPA Grokking: Target Encoder Intervention & Predictor Ablation

## What's at Stake

The JEPA architecture has three moving parts: context encoder, predictor, and EMA target encoder.
Which components are actually necessary for grokking? When are they necessary?

**Experiment 3: Target Encoder Freeze Intervention**  
Freeze the target encoder at various points during training and continue training only the
context encoder + predictor. If grokking still occurs → the target encoder's evolution is not
necessary after that point. If it stops → the target encoder must keep drifting for grokking.

**Experiment 3b (bonus): Target Encoder Drift Analysis**  
Track how fast the target codes change between checkpoints. Does the target encoder stabilize
before, during, or after grokking? This directly complements Exp 3 — if the target is still
drifting rapidly when we freeze it, we're killing a moving target.

**Experiment 4: Predictor Ablation at Different Stages**  
At different checkpoints (pre-grokking, during, post-grokking), remove the predictor and
replace it with identity (direct context → target matching). Continue training.
If accuracy survives → the predictor was needed for dynamics, not the final algorithm.
If it collapses → the predictor is structurally essential at that stage.

### Strategy
Phase 1: Full training run with checkpoints saved at key intervals  
Phase 2: Reload checkpoints and run interventions from each point

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
import os
from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader

# ── Configuration ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Task
p = 97
train_frac = 0.3

# Architecture
LATENT_DIM = 128
HIDDEN_DIM = 256
PREDICTOR_DIM = 64

# Training
EPOCHS = 60_000
LR = 1e-3
WEIGHT_DECAY = 1.0
EMA_DECAY = 0.996

# Evaluation & checkpoint schedule
EVAL_EVERY = 500        # light eval (accuracy only)
CKPT_EVERY = 5000       # save full checkpoint for interventions

# Intervention training budget: how many MORE epochs to train after freezing/ablating
INTERVENTION_EPOCHS = 30_000
INTERVENTION_EVAL_EVERY = 500

CKPT_DIR = 'phase1_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'Task: (a+b) mod {p}')
print(f'Full training: {EPOCHS:,} epochs')
print(f'Checkpoints saved every {CKPT_EVERY} epochs → {EPOCHS // CKPT_EVERY} checkpoints')
print(f'Intervention budget: {INTERVENTION_EPOCHS:,} additional epochs per experiment')

## Data & Model Definitions

In [ ]:
def generate_data(seed=SEED):
    pairs = torch.cartesian_prod(torch.arange(p), torch.arange(p))
    targets = (pairs[:, 0] + pairs[:, 1]) % p
    n = len(pairs)
    n_train = int(train_frac * n)
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    train_idx, val_idx = perm[:n_train], perm[n_train:]
    return (
        pairs.to(device), targets.to(device),
        pairs[train_idx].to(device), targets[train_idx].to(device),
        pairs[val_idx].to(device), targets[val_idx].to(device),
    )

pairs, targets, train_pairs, train_targets, val_pairs, val_targets = generate_data(SEED)
print(f'Train: {len(train_pairs)}, Val: {len(val_pairs)}')

class ContextEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )
    def forward(self, x):
        e = self.emb(x)
        e = e.view(e.size(0), -1)
        return F.normalize(self.net(e), dim=-1)

class TargetEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )
    def forward(self, x):
        e = self.emb(x)
        return F.normalize(self.net(e), dim=-1)

class Predictor(nn.Module):
    def __init__(self, latent_dim, predictor_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, latent_dim),
        )
    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────

@torch.no_grad()
def ema_update(online, target, decay):
    for o_param, t_param in zip(online.parameters(), target.parameters()):
        t_param.data.mul_(decay).add_(o_param.data, alpha=1 - decay)

@torch.no_grad()
def linear_probe_accuracy(encoder, val_pairs, val_targets, train_pairs, train_targets, p):
    z_train = encoder(train_pairs)
    z_val = encoder(val_pairs)
    y_train = F.one_hot(train_targets, num_classes=p).float()
    reg = 1e-3
    ZtZ = z_train.T @ z_train + reg * torch.eye(z_train.shape[1], device=device)
    ZtY = z_train.T @ y_train
    W = torch.linalg.solve(ZtZ, ZtY)
    train_acc = (z_train @ W).argmax(dim=1).eq(train_targets).float().mean().item()
    val_acc = (z_val @ W).argmax(dim=1).eq(val_targets).float().mean().item()
    return train_acc, val_acc

@torch.no_grad()
def nearest_code_accuracy(context_enc, predictor, target_enc_ema, pairs, tgts):
    """Direct JEPA prediction: argmax_c (z_pred · code_c) == true_c."""
    z_pred = predictor(context_enc(pairs))
    target_codes = target_enc_ema(torch.arange(p, device=device))
    sim = z_pred @ target_codes.T
    return sim.argmax(dim=1).eq(tgts).float().mean().item()

@torch.no_grad()
def direct_match_accuracy(context_enc, target_enc_ema, pairs, tgts):
    """No-predictor accuracy: argmax_c (z_ctx · code_c) == true_c."""
    z_ctx = context_enc(pairs)
    target_codes = target_enc_ema(torch.arange(p, device=device))
    sim = z_ctx @ target_codes.T
    return sim.argmax(dim=1).eq(tgts).float().mean().item()

@torch.no_grad()
def get_target_codes(target_enc_ema):
    """Get all p target codes as a (p, d) matrix."""
    return target_enc_ema(torch.arange(p, device=device))

## Phase 1: Full Training Run with Checkpoint Saving

Train the full JEPA model for 100K epochs, saving complete model state every 5K epochs.
Also save target codes at every eval point for drift analysis (Exp 3b).

In [ ]:
# Initialize fresh model
context_enc = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
target_enc  = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
predictor   = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
target_enc_ema = deepcopy(target_enc)
for param in target_enc_ema.parameters():
    param.requires_grad = False

# Compile models for faster training
context_enc = torch.compile(context_enc)
target_enc = torch.compile(target_enc)
predictor = torch.compile(predictor)
target_enc_ema = torch.compile(target_enc_ema)

optimizer = optim.AdamW(
    list(context_enc.parameters()) + list(predictor.parameters()) + list(target_enc.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)

train_loader = DataLoader(
    TensorDataset(train_pairs, train_targets),
    batch_size=len(train_pairs), shuffle=True
)

# Storage
phase1_history = {
    'epoch': [], 'loss': [],
    'val_probe_acc': [], 'train_probe_acc': [],
    'val_nearest_acc': [], 'train_nearest_acc': [],
}

# Store target codes at each eval point for drift analysis
target_code_snapshots = {}  # epoch → (p, d) tensor

# Track which checkpoints we saved
saved_checkpoints = []

print(f'Phase 1: Full training for {EPOCHS:,} epochs...')
start = time.time()

for epoch in range(EPOCHS):
    context_enc.train()
    target_enc.train()
    predictor.train()

    for bp, bt in train_loader:
        optimizer.zero_grad()
        z_ctx = context_enc(bp)
        z_pred = predictor(z_ctx)
        with torch.no_grad():
            z_tgt = target_enc_ema(bt)
        loss = -(z_pred * z_tgt).sum(dim=-1).mean()
        loss.backward()
        optimizer.step()
        ema_update(target_enc, target_enc_ema, EMA_DECAY)

    # ── Light eval ─────────────────────────────────────────────────────
    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS - 1:
        context_enc.eval()
        predictor.eval()

        train_probe, val_probe = linear_probe_accuracy(
            context_enc, val_pairs, val_targets, train_pairs, train_targets, p)
        train_nearest = nearest_code_accuracy(
            context_enc, predictor, target_enc_ema, train_pairs, train_targets)
        val_nearest = nearest_code_accuracy(
            context_enc, predictor, target_enc_ema, val_pairs, val_targets)

        phase1_history['epoch'].append(epoch)
        phase1_history['loss'].append(loss.item())
        phase1_history['train_probe_acc'].append(train_probe)
        phase1_history['val_probe_acc'].append(val_probe)
        phase1_history['train_nearest_acc'].append(train_nearest)
        phase1_history['val_nearest_acc'].append(val_nearest)

        # Snapshot target codes for drift analysis
        target_code_snapshots[epoch] = get_target_codes(target_enc_ema).cpu()

    # ── Full checkpoint ────────────────────────────────────────────────
    if epoch % CKPT_EVERY == 0 or epoch == EPOCHS - 1:
        ckpt_path = os.path.join(CKPT_DIR, f'ckpt_epoch_{epoch:06d}.pt')
        torch.save({
            'epoch': epoch,
            'context_enc': context_enc.state_dict(),
            'target_enc': target_enc.state_dict(),
            'target_enc_ema': target_enc_ema.state_dict(),
            'predictor': predictor.state_dict(),
            'optimizer': optimizer.state_dict(),
        }, ckpt_path)
        saved_checkpoints.append(epoch)

    # Print
    if epoch % (EVAL_EVERY * 10) == 0 or epoch == EPOCHS - 1:
        elapsed = (time.time() - start) / 60
        print(f'Epoch {epoch:6d} [{elapsed:5.1f}m] | '
              f'Loss {loss.item():+.4f} | '
              f'Probe: tr={train_probe*100:.1f}% val={val_probe*100:.1f}% | '
              f'Nearest: tr={train_nearest*100:.1f}% val={val_nearest*100:.1f}%')

elapsed = (time.time() - start) / 60
print(f'\nPhase 1 complete in {elapsed:.1f} min.')
print(f'Saved {len(saved_checkpoints)} checkpoints: {saved_checkpoints[:5]}...{saved_checkpoints[-3:]}')
print(f'Target code snapshots: {len(target_code_snapshots)} eval points')

In [ ]:
# Identify key epochs for interventions
epochs_arr = np.array(phase1_history['epoch'])
val_acc_arr = np.array(phase1_history['val_probe_acc'])

# Find grokking epoch (val probe > 90%)
grok_mask = val_acc_arr > 0.9
if grok_mask.any():
    grok_epoch = epochs_arr[grok_mask][0]
else:
    grok_epoch = EPOCHS // 2  # fallback

# Select intervention points: early, pre-grok, at grok, post-grok
# Round to nearest saved checkpoint
def nearest_ckpt(target_epoch):
    return min(saved_checkpoints, key=lambda x: abs(x - target_epoch))

intervention_epochs = {
    'early':     nearest_ckpt(int(grok_epoch * 0.1)),
    'pre_grok':  nearest_ckpt(int(grok_epoch * 0.7)),
    'at_grok':   nearest_ckpt(grok_epoch),
    'post_grok': nearest_ckpt(min(int(grok_epoch * 1.5), EPOCHS - 1)),
}

print(f'Grokking detected at epoch: {grok_epoch}')
print(f'\nIntervention checkpoints:')
for name, ep in intervention_epochs.items():
    idx = np.argmin(np.abs(epochs_arr - ep))
    acc = val_acc_arr[idx]
    print(f'  {name:12s}: epoch {ep:6d} (val probe acc = {acc*100:.1f}%)')

## Experiment 3b (Bonus): Target Encoder Drift Analysis

Before running interventions, let's understand how the target encoder evolves.
At each eval checkpoint we have the 97 target codes. We measure:
1. **Drift rate**: mean cosine distance between codes at epoch t vs epoch t-Δ
2. **Cumulative drift**: distance from initial codes
3. **Code stability**: fraction of codes that haven't moved significantly

This tells us *when* the target encoder stabilizes, which directly predicts
whether freezing at that point will harm training (Exp 3).

In [ ]:
snapshot_epochs = sorted(target_code_snapshots.keys())

# ── Compute drift metrics ──────────────────────────────────────────────
drift_rate = []       # cos distance from previous snapshot
cumulative_drift = [] # cos distance from epoch 0
drift_epochs = []     # corresponding epochs

initial_codes = target_code_snapshots[snapshot_epochs[0]]  # (p, d)

for i in range(1, len(snapshot_epochs)):
    ep = snapshot_epochs[i]
    ep_prev = snapshot_epochs[i - 1]

    codes_now = target_code_snapshots[ep]     # (p, d)
    codes_prev = target_code_snapshots[ep_prev]

    # Per-code cosine similarity to previous snapshot
    cos_to_prev = (codes_now * codes_prev).sum(dim=-1)  # (p,)
    rate = 1 - cos_to_prev.mean().item()  # angular distance (0 = identical)

    # Per-code cosine similarity to initial
    cos_to_init = (codes_now * initial_codes).sum(dim=-1)
    cum = 1 - cos_to_init.mean().item()

    drift_rate.append(rate)
    cumulative_drift.append(cum)
    drift_epochs.append(ep)

drift_epochs = np.array(drift_epochs)
drift_rate = np.array(drift_rate)
cumulative_drift = np.array(cumulative_drift)

# ── Find stabilization point ──────────────────────────────────────────
# Define "stable" = drift rate < 1% of peak drift rate for 5 consecutive points
peak_drift = drift_rate.max()
stable_threshold = peak_drift * 0.01
stable_epoch = None
for i in range(len(drift_rate) - 5):
    if np.all(drift_rate[i:i+5] < stable_threshold):
        stable_epoch = drift_epochs[i]
        break

print('Target Encoder Drift Analysis:')
print(f'  Peak drift rate: {peak_drift:.6f} (at epoch {drift_epochs[np.argmax(drift_rate)]})')
print(f'  Final drift rate: {drift_rate[-1]:.6f}')
print(f'  Total cumulative drift from init: {cumulative_drift[-1]:.4f}')
if stable_epoch is not None:
    print(f'  Stabilization point: epoch {stable_epoch}')
    print(f'  Grokking epoch: {grok_epoch}')
    if stable_epoch < grok_epoch:
        print(f'  → Target encoder stabilizes BEFORE grokking')
        print(f'    Implication: freezing near grokking should be safe')
    else:
        print(f'  → Target encoder still drifting AT grokking')
        print(f'    Implication: freezing may interfere with grokking dynamics')
else:
    print(f'  Target encoder never fully stabilized (drift > threshold throughout)')

In [ ]:
# ── Visualization ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Drift rate over time
ax = axes[0]
ax.semilogy(drift_epochs, drift_rate, 'b-', linewidth=1.5)
ax.axvline(x=grok_epoch, color='orange', linestyle='--', alpha=0.7, label=f'Grok @ {grok_epoch}')
if stable_epoch is not None:
    ax.axvline(x=stable_epoch, color='green', linestyle='--', alpha=0.7, label=f'Stable @ {stable_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Drift Rate (1 - cos to previous)')
ax.set_title('Target Code Drift Rate', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 2: Cumulative drift
ax = axes[1]
ax.plot(drift_epochs, cumulative_drift, 'r-', linewidth=1.5)
ax.axvline(x=grok_epoch, color='orange', linestyle='--', alpha=0.7, label=f'Grok @ {grok_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cumulative Drift (1 - cos to initial)')
ax.set_title('Cumulative Drift from Initialization', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 3: Drift rate overlaid with validation accuracy
ax = axes[2]
ax_acc = ax.twinx()
l1, = ax.semilogy(drift_epochs, drift_rate, 'b-', linewidth=1.5, label='Drift rate')
l2, = ax_acc.plot(epochs_arr, np.array(phase1_history['val_probe_acc'])*100,
                  'r-', linewidth=1.5, alpha=0.7, label='Val probe acc')
ax.set_xlabel('Epoch')
ax.set_ylabel('Drift Rate (log)', color='blue')
ax_acc.set_ylabel('Validation Accuracy (%)', color='red')
ax.set_title('Drift vs Accuracy', fontweight='bold')
ax.legend(handles=[l1, l2], fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('Experiment 3b: Target Encoder Drift Analysis', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('exp3b_target_drift.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 3: Target Encoder Freeze Intervention

For each intervention point (early, pre-grok, at-grok, post-grok):
1. Reload the checkpoint from that epoch
2. **Freeze** the EMA target encoder (stop EMA updates)
3. Continue training only context encoder + predictor for 30K more epochs
4. Track whether grokking occurs / accuracy improves

The question: does the target encoder need to keep evolving for grokking?

In [ ]:
def run_freeze_intervention(ckpt_epoch, label, intervention_epochs=INTERVENTION_EPOCHS):
    """Reload checkpoint, freeze target encoder, continue training."""
    ckpt_path = os.path.join(CKPT_DIR, f'ckpt_epoch_{ckpt_epoch:06d}.pt')
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    # Rebuild models
    ctx = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    pred = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
    tgt_ema = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)

    ctx.load_state_dict(ckpt['context_enc'])
    pred.load_state_dict(ckpt['predictor'])
    tgt_ema.load_state_dict(ckpt['target_enc_ema'])

    # FREEZE: no EMA updates, no gradients
    for param in tgt_ema.parameters():
        param.requires_grad = False

    # Only optimize context encoder + predictor
    opt = optim.AdamW(
        list(ctx.parameters()) + list(pred.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )

    history = {'epoch': [], 'val_probe_acc': [], 'val_nearest_acc': [], 'loss': []}

    start = time.time()
    for ep in range(intervention_epochs):
        ctx.train()
        pred.train()
        for bp, bt in train_loader:
            opt.zero_grad()
            z_ctx = ctx(bp)
            z_pred = pred(z_ctx)
            with torch.no_grad():
                z_tgt = tgt_ema(bt)
            loss = -(z_pred * z_tgt).sum(dim=-1).mean()
            loss.backward()
            opt.step()
            # NO ema_update — target is frozen

        if ep % INTERVENTION_EVAL_EVERY == 0 or ep == intervention_epochs - 1:
            ctx.eval()
            pred.eval()
            _, val_probe = linear_probe_accuracy(
                ctx, val_pairs, val_targets, train_pairs, train_targets, p)
            val_nearest = nearest_code_accuracy(ctx, pred, tgt_ema, val_pairs, val_targets)

            history['epoch'].append(ckpt_epoch + ep)
            history['val_probe_acc'].append(val_probe)
            history['val_nearest_acc'].append(val_nearest)
            history['loss'].append(loss.item())

    elapsed = (time.time() - start) / 60
    final_probe = history['val_probe_acc'][-1]
    final_nearest = history['val_nearest_acc'][-1]
    print(f'  [{label}] Freeze @ {ckpt_epoch} → +{intervention_epochs} epochs ({elapsed:.1f}m): '
          f'probe={final_probe*100:.1f}%, nearest={final_nearest*100:.1f}%')

    return history

# ── Run all freeze interventions ───────────────────────────────────────
print('Experiment 3: Target Encoder Freeze Interventions')
print('='*60)

freeze_results = {}
for name, ep in intervention_epochs.items():
    freeze_results[name] = run_freeze_intervention(ep, name)

print('\nAll freeze interventions complete.')

In [ ]:
# ── Experiment 3: Visualization ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {'early': '#ef4444', 'pre_grok': '#f59e0b', 'at_grok': '#10b981', 'post_grok': '#3b82f6'}

# Panel 1: Val probe accuracy after freezing
ax = axes[0]
# Baseline (no freeze)
ax.plot(epochs_arr, np.array(phase1_history['val_probe_acc'])*100,
        'k-', linewidth=2, alpha=0.4, label='Baseline (no freeze)')
for name, hist in freeze_results.items():
    ep_freeze = intervention_epochs[name]
    ax.plot(hist['epoch'], [a*100 for a in hist['val_probe_acc']],
            '-', color=colors[name], linewidth=2,
            label=f'Freeze @ {ep_freeze} ({name})')
    ax.axvline(x=ep_freeze, color=colors[name], linestyle=':', alpha=0.3)
ax.axhline(y=100/p, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Probe Accuracy (%)')
ax.set_title('Linear Probe After Target Freeze', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# Panel 2: Val nearest-code accuracy after freezing
ax = axes[1]
ax.plot(epochs_arr, np.array(phase1_history['val_nearest_acc'])*100,
        'k-', linewidth=2, alpha=0.4, label='Baseline (no freeze)')
for name, hist in freeze_results.items():
    ep_freeze = intervention_epochs[name]
    ax.plot(hist['epoch'], [a*100 for a in hist['val_nearest_acc']],
            '-', color=colors[name], linewidth=2,
            label=f'Freeze @ {ep_freeze} ({name})')
    ax.axvline(x=ep_freeze, color=colors[name], linestyle=':', alpha=0.3)
ax.axhline(y=100/p, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Nearest-Code Accuracy (%)')
ax.set_title('Direct Prediction After Target Freeze', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

plt.suptitle('Experiment 3: Does the Target Encoder Need to Keep Evolving?',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('exp3_target_freeze.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ─────────────────────────────────────────────────────
print('\nExperiment 3 Results:')
print(f'{"Stage":>12s} | {"Freeze @":>8s} | {"Final Probe":>12s} | {"Final Nearest":>14s} | {"Grokked?":>8s}')
print(f'{"-"*12}-+-{"-"*8}-+-{"-"*12}-+-{"-"*14}-+-{"-"*8}')
for name, hist in freeze_results.items():
    ep = intervention_epochs[name]
    fp = hist['val_probe_acc'][-1]
    fn = hist['val_nearest_acc'][-1]
    grokked = '✓' if fp > 0.9 else '✗'
    print(f'{name:>12s} | {ep:>8d} | {fp*100:>11.1f}% | {fn*100:>13.1f}% | {grokked:>8s}')

## Experiment 4: Predictor Ablation at Different Stages

For each intervention point:
1. Reload the checkpoint
2. **Remove the predictor** — replace it with identity (direct cosine loss between `z_ctx` and `z_target`)
3. Continue training context encoder (with EMA target updates) for 30K more epochs
4. Track whether accuracy improves, stays flat, or collapses

This tests the predictor's role at each stage:
- **If removing post-grok preserves accuracy** → predictor was needed only during training dynamics
- **If removing causes collapse** → predictor is structurally essential (prevents co-adaptation)
- **If removing early blocks grokking** → predictor's bottleneck is needed for representation learning

In [ ]:
def run_predictor_ablation(ckpt_epoch, label, intervention_epochs=INTERVENTION_EPOCHS):
    """Reload checkpoint, remove predictor, continue training with direct matching."""
    ckpt_path = os.path.join(CKPT_DIR, f'ckpt_epoch_{ckpt_epoch:06d}.pt')
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    # Rebuild models
    ctx = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    tgt_enc = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    tgt_ema = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)

    ctx.load_state_dict(ckpt['context_enc'])
    tgt_enc.load_state_dict(ckpt['target_enc'])
    tgt_ema.load_state_dict(ckpt['target_enc_ema'])

    for param in tgt_ema.parameters():
        param.requires_grad = False

    # NO predictor — optimize context encoder + target encoder
    opt = optim.AdamW(
        list(ctx.parameters()) + list(tgt_enc.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )

    history = {
        'epoch': [], 'val_probe_acc': [], 'val_direct_acc': [],
        'loss': [], 'cos_mean': [], 'cos_std': [],
    }

    start = time.time()
    for ep in range(intervention_epochs):
        ctx.train()
        tgt_enc.train()
        for bp, bt in train_loader:
            opt.zero_grad()
            z_ctx = ctx(bp)  # this IS the prediction (no predictor)
            with torch.no_grad():
                z_tgt = tgt_ema(bt)
            loss = -(z_ctx * z_tgt).sum(dim=-1).mean()
            loss.backward()
            opt.step()
            ema_update(tgt_enc, tgt_ema, EMA_DECAY)

        if ep % INTERVENTION_EVAL_EVERY == 0 or ep == intervention_epochs - 1:
            ctx.eval()
            _, val_probe = linear_probe_accuracy(
                ctx, val_pairs, val_targets, train_pairs, train_targets, p)
            val_direct = direct_match_accuracy(ctx, tgt_ema, val_pairs, val_targets)

            # Track cosine similarity for collapse detection
            with torch.no_grad():
                z_val = ctx(val_pairs)
                z_sub = z_val[:1000]
                cos_matrix = z_sub @ z_sub.T
                mask = ~torch.eye(len(z_sub), dtype=torch.bool, device=device)
                cos_vals = cos_matrix[mask]

            history['epoch'].append(ckpt_epoch + ep)
            history['val_probe_acc'].append(val_probe)
            history['val_direct_acc'].append(val_direct)
            history['loss'].append(loss.item())
            history['cos_mean'].append(cos_vals.mean().item())
            history['cos_std'].append(cos_vals.std().item())

    elapsed = (time.time() - start) / 60
    final_probe = history['val_probe_acc'][-1]
    final_direct = history['val_direct_acc'][-1]
    final_cos = history['cos_mean'][-1]
    collapsed = final_cos > 0.95 and history['cos_std'][-1] < 0.01
    status = 'COLLAPSED' if collapsed else 'OK'

    print(f'  [{label}] Remove pred @ {ckpt_epoch} → +{intervention_epochs} epochs ({elapsed:.1f}m): '
          f'probe={final_probe*100:.1f}%, direct={final_direct*100:.1f}%, '
          f'cos_μ={final_cos:.3f} [{status}]')

    history['collapsed'] = collapsed
    return history

# ── Run all predictor ablations ────────────────────────────────────────
print('Experiment 4: Predictor Ablation at Different Stages')
print('='*60)

ablation_results = {}
for name, ep in intervention_epochs.items():
    ablation_results[name] = run_predictor_ablation(ep, name)

print('\nAll predictor ablations complete.')

In [ ]:
# ── Experiment 4: Visualization ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Val probe accuracy
ax = axes[0, 0]
ax.plot(epochs_arr, np.array(phase1_history['val_probe_acc'])*100,
        'k-', linewidth=2, alpha=0.4, label='Baseline (with predictor)')
for name, hist in ablation_results.items():
    ep_ablate = intervention_epochs[name]
    ax.plot(hist['epoch'], [a*100 for a in hist['val_probe_acc']],
            '-', color=colors[name], linewidth=2,
            label=f'No pred @ {ep_ablate} ({name})')
    ax.axvline(x=ep_ablate, color=colors[name], linestyle=':', alpha=0.3)
ax.axhline(y=100/p, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Probe Accuracy (%)')
ax.set_title('Linear Probe After Predictor Removal', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# Panel 2: Direct match accuracy (z_ctx vs target code, no predictor)
ax = axes[0, 1]
for name, hist in ablation_results.items():
    ep_ablate = intervention_epochs[name]
    ax.plot(hist['epoch'], [a*100 for a in hist['val_direct_acc']],
            '-', color=colors[name], linewidth=2,
            label=f'No pred @ {ep_ablate} ({name})')
ax.axhline(y=100/p, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Direct Match Accuracy (%)')
ax.set_title('z_ctx → Nearest Code (No Predictor)', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# Panel 3: Collapse indicator (pairwise cosine mean)
ax = axes[1, 0]
for name, hist in ablation_results.items():
    ep_ablate = intervention_epochs[name]
    ax.plot(hist['epoch'], hist['cos_mean'],
            '-', color=colors[name], linewidth=2,
            label=f'No pred @ {ep_ablate} ({name})')
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.3, label='Full collapse')
ax.axhline(y=0.0, color='gray', linestyle='--', alpha=0.3, label='Orthogonal')
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean Pairwise Cosine')
ax.set_title('Collapse Indicator', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# Panel 4: Summary bar chart
ax = axes[1, 1]
stage_names = list(ablation_results.keys())
final_probes = [ablation_results[n]['val_probe_acc'][-1]*100 for n in stage_names]
final_directs = [ablation_results[n]['val_direct_acc'][-1]*100 for n in stage_names]
x = np.arange(len(stage_names))
w = 0.35
ax.bar(x - w/2, final_probes, w, color='steelblue', alpha=0.8, label='Probe acc', edgecolor='black')
ax.bar(x + w/2, final_directs, w, color='coral', alpha=0.8, label='Direct acc', edgecolor='black')
# Mark collapsed
for i, name in enumerate(stage_names):
    if ablation_results[name]['collapsed']:
        ax.annotate('COLLAPSED', xy=(i, max(final_probes[i], final_directs[i]) + 2),
                    ha='center', fontsize=7, color='red', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'{n}\n(@{intervention_epochs[n]})' for n in stage_names], fontsize=8)
ax.axhline(y=100/p, color='gray', linestyle='--', alpha=0.3)
ax.set_ylabel('Final Validation Accuracy (%)')
ax.set_title('Predictor Ablation Summary', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.suptitle('Experiment 4: Is the Predictor Essential, and When?',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('exp4_predictor_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

## Combined Summary & Interpretation

In [ ]:
print('='*70)
print('COMBINED SUMMARY: Target Encoder & Predictor Interventions')
print('='*70)

# ── Exp 3b: Drift ─────────────────────────────────────────────────────
print(f'\n--- Experiment 3b: Target Encoder Drift ---')
print(f'  Peak drift rate: {peak_drift:.6f} (epoch {drift_epochs[np.argmax(drift_rate)]})')
print(f'  Grokking epoch: {grok_epoch}')
if stable_epoch is not None:
    print(f'  Stabilization:  epoch {stable_epoch}')
    print(f'  Target encoder {"stabilizes BEFORE" if stable_epoch < grok_epoch else "still drifting AT"} grokking.')
else:
    print(f'  Target encoder never fully stabilized.')

# ── Exp 3: Freeze ─────────────────────────────────────────────────────
print(f'\n--- Experiment 3: Target Encoder Freeze ---')
print(f'{"Stage":>12s} | {"Freeze @":>8s} | {"Probe":>7s} | {"Nearest":>7s} | {"Verdict":>20s}')
print(f'{"-"*12}-+-{"-"*8}-+-{"-"*7}-+-{"-"*7}-+-{"-"*20}')
for name, hist in freeze_results.items():
    ep = intervention_epochs[name]
    fp = hist['val_probe_acc'][-1]
    fn = hist['val_nearest_acc'][-1]
    if fp > 0.9:
        verdict = '✓ Grokked'
    elif fp > 0.3:
        verdict = '~ Partial'
    else:
        verdict = '✗ Failed'
    print(f'{name:>12s} | {ep:>8d} | {fp*100:>6.1f}% | {fn*100:>6.1f}% | {verdict:>20s}')

# Interpret
early_freeze = freeze_results['early']['val_probe_acc'][-1]
at_grok_freeze = freeze_results['at_grok']['val_probe_acc'][-1]
post_grok_freeze = freeze_results['post_grok']['val_probe_acc'][-1]

print(f'\n  Interpretation:')
if early_freeze < 0.3 and at_grok_freeze > 0.8:
    print(f'  → Target encoder evolution IS necessary for grokking.')
    print(f'    Freezing early prevents grokking; freezing late preserves it.')
    print(f'    The co-evolution of target and context encoders drives generalization.')
elif early_freeze > 0.8:
    print(f'  → Target encoder evolution is NOT necessary.')
    print(f'    Even early freezing leads to grokking. The context encoder + predictor')
    print(f'    can learn from a fixed target representation.')
else:
    print(f'  → Mixed results. See individual curves for nuanced interpretation.')

# ── Exp 4: Predictor ablation ─────────────────────────────────────────
print(f'\n--- Experiment 4: Predictor Ablation ---')
print(f'{"Stage":>12s} | {"Ablate @":>8s} | {"Probe":>7s} | {"Direct":>7s} | {"Collapsed":>10s} | {"Verdict":>20s}')
print(f'{"-"*12}-+-{"-"*8}-+-{"-"*7}-+-{"-"*7}-+-{"-"*10}-+-{"-"*20}')
for name, hist in ablation_results.items():
    ep = intervention_epochs[name]
    fp = hist['val_probe_acc'][-1]
    fd = hist['val_direct_acc'][-1]
    col = hist['collapsed']
    if col:
        verdict = '⚠ Collapsed'
    elif fp > 0.9:
        verdict = '✓ Survived'
    elif fp > 0.3:
        verdict = '~ Partial'
    else:
        verdict = '✗ Failed'
    print(f'{name:>12s} | {ep:>8d} | {fp*100:>6.1f}% | {fd*100:>6.1f}% | {str(col):>10s} | {verdict:>20s}')

# Interpret
early_ablate = ablation_results['early']['val_probe_acc'][-1]
post_ablate = ablation_results['post_grok']['val_probe_acc'][-1]
early_collapse = ablation_results['early']['collapsed']
post_collapse = ablation_results['post_grok']['collapsed']

print(f'\n  Interpretation:')
if early_collapse and not post_collapse and post_ablate > 0.8:
    print(f'  → Predictor is essential DURING training but dispensable AFTER grokking.')
    print(f'    Early removal causes collapse (no asymmetric bottleneck → co-adaptation).')
    print(f'    Post-grok removal preserves accuracy — the predictor shaped the representation')
    print(f'    but is not needed for the final algorithm.')
elif early_collapse and post_collapse:
    print(f'  → Predictor is ALWAYS essential. Removing it causes collapse at any stage.')
    print(f'    The asymmetric architecture is structurally necessary, not just for training dynamics.')
elif not early_collapse and not post_collapse:
    print(f'  → Predictor is NOT essential at any stage.')
    print(f'    Direct context → target matching works. The predictor may just be')
    print(f'    accelerating training, not enabling a fundamentally different learning dynamic.')
else:
    print(f'  → Mixed results. The predictor\'s role depends on training stage.')

print(f'\n{"="*70}')

# Part 6: Function Approximation, Causal Intervention & Task Variation

**Description:** Fourier decomposition of class means, causal interventions, task variation (multiplication, XOR, polynomial)

---

# JEPA Grokking: Function Approximation, Causal Intervention & Task Variation

## What This Notebook Establishes

**Experiment 5: Function Approximation View**  
What symbolic function does the context encoder + predictor compute?  
For each residue class c, compute the average predicted latent and decompose it  
into a Fourier series. How many components are needed? Is there phase structure?

**Important caveat:** Any function of (a+b) mod p is *perfectly* representable in the  
Fourier basis of Z/pZ — that's what a complete orthonormal basis means. So high Fourier  
R² doesn't mean "the model uses Fourier circuits." The interesting questions are:  
(1) How many components are needed? (2) Is there phase/amplitude structure?  
(3) How does this compare to the target codes the model is *trying* to match?

**Experiment 6: Causal Intervention**  
Can we prove the network relies on specific Fourier components of the sum?  
Zero out frequency bands and measure the effect on accuracy.  
Add noise aligned vs orthogonal to the Fourier basis — which hurts more?

**Experiment 7: Task Variation**  
Does JEPA grok on other algebraic tasks? Train on modular multiplication,  
XOR, and polynomial evaluation. Does the algorithm change?

### Requirements
- Experiments 5 & 6 load the checkpoint from the Exp 1/2/8 notebook  
- Experiment 7 trains fresh models from scratch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
import os
from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

p = 97
train_frac = 0.3
LATENT_DIM = 128
HIDDEN_DIM = 256
PREDICTOR_DIM = 64
LR = 1e-3
WEIGHT_DECAY = 1.0
EMA_DECAY = 0.996

In [ ]:
# ── Model definitions ──────────────────────────────────────────────────

class ContextEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )
    def forward(self, x):
        e = self.emb(x)
        return F.normalize(self.net(e.view(e.size(0), -1)), dim=-1)

class TargetEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )
    def forward(self, x):
        e = self.emb(x)
        return F.normalize(self.net(e), dim=-1)

class Predictor(nn.Module):
    def __init__(self, latent_dim, predictor_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, predictor_dim), nn.GELU(),
            nn.Linear(predictor_dim, predictor_dim), nn.GELU(),
            nn.Linear(predictor_dim, latent_dim),
        )
    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)

# ── Helpers ─────────────────────────────────────────────────────────────

@torch.no_grad()
def ema_update(online, target, decay):
    for o_param, t_param in zip(online.parameters(), target.parameters()):
        t_param.data.mul_(decay).add_(o_param.data, alpha=1 - decay)

@torch.no_grad()
def linear_probe_accuracy(encoder, val_pairs, val_targets, train_pairs, train_targets, num_classes):
    z_train = encoder(train_pairs)
    z_val = encoder(val_pairs)
    y_train = F.one_hot(train_targets, num_classes=num_classes).float()
    reg = 1e-3
    ZtZ = z_train.T @ z_train + reg * torch.eye(z_train.shape[1], device=device)
    ZtY = z_train.T @ y_train
    W = torch.linalg.solve(ZtZ, ZtY)
    train_acc = (z_train @ W).argmax(dim=1).eq(train_targets).float().mean().item()
    val_acc = (z_val @ W).argmax(dim=1).eq(val_targets).float().mean().item()
    return train_acc, val_acc

@torch.no_grad()
def nearest_code_accuracy(context_enc, predictor, target_enc_ema, pairs, tgts, num_classes):
    z_pred = predictor(context_enc(pairs))
    target_codes = target_enc_ema(torch.arange(num_classes, device=device))
    sim = z_pred @ target_codes.T
    return sim.argmax(dim=1).eq(tgts).float().mean().item()

## Load Checkpoint & Data

In [ ]:
# ── Data ────────────────────────────────────────────────────────────────
def generate_data(p, operation='add', seed=SEED):
    """Generate (a, op, b) mod p data for different operations."""
    pairs = torch.cartesian_prod(torch.arange(p), torch.arange(p))
    a, b = pairs[:, 0], pairs[:, 1]

    if operation == 'add':
        targets = (a + b) % p
    elif operation == 'multiply':
        targets = (a * b) % p
    elif operation == 'xor':
        targets = a ^ b  # bitwise XOR (works for integers)
    elif operation == 'polynomial':
        # (a^2 + b^2 + a*b) mod p
        targets = (a**2 + b**2 + a * b) % p
    else:
        raise ValueError(f'Unknown operation: {operation}')

    n = len(pairs)
    n_train = int(train_frac * n)
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    train_idx, val_idx = perm[:n_train], perm[n_train:]
    return (
        pairs.to(device), targets.to(device),
        pairs[train_idx].to(device), targets[train_idx].to(device),
        pairs[val_idx].to(device), targets[val_idx].to(device),
    )

# Load addition data
pairs, targets, train_pairs, train_targets, val_pairs, val_targets = generate_data(p, 'add')
print(f'Addition data: {len(pairs)} pairs, {len(train_pairs)} train, {len(val_pairs)} val')

# ── Load checkpoint ────────────────────────────────────────────────────
CKPT_PATH = '/kaggle/input/datasets/kuriangeorge/jepa-grokking-checkpoint/jepa_grokking_checkpoint.pt'
if not os.path.exists(CKPT_PATH):
    CKPT_PATH = 'jepa_grokking_checkpoint.pt'  # local fallback

checkpoint = torch.load(CKPT_PATH, map_location=device, weights_only=False)

context_enc = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
predictor = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
target_enc_ema = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)

context_enc.load_state_dict(checkpoint['context_enc'])
predictor.load_state_dict(checkpoint['predictor'])
target_enc_ema.load_state_dict(checkpoint['target_enc_ema'])

context_enc.eval()
predictor.eval()
target_enc_ema.eval()
print('Checkpoint loaded.')

---
## Experiment 5: Function Approximation View

### Step 1: Compute class-mean predicted latents
For each residue c ∈ {0, ..., 96}, average z_pred over all validation pairs (a,b) where (a+b)≡c mod 97.

In [ ]:
with torch.no_grad():
    z_pred_all = predictor(context_enc(val_pairs))  # (N_val, 128)
    target_codes = target_enc_ema(torch.arange(p, device=device))  # (97, 128)

# Class-mean predicted latents
z_pred_class_mean = torch.zeros(p, LATENT_DIM, device=device)
z_pred_class_std = torch.zeros(p, device=device)
class_counts = torch.zeros(p, device=device)

for c in range(p):
    mask = val_targets == c
    if mask.sum() > 0:
        z_class = z_pred_all[mask]
        z_pred_class_mean[c] = z_class.mean(dim=0)
        # Spread: mean pairwise cosine distance within class
        z_pred_class_std[c] = 1 - (z_class @ z_class.T).mean().item()
        class_counts[c] = mask.sum().item()

# Normalize class means
z_pred_class_mean_norm = F.normalize(z_pred_class_mean, dim=-1)

# Compare class means to target codes
cos_mean_vs_target = (z_pred_class_mean_norm * target_codes).sum(dim=-1)  # (97,)

print('Class-mean predicted latents vs target codes:')
print(f'  Mean cosine:  {cos_mean_vs_target.mean().item():.4f} ± {cos_mean_vs_target.std().item():.4f}')
print(f'  Min cosine:   {cos_mean_vs_target.min().item():.4f}')
print(f'  All > 0.99:   {(cos_mean_vs_target > 0.99).all().item()}')
print(f'  All > 0.95:   {(cos_mean_vs_target > 0.95).all().item()}')
print(f'\nIntra-class spread (1 - cos, lower = tighter):')
print(f'  Mean: {z_pred_class_std.mean().item():.6f}')
print(f'  Max:  {z_pred_class_std.max().item():.6f}')
print(f'\nSamples per class: min={class_counts.min().item():.0f}, max={class_counts.max().item():.0f}')

### Step 2: Fourier decomposition of class means

The class-mean latent for class c is a vector in R^128. Treating c as an index in Z/97Z,
we decompose each latent dimension as a function of c into Fourier components:

$$v_c^{(d)} = a_0^{(d)} + \sum_{k=1}^{48} \left[ a_k^{(d)} \cos\left(\frac{2\pi k c}{97}\right) + b_k^{(d)} \sin\left(\frac{2\pi k c}{97}\right) \right]$$

**Note:** Since the Fourier basis is complete for Z/97Z, this decomposition is *exact*
with all 48 frequency pairs. The question isn't whether it works — it's how many
components carry the discriminative signal.

In [ ]:
# Work with target codes (what the model is trying to match)
tc_np = target_codes.cpu().numpy()  # (97, 128)
# And class-mean predictions (what the model actually outputs)
zm_np = z_pred_class_mean_norm.cpu().numpy()  # (97, 128)

# ── DFT of target codes ────────────────────────────────────────────────
# FFT along the class axis for each latent dimension
tc_fft = np.fft.fft(tc_np, axis=0)  # (97, 128) complex
tc_power = np.abs(tc_fft) ** 2       # power per (frequency, dimension)

# Average power spectrum across latent dimensions
tc_mean_power = tc_power.mean(axis=1)  # (97,)
tc_mean_power_norm = tc_mean_power / tc_mean_power.sum()

# Same for class-mean predictions
zm_fft = np.fft.fft(zm_np, axis=0)
zm_power = np.abs(zm_fft) ** 2
zm_mean_power = zm_power.mean(axis=1)
zm_mean_power_norm = zm_mean_power / zm_mean_power.sum()

# ── How many frequencies for X% of power? ──────────────────────────────
# Sort frequencies by power (exclude DC)
freq_power = tc_mean_power_norm.copy()
freq_power[0] = 0  # exclude DC
sorted_idx = np.argsort(freq_power)[::-1]
cumpower = np.cumsum(freq_power[sorted_idx])

for threshold in [0.90, 0.95, 0.99]:
    n_needed = np.searchsorted(cumpower, threshold) + 1
    print(f'  Frequencies for {threshold:.0%} power (target codes): {n_needed} / {p//2}')

print(f'\nTop 10 frequencies by power (target codes):')
for i in range(10):
    k = sorted_idx[i]
    print(f'  k={k:3d}: power={freq_power[k]*100:.2f}%, cumulative={cumpower[i]*100:.1f}%')

In [ ]:
# ── Incremental Fourier reconstruction accuracy ─────────────────────────
# Add frequencies one at a time (by power) and measure nearest-code accuracy
# This answers: "how many Fourier components does the model NEED?"

# Build Fourier basis matrix for Z/pZ
c_vals = np.arange(p)  # class indices 0..96
fourier_basis = np.zeros((p, p))  # (97, 97): each column is a basis function
fourier_basis[:, 0] = 1 / np.sqrt(p)  # DC
for k in range(1, (p + 1) // 2):
    fourier_basis[:, 2*k - 1] = np.cos(2 * np.pi * k * c_vals / p) * np.sqrt(2/p)
    fourier_basis[:, 2*k] = np.sin(2 * np.pi * k * c_vals / p) * np.sqrt(2/p)

# Project target codes onto Fourier basis
# tc_np: (97, 128), fourier_basis: (97, 97)
# coefficients: (97, 128) — for each basis function, the projection onto each latent dim
fourier_coeffs = fourier_basis.T @ tc_np  # (97, 128)

# Power per basis function (summed across latent dims)
basis_power = (fourier_coeffs ** 2).sum(axis=1)  # (97,)
basis_power[0] = 0  # exclude DC for ranking
basis_order = np.argsort(basis_power)[::-1]

# Reconstruct with increasing number of basis functions
reconstruction_accs = []
reconstruction_cos = []
n_components_list = list(range(1, min(p, 50) + 1))  # test up to 50 components

for n_comp in n_components_list:
    # Use top-n_comp basis functions (by power)
    selected = basis_order[:n_comp]
    # Also always include DC
    selected_with_dc = np.unique(np.concatenate([[0], selected]))

    # Reconstruct: sum of selected basis functions × their coefficients
    tc_recon = fourier_basis[:, selected_with_dc] @ fourier_coeffs[selected_with_dc]  # (97, 128)

    # Normalize for cosine comparison
    tc_recon_norm = tc_recon / (np.linalg.norm(tc_recon, axis=1, keepdims=True) + 1e-10)

    # Nearest-code accuracy: for each val pair, does argmax(z_pred · recon_code) == true_c?
    z_pred_np = z_pred_all.cpu().numpy()
    sim = z_pred_np @ tc_recon_norm.T  # (N_val, 97)
    pred_classes = sim.argmax(axis=1)
    true_classes = val_targets.cpu().numpy()
    acc = (pred_classes == true_classes).mean()
    reconstruction_accs.append(acc)

    # Mean cosine between reconstructed and original target codes
    cos_recon = (tc_recon_norm * tc_np / (np.linalg.norm(tc_np, axis=1, keepdims=True) + 1e-10)).sum(axis=1).mean()
    reconstruction_cos.append(cos_recon)

# Find minimum components for thresholds
acc_arr = np.array(reconstruction_accs)
for thr in [0.90, 0.95, 0.99, 1.0]:
    mask = acc_arr >= thr
    if mask.any():
        n_min = n_components_list[np.where(mask)[0][0]]
        print(f'  Components for {thr:.0%} accuracy: {n_min}')
    else:
        print(f'  Components for {thr:.0%} accuracy: >{n_components_list[-1]}')

### Step 3: Phase structure analysis

If the Fourier coefficients A_k (vectors in R^128) are related by phase shifts,
that means the target codes have a specific geometric structure:
code_c = Σ_k |A_k| · exp(i·2πkc/p + φ_k) where φ_k is a fixed phase.

We check: are the A_k vectors collinear (same direction, different phase)?
Or do they point in different directions (no phase structure)?

In [ ]:
# For each frequency k, the Fourier coefficient is a pair (cos_coeff, sin_coeff) in R^128
# Compute the amplitude and phase per frequency per latent dimension

# tc_fft is complex (97, 128). For frequency k:
#   amplitude[k, d] = |tc_fft[k, d]|
#   phase[k, d] = angle(tc_fft[k, d])

amplitudes = np.abs(tc_fft)   # (97, 128)
phases = np.angle(tc_fft)      # (97, 128)

# Phase coherence: for each frequency k, how consistent is the phase across dimensions?
# If phase[k, :] is constant → perfect phase coherence (all dims shift together)
# Measure: circular variance of phase across dimensions
phase_coherence = []
for k in range(1, p // 2 + 1):
    ph = phases[k]  # (128,) phases for this frequency
    # Weight by amplitude (ignore low-amplitude dimensions)
    amp = amplitudes[k]
    weights = amp / (amp.sum() + 1e-10)

    # Weighted mean resultant length (circular statistics)
    R = np.abs(np.sum(weights * np.exp(1j * ph)))
    phase_coherence.append(R)

phase_coherence = np.array(phase_coherence)
freqs = np.arange(1, p // 2 + 1)

print('Phase coherence analysis (1.0 = all dims shift together, 0.0 = random phases):')
print(f'  Mean coherence: {phase_coherence.mean():.4f}')
print(f'  Max coherence:  {phase_coherence.max():.4f} (freq k={freqs[phase_coherence.argmax()]})')
print(f'  Min coherence:  {phase_coherence.min():.4f}')
if phase_coherence.mean() > 0.5:
    print('  → HIGH phase coherence: target codes have structured phase relationships')
elif phase_coherence.mean() > 0.2:
    print('  → MODERATE phase coherence: partial phase structure')
else:
    print('  → LOW phase coherence: target codes have no systematic phase structure')
    print('    This means each latent dimension uses frequencies independently')
    print('    → target codes are effectively random-looking, not Fourier-structured')

In [ ]:
# ── Experiment 5: Visualization ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Power spectrum of target codes
ax = axes[0, 0]
ax.bar(np.arange(p // 2 + 1), tc_mean_power_norm[:p // 2 + 1],
       color='steelblue', alpha=0.8, width=1.0)
ax.axhline(y=1/(p-1), color='red', linestyle='--', alpha=0.5, label=f'Uniform ({1/(p-1)*100:.2f}%)')
ax.set_xlabel('Frequency k')
ax.set_ylabel('Normalized Power')
ax.set_title('Target Code Fourier Spectrum', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Panel 2: Reconstruction accuracy vs # components
ax = axes[0, 1]
ax.plot(n_components_list, [a*100 for a in reconstruction_accs], 'b-o', markersize=3)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.3)
for thr in [90, 95, 99]:
    mask = acc_arr >= thr/100
    if mask.any():
        n_min = n_components_list[np.where(mask)[0][0]]
        ax.axvline(x=n_min, color='red' if thr==99 else 'orange', linestyle='--', alpha=0.5,
                   label=f'{thr}% @ {n_min} components')
ax.set_xlabel('Number of Fourier Components')
ax.set_ylabel('Nearest-Code Accuracy (%)')
ax.set_title('Reconstruction Accuracy vs Components', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 3: Phase coherence per frequency
ax = axes[1, 0]
ax.bar(freqs, phase_coherence, color='teal', alpha=0.7, width=1.0)
ax.axhline(y=phase_coherence.mean(), color='red', linestyle='--',
           label=f'Mean: {phase_coherence.mean():.3f}')
ax.set_xlabel('Frequency k')
ax.set_ylabel('Phase Coherence')
ax.set_title('Phase Coherence Across Latent Dimensions', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Panel 4: Class-mean prediction vs target code cosine
ax = axes[1, 1]
cos_plot = cos_mean_vs_target.cpu().numpy()
ax.bar(np.arange(p), cos_plot, color='steelblue', alpha=0.7, width=1.0)
ax.axhline(y=cos_plot.mean(), color='red', linestyle='--',
           label=f'Mean: {cos_plot.mean():.4f}')
ax.set_xlabel('Target class c')
ax.set_ylabel('Cosine(class-mean z_pred, target code)')
ax.set_title('Prediction Alignment Per Class', fontweight='bold')
ax.set_ylim(min(0.8, cos_plot.min() - 0.02), 1.01)
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Experiment 5: Function Approximation View', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('exp5_function_approximation.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ────────────────────────────────────────────────────────────
print('\nExperiment 5 Summary:')
print(f'  Target codes use effective dimension ~{np.searchsorted(cumpower, 0.95)+1} Fourier components for 95% power')
print(f'  Power spectrum is {"FLAT (uniform)" if tc_mean_power_norm[1:p//2+1].std() < 0.005 else "STRUCTURED (non-uniform)"}')
print(f'  Phase coherence: {phase_coherence.mean():.3f} ({"high" if phase_coherence.mean() > 0.5 else "low"})')
print(f'  Interpretation: target codes are {"structured Fourier codes" if phase_coherence.mean() > 0.5 else "near-orthogonal codebook with no Fourier structure"}')
print(f'  The Fourier decomposition is exact (complete basis) but the target codes')
print(f'  themselves have {"" if phase_coherence.mean() > 0.5 else "no "}systematic frequency structure.')

---
## Experiment 6: Causal Intervention

Two complementary tests:

**6a. Fourier band ablation:** Reconstruct target codes using only a subset of frequencies.
Use these degraded codes for nearest-code matching. If low frequencies suffice → the
discriminative signal is in low frequencies. If all frequencies contribute equally → flat codebook.

**6b. Aligned vs orthogonal noise:** Add noise to context latents that is aligned with
the Fourier subspace of (a+b) vs noise orthogonal to it. If aligned noise hurts more →
the model relies on Fourier-structured directions. If equal → the model uses all directions.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6a. Fourier Band Ablation on Target Codes
# ══════════════════════════════════════════════════════════════════════════
print('='*60)
print('Experiment 6a: Fourier Band Ablation')
print('='*60)

z_pred_np = z_pred_all.cpu().numpy()
true_classes = val_targets.cpu().numpy()

# Define frequency bands
bands = {
    'DC only (k=0)':          [0],
    'Low (k=1..5)':           list(range(1, 6)),
    'Low (k=1..10)':          list(range(1, 11)),
    'Mid (k=11..25)':         list(range(11, 26)),
    'High (k=26..48)':        list(range(26, 49)),
    'Low+Mid (k=1..25)':      list(range(1, 26)),
    'All (k=0..48)':          list(range(0, 49)),
    'Top-5 by power':         sorted_idx[:5].tolist(),
    'Top-10 by power':        sorted_idx[:10].tolist(),
    'Top-20 by power':        sorted_idx[:20].tolist(),
    'Remove top-5 by power':  [k for k in range(p) if k not in sorted_idx[:5]],
}

print(f'\n{"Band":>30s} | {"Accuracy":>10s}')
print(f'{"-"*30}-+-{"-"*10}')

band_results = {}
for band_name, freq_list in bands.items():
    # Zero out all frequencies NOT in the list
    tc_fft_filtered = np.zeros_like(tc_fft)
    for k in freq_list:
        if k < p:
            tc_fft_filtered[k] = tc_fft[k]
            # Also include conjugate frequency for real-valued reconstruction
            if k > 0 and k < p:
                tc_fft_filtered[p - k] = tc_fft[p - k]

    # Inverse FFT to get filtered target codes
    tc_filtered = np.fft.ifft(tc_fft_filtered, axis=0).real  # (97, 128)
    tc_filtered_norm = tc_filtered / (np.linalg.norm(tc_filtered, axis=1, keepdims=True) + 1e-10)

    # Nearest-code accuracy with filtered codes
    sim = z_pred_np @ tc_filtered_norm.T
    acc = (sim.argmax(axis=1) == true_classes).mean()
    band_results[band_name] = acc
    print(f'{band_name:>30s} | {acc*100:>9.2f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6b. Aligned vs Orthogonal Noise Injection
# ══════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('Experiment 6b: Aligned vs Orthogonal Noise')
print('='*60)

# Build the Fourier subspace of (a+b) mod p
# For each val pair (a,b), compute Fourier features of the sum s = (a+b) mod p
# These span the "sum-relevant" subspace

with torch.no_grad():
    z_ctx_val = context_enc(val_pairs)  # (N_val, 128)

z_ctx_np = z_ctx_val.cpu().numpy()  # (N_val, 128)
sums = ((val_pairs[:, 0] + val_pairs[:, 1]) % p).cpu().numpy()  # (N_val,)

# Fourier features of the sum: cos(2πks/p), sin(2πks/p) for k=1..48
n_freq = p // 2
fourier_features = np.zeros((len(sums), 2 * n_freq))
for k in range(n_freq):
    freq = k + 1
    fourier_features[:, 2*k] = np.cos(2 * np.pi * freq * sums / p)
    fourier_features[:, 2*k + 1] = np.sin(2 * np.pi * freq * sums / p)

# Find the principal directions that relate z_ctx to Fourier features of the sum
# Regression: z_ctx ≈ fourier_features @ W
# The column space of W.T gives the "Fourier-aligned" subspace in latent space
W_fourier, _, _, _ = np.linalg.lstsq(
    np.hstack([fourier_features, np.ones((len(fourier_features), 1))]),
    z_ctx_np, rcond=None
)  # W: (97, 128)

# Get orthonormal basis for the Fourier-aligned subspace
U_aligned, S_aligned, Vt_aligned = np.linalg.svd(W_fourier[:-1].T, full_matrices=False)
# U_aligned: (128, r) — columns are the Fourier-aligned directions
# Keep directions with significant singular values
sig_threshold = S_aligned.max() * 0.01
n_aligned_dims = (S_aligned > sig_threshold).sum()
P_aligned = U_aligned[:, :n_aligned_dims]  # (128, r) projection onto aligned subspace
print(f'Fourier-aligned subspace dimension: {n_aligned_dims} / {LATENT_DIM}')

# Noise injection experiment
noise_scales = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5]
results_aligned = []
results_orthogonal = []
results_random = []

tc_np_norm = tc_np / (np.linalg.norm(tc_np, axis=1, keepdims=True) + 1e-10)

rng = np.random.RandomState(SEED)

for scale in noise_scales:
    accs_a, accs_o, accs_r = [], [], []

    for trial in range(5):  # average over 5 noise trials
        raw_noise = rng.randn(*z_ctx_np.shape).astype(np.float32)

        # Aligned noise: project onto Fourier subspace
        noise_aligned = raw_noise @ P_aligned @ P_aligned.T
        noise_aligned = noise_aligned / (np.linalg.norm(noise_aligned, axis=1, keepdims=True) + 1e-10)

        # Orthogonal noise: remove Fourier component
        noise_ortho = raw_noise - raw_noise @ P_aligned @ P_aligned.T
        noise_ortho = noise_ortho / (np.linalg.norm(noise_ortho, axis=1, keepdims=True) + 1e-10)

        # Random noise (baseline)
        noise_random = raw_noise / (np.linalg.norm(raw_noise, axis=1, keepdims=True) + 1e-10)

        # Perturb context latents and measure accuracy through predictor
        for noise, acc_list in [(noise_aligned, accs_a), (noise_ortho, accs_o), (noise_random, accs_r)]:
            z_perturbed = z_ctx_np + scale * noise
            z_perturbed_norm = z_perturbed / (np.linalg.norm(z_perturbed, axis=1, keepdims=True) + 1e-10)

            # Pass through predictor
            z_pt = torch.from_numpy(z_perturbed_norm).float().to(device)
            with torch.no_grad():
                z_pred_perturbed = predictor(z_pt).cpu().numpy()

            sim = z_pred_perturbed @ tc_np_norm.T
            acc = (sim.argmax(axis=1) == true_classes).mean()
            acc_list.append(acc)

    results_aligned.append(np.mean(accs_a))
    results_orthogonal.append(np.mean(accs_o))
    results_random.append(np.mean(accs_r))

print(f'\n{"Scale":>6s} | {"Aligned":>10s} | {"Orthogonal":>10s} | {"Random":>10s}')
print(f'{"-"*6}-+-{"-"*10}-+-{"-"*10}-+-{"-"*10}')
for i, scale in enumerate(noise_scales):
    print(f'{scale:>6.2f} | {results_aligned[i]*100:>9.2f}% | {results_orthogonal[i]*100:>9.2f}% | {results_random[i]*100:>9.2f}%')

In [ ]:
# ── Experiment 6: Visualization ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Fourier band ablation
ax = axes[0]
band_names_short = ['DC', 'k1-5', 'k1-10', 'k11-25', 'k26-48', 'k1-25', 'All',
                     'Top5', 'Top10', 'Top20', '-Top5']
band_accs = [band_results[k]*100 for k in bands.keys()]
colors_band = ['gray'] + ['#3b82f6']*2 + ['#f59e0b']*2 + ['#10b981'] + ['#10b981'] + \
              ['#ef4444']*3 + ['#9333ea']
ax.barh(range(len(band_names_short)), band_accs, color=colors_band, alpha=0.8, edgecolor='black')
ax.set_yticks(range(len(band_names_short)))
ax.set_yticklabels(band_names_short, fontsize=8)
ax.set_xlabel('Nearest-Code Accuracy (%)')
ax.set_title('6a: Fourier Band Ablation', fontweight='bold')
ax.axvline(x=100/p, color='gray', linestyle='--', alpha=0.3)
ax.grid(alpha=0.3, axis='x')

# Panel 2: Noise injection
ax = axes[1]
ax.plot(noise_scales, [a*100 for a in results_aligned], 'r-o', linewidth=2, label='Fourier-aligned noise')
ax.plot(noise_scales, [a*100 for a in results_orthogonal], 'b-o', linewidth=2, label='Orthogonal noise')
ax.plot(noise_scales, [a*100 for a in results_random], 'k--o', linewidth=1.5, alpha=0.6, label='Random noise')
ax.axhline(y=100/p, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Noise Scale')
ax.set_ylabel('Accuracy (%)')
ax.set_title('6b: Aligned vs Orthogonal Noise', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 3: Damage ratio (aligned/orthogonal)
ax = axes[2]
damage_aligned = [100 - a*100 for a in results_aligned]
damage_ortho = [100 - a*100 for a in results_orthogonal]
damage_ratio = [a / (o + 1e-6) for a, o in zip(damage_aligned, damage_ortho)]
ax.plot(noise_scales, damage_ratio, 'g-o', linewidth=2)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Equal damage')
ax.set_xlabel('Noise Scale')
ax.set_ylabel('Damage Ratio (aligned / orthogonal)')
ax.set_title('Relative Sensitivity', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Experiment 6: Causal Intervention', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('exp6_causal_intervention.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nExperiment 6 Summary:')
mid_idx = noise_scales.index(0.1)
print(f'  At noise scale 0.1:')
print(f'    Aligned noise:    {results_aligned[mid_idx]*100:.1f}%')
print(f'    Orthogonal noise: {results_orthogonal[mid_idx]*100:.1f}%')
print(f'    Damage ratio:     {damage_ratio[mid_idx]:.2f}x')
if damage_ratio[mid_idx] > 2:
    print(f'  → Fourier-aligned directions are MUCH more sensitive')
    print(f'    The model causally relies on Fourier-structured components')
elif damage_ratio[mid_idx] > 1.2:
    print(f'  → Fourier-aligned directions are moderately more sensitive')
else:
    print(f'  → Similar sensitivity in all directions')
    print(f'    The model uses the full latent space, not just Fourier directions')

---
## Experiment 7: Task Variation

Train the same JEPA architecture on three different tasks:
1. **Modular multiplication** (a·b) mod 97
2. **Bitwise XOR** a ⊕ b (for a, b ∈ {0, ..., 96})
3. **Polynomial** (a² + b² + a·b) mod 97

For each: does grokking occur? How many epochs? What accuracy?

Training budget: 50K epochs per task (shorter than 100K for addition, but
addition grokked at 38K so 50K should be sufficient to detect grokking).

In [ ]:
TASK_EPOCHS = 50_000
TASK_EVAL_EVERY = 500

def train_jepa_on_task(operation, num_classes, epochs=TASK_EPOCHS, eval_every=TASK_EVAL_EVERY):
    """Train JEPA from scratch on a given binary operation."""
    print(f'\nTraining JEPA on: {operation} (mod {p})')
    print(f'  Classes: {num_classes}, Epochs: {epochs}')

    # Generate data
    _, tgts, tr_pairs, tr_tgts, va_pairs, va_tgts = generate_data(p, operation)

    # Check number of unique classes
    actual_classes = tgts.unique().numel()
    print(f'  Unique target values: {actual_classes}')
    if actual_classes != num_classes:
        print(f'  WARNING: expected {num_classes} classes, got {actual_classes}')
        num_classes = actual_classes

    # Initialize fresh model
    ctx = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    tgt = TargetEncoder(num_classes, LATENT_DIM, HIDDEN_DIM).to(device)
    pred = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
    tgt_ema = deepcopy(tgt)
    for param in tgt_ema.parameters():
        param.requires_grad = False

    opt = optim.AdamW(
        list(ctx.parameters()) + list(pred.parameters()) + list(tgt.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )

    loader = DataLoader(TensorDataset(tr_pairs, tr_tgts),
                        batch_size=len(tr_pairs), shuffle=True)

    history = {
        'epoch': [], 'loss': [],
        'train_probe_acc': [], 'val_probe_acc': [],
        'train_nearest_acc': [], 'val_nearest_acc': [],
    }

    start = time.time()
    for epoch in range(epochs):
        ctx.train(); tgt.train(); pred.train()
        for bp, bt in loader:
            opt.zero_grad()
            z_ctx = ctx(bp)
            z_pred = pred(z_ctx)
            with torch.no_grad():
                z_tgt = tgt_ema(bt)
            loss = -(z_pred * z_tgt).sum(dim=-1).mean()
            loss.backward()
            opt.step()
            ema_update(tgt, tgt_ema, EMA_DECAY)

        if epoch % eval_every == 0 or epoch == epochs - 1:
            ctx.eval(); pred.eval()
            tr_probe, va_probe = linear_probe_accuracy(
                ctx, va_pairs, va_tgts, tr_pairs, tr_tgts, num_classes)
            tr_near = nearest_code_accuracy(ctx, pred, tgt_ema, tr_pairs, tr_tgts, num_classes)
            va_near = nearest_code_accuracy(ctx, pred, tgt_ema, va_pairs, va_tgts, num_classes)

            history['epoch'].append(epoch)
            history['loss'].append(loss.item())
            history['train_probe_acc'].append(tr_probe)
            history['val_probe_acc'].append(va_probe)
            history['train_nearest_acc'].append(tr_near)
            history['val_nearest_acc'].append(va_near)

        if epoch % (eval_every * 10) == 0 or epoch == epochs - 1:
            elapsed = (time.time() - start) / 60
            print(f'  Epoch {epoch:6d} [{elapsed:5.1f}m] | '
                  f'Probe: tr={tr_probe*100:.1f}% val={va_probe*100:.1f}% | '
                  f'Nearest: tr={tr_near*100:.1f}% val={va_near*100:.1f}%')

    elapsed = (time.time() - start) / 60
    print(f'  Complete in {elapsed:.1f} min.')

    # Detect grokking
    va_arr = np.array(history['val_probe_acc'])
    grok_mask = va_arr > 0.9
    grok_ep = np.array(history['epoch'])[grok_mask][0] if grok_mask.any() else None

    return {
        'history': history,
        'grok_epoch': grok_ep,
        'final_probe': history['val_probe_acc'][-1],
        'final_nearest': history['val_nearest_acc'][-1],
        'num_classes': num_classes,
    }

In [ ]:
# ── Task 1: Modular Multiplication ─────────────────────────────────────
# Note: (a*b) mod p has p classes but class 0 is special (any a*0 = 0)
# This makes multiplication harder — less uniform class distribution
result_multiply = train_jepa_on_task('multiply', p)

In [ ]:
# ── Task 2: Bitwise XOR ────────────────────────────────────────────────
# XOR on integers 0..96. Number of unique outputs = max possible XOR value + 1
# For 7-bit numbers (96 < 128), XOR can produce values up to 127
xor_vals = set()
for a in range(p):
    for b in range(p):
        xor_vals.add(a ^ b)
n_xor_classes = max(xor_vals) + 1  # need embedding up to max value
print(f'XOR: {len(xor_vals)} unique outputs, max value = {max(xor_vals)}')
print(f'Target encoder needs vocab size = {n_xor_classes}')

# For XOR, target encoder needs larger vocab
# Override with custom training that handles this
result_xor = train_jepa_on_task('xor', n_xor_classes)

In [ ]:
# ── Task 3: Polynomial ─────────────────────────────────────────────────
# (a^2 + b^2 + a*b) mod 97 — still mod p, so p classes
result_poly = train_jepa_on_task('polynomial', p)

In [ ]:
# ── Experiment 7: Visualization ─────────────────────────────────────────
task_results = {
    'Addition\n(a+b) mod p': None,  # from loaded checkpoint, not re-trained
    'Multiplication\n(a·b) mod p': result_multiply,
    'XOR\na ⊕ b': result_xor,
    'Polynomial\n(a²+b²+ab) mod p': result_poly,
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1-3: Learning curves for each new task
task_list = ['Multiplication\n(a·b) mod p', 'XOR\na ⊕ b', 'Polynomial\n(a²+b²+ab) mod p']
results_list = [result_multiply, result_xor, result_poly]
panel_positions = [(0, 0), (0, 1), (1, 0)]

for (task_name, result), (row, col) in zip(zip(task_list, results_list), panel_positions):
    ax = axes[row, col]
    h = result['history']
    ep = np.array(h['epoch'])

    ax.plot(ep, [a*100 for a in h['val_probe_acc']], 'b-', linewidth=1.5, label='Probe (val)')
    ax.plot(ep, [a*100 for a in h['val_nearest_acc']], 'r-', linewidth=1.5, label='Nearest (val)')
    ax.plot(ep, [a*100 for a in h['train_probe_acc']], 'b--', linewidth=1, alpha=0.5, label='Probe (train)')

    chance = 100 / result['num_classes']
    ax.axhline(y=chance, color='gray', linestyle='--', alpha=0.3)

    if result['grok_epoch'] is not None:
        ax.axvline(x=result['grok_epoch'], color='orange', linestyle='--', alpha=0.5,
                   label=f'Grok @ {result["grok_epoch"]}')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(task_name, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

# Panel 4: Summary comparison
ax = axes[1, 1]
all_tasks = ['Addition', 'Multiply', 'XOR', 'Polynomial']
all_probe = [100.0, result_multiply['final_probe']*100,
             result_xor['final_probe']*100, result_poly['final_probe']*100]
all_nearest = [100.0, result_multiply['final_nearest']*100,
               result_xor['final_nearest']*100, result_poly['final_nearest']*100]
all_grok = ['38K',
            f'{result_multiply["grok_epoch"]//1000}K' if result_multiply['grok_epoch'] else 'None',
            f'{result_xor["grok_epoch"]//1000}K' if result_xor['grok_epoch'] else 'None',
            f'{result_poly["grok_epoch"]//1000}K' if result_poly['grok_epoch'] else 'None']

x = np.arange(len(all_tasks))
w = 0.35
ax.bar(x - w/2, all_probe, w, color='steelblue', alpha=0.8, label='Probe acc', edgecolor='black')
ax.bar(x + w/2, all_nearest, w, color='coral', alpha=0.8, label='Nearest acc', edgecolor='black')

# Annotate grokking epoch
for i, g in enumerate(all_grok):
    ax.annotate(f'Grok: {g}', xy=(i, max(all_probe[i], all_nearest[i]) + 2),
                ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(all_tasks)
ax.set_ylabel('Final Accuracy (%)')
ax.set_title('Task Comparison', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.suptitle('Experiment 7: Does JEPA Grok on Other Tasks?', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('exp7_task_variation.png', dpi=150, bbox_inches='tight')
plt.show()

## Combined Summary

In [ ]:
print('='*70)
print('COMBINED SUMMARY: Experiments 5, 6, 7')
print('='*70)

print(f'\n--- Experiment 5: Function Approximation ---')
print(f'  Class-mean prediction alignment: {cos_mean_vs_target.mean().item():.4f}')
print(f'  Target code Fourier spectrum: {"flat (no structure)" if tc_mean_power_norm[1:p//2+1].std() < 0.005 else "non-uniform"}')
print(f'  Phase coherence: {phase_coherence.mean():.3f}')
for thr in [0.90, 0.95, 0.99]:
    mask = acc_arr >= thr
    if mask.any():
        n_min = n_components_list[np.where(mask)[0][0]]
        print(f'  Components for {thr:.0%} reconstruction accuracy: {n_min}')

print(f'\n--- Experiment 6: Causal Intervention ---')
mid_idx = noise_scales.index(0.1)
print(f'  Fourier-aligned subspace: {n_aligned_dims} / {LATENT_DIM} dimensions')
print(f'  At noise=0.1: aligned={results_aligned[mid_idx]*100:.1f}%, '
      f'orthogonal={results_orthogonal[mid_idx]*100:.1f}%, '
      f'ratio={damage_ratio[mid_idx]:.2f}x')
print(f'  Band ablation: top-10 freqs → {band_results["Top-10 by power"]*100:.1f}%, '
      f'all freqs → {band_results["All (k=0..48)"]*100:.1f}%')

print(f'\n--- Experiment 7: Task Variation ---')
print(f'{"Task":>20s} | {"Grok?":>8s} | {"Grok @":>8s} | {"Probe":>7s} | {"Nearest":>7s}')
print(f'{"-"*20}-+-{"-"*8}-+-{"-"*8}-+-{"-"*7}-+-{"-"*7}')
for task_name, result, grok_str in [
    ('Addition', None, '38K'),
    ('Multiplication', result_multiply, None),
    ('XOR', result_xor, None),
    ('Polynomial', result_poly, None),
]:
    if result is None:
        print(f'{task_name:>20s} | {"✓":>8s} | {grok_str:>8s} | {100.0:>6.1f}% | {100.0:>6.1f}%')
    else:
        grokked = '✓' if result['grok_epoch'] else '✗'
        grok_ep = f'{result["grok_epoch"]//1000}K' if result['grok_epoch'] else 'N/A'
        print(f'{task_name:>20s} | {grokked:>8s} | {grok_ep:>8s} | '
              f'{result["final_probe"]*100:>6.1f}% | {result["final_nearest"]*100:>6.1f}%')

# Overall verdict
n_grokked = sum(1 for r in [result_multiply, result_xor, result_poly] if r['grok_epoch'] is not None)
print(f'\n  Tasks that grokked: {n_grokked + 1}/4 (including addition)')
if n_grokked >= 2:
    print(f'  → JEPA grokking generalizes across multiple algebraic tasks!')
    print(f'    This is NOT specific to modular addition.')
elif n_grokked == 1:
    print(f'  → JEPA grokking extends to some but not all tasks.')
    print(f'    Task structure matters.')
else:
    print(f'  → JEPA grokking appears SPECIFIC to modular addition.')
    print(f'    The finding may not generalize.')

print(f'\n{"="*70}')

# Part 7: Internal Fourier Audit

**Description:** Deep audit of every internal component for hidden Fourier structure

---

# Internal Fourier Audit: Where Is the Structure Hiding?

## Motivation

A reviewer raised a valid concern: the existing Fourier analysis checks **outputs**
(target encoder codes, context encoder per-class means, predictor outputs) but not
**internals** (embedding weights, hidden activations, predictor weights, individual neurons).

Nanda et al. (2023) specifically found Fourier structure in:
1. **Embedding weights** $W_E$ — tokens embedded on a circle
2. **Neuron-logit map** $W_L$ — sinusoidal functions of key frequencies
3. **MLP activations** projected onto sinusoids — trigonometric identities
4. **Attention patterns** — consistent with rotation composition

This notebook performs the equivalent analysis on every internal component of
the trained Label-JEPA model. If Fourier circuits exist anywhere in the network,
this audit will find them.

## Components to Audit

### Context Encoder
| Layer | Shape | What to check |
|-------|-------|---------------|
| `emb.weight` | (97, 256) | DFT along token axis — do embeddings lie on a circle? |
| `net.0` (Linear) | (512→256) | Hidden activations per residue class |
| `net.2` (Linear) | (256→256) | Hidden activations per residue class |
| `net.4` (Linear) | (256→128) | Pre-normalization output |

### Target Encoder (EMA)
| Layer | Shape | What to check |
|-------|-------|---------------|
| `emb.weight` | (97, 256) | DFT along token axis |
| `net.0` (Linear) | (256→256) | Hidden activations for residues 0..96 |
| `net.2` (Linear) | (256→128) | Pre-normalization output |

### Predictor
| Layer | Shape | What to check |
|-------|-------|---------------|
| `net.0` (Linear) | (128→64) | Weight matrix Fourier structure |
| `net.2` (Linear) | (64→64) | Bottleneck activations |
| `net.4` (Linear) | (64→128) | Output weight matrix |

## 1. Setup & Model Loading

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from copy import deepcopy

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

p = 97
LATENT_DIM = 128
HIDDEN_DIM = 256
PREDICTOR_DIM = 64

## 2. Architecture (identical to core notebook)

In [ ]:
class ContextEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)
        e = e.view(e.size(0), -1)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class TargetEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class Predictor(nn.Module):
    def __init__(self, latent_dim, predictor_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, latent_dim),
        )

    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)

print("Architecture defined.")

## 3. Load Trained Checkpoint

Load the checkpoint from the core notebook. If not available,
retrain with the same hyperparameters.

In [ ]:
import os
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy
import time

# Generate data (same as core notebook)
pairs = torch.cartesian_prod(torch.arange(p), torch.arange(p))
targets = (pairs[:, 0] + pairs[:, 1]) % p
n = len(pairs)
n_train = int(0.3 * n)
perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:n_train], perm[n_train:]
train_pairs = pairs[train_idx].to(device)
train_targets = targets[train_idx].to(device)
val_pairs = pairs[val_idx].to(device)
val_targets = targets[val_idx].to(device)

# Try loading checkpoint
ckpt_path = "jepa_grokking_checkpoint.pt"
loaded = False

if os.path.exists(ckpt_path):
    try:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        context_enc = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
        target_enc_ema = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
        predictor = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
        context_enc.load_state_dict(ckpt["context_enc"])
        target_enc_ema.load_state_dict(ckpt["target_enc_ema"])
        predictor.load_state_dict(ckpt["predictor"])
        print(f"Loaded checkpoint from {ckpt_path}")
        loaded = True
    except Exception as e:
        print(f"Failed to load checkpoint: {e}")

if not loaded:
    print("No checkpoint found. Training from scratch...")
    print("(This will take ~20-50 minutes depending on hardware)")

    EPOCHS = 100_000
    LR = 1e-3
    WEIGHT_DECAY = 1.0
    EMA_DECAY = 0.996

    torch.manual_seed(SEED)
    context_enc = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    target_enc = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    predictor = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
    target_enc_ema = deepcopy(target_enc)
    for param in target_enc_ema.parameters():
        param.requires_grad = False

# Compile models for faster training
context_enc = torch.compile(context_enc)
target_enc = torch.compile(target_enc)
predictor = torch.compile(predictor)
target_enc_ema = torch.compile(target_enc_ema)

    optimizer = torch.optim.AdamW(
        list(context_enc.parameters()) +
        list(predictor.parameters()) +
        list(target_enc.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY,
    )

    train_loader = DataLoader(
        TensorDataset(train_pairs, train_targets),
        batch_size=n_train, shuffle=True,
    )

    start = time.time()
    for epoch in range(EPOCHS):
        context_enc.train(); target_enc.train(); predictor.train()
        for bp, bt in train_loader:
            optimizer.zero_grad()
            z_ctx = context_enc(bp)
            z_pred = predictor(z_ctx)
            with torch.no_grad():
                z_tgt = target_enc_ema(bt)
            loss = -(z_pred * z_tgt).sum(dim=-1).mean()
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                for o, t in zip(target_enc.parameters(), target_enc_ema.parameters()):
                    t.data.mul_(EMA_DECAY).add_(o.data, alpha=1 - EMA_DECAY)

        if epoch % 10000 == 0:
            context_enc.eval()
            with torch.no_grad():
                z_tr = context_enc(train_pairs)
                z_va = context_enc(val_pairs)
                y_tr = F.one_hot(train_targets, p).float()
                reg = 1e-3
                W = torch.linalg.solve(
                    z_tr.T @ z_tr + reg * torch.eye(LATENT_DIM, device=device),
                    z_tr.T @ y_tr,
                )
                val_acc = (z_va @ W).argmax(1).eq(val_targets).float().mean().item()
            print(f"  Epoch {epoch:6d} ({(time.time()-start)/60:.1f}m) | Val: {val_acc*100:.1f}%")

    torch.save({
        "context_enc": context_enc.state_dict(),
        "target_enc_ema": target_enc_ema.state_dict(),
        "predictor": predictor.state_dict(),
    }, ckpt_path)
    print(f"Training complete. Checkpoint saved.")

context_enc.eval()
target_enc_ema.eval()
predictor.eval()

# Quick validation
with torch.no_grad():
    z_tr = context_enc(train_pairs)
    z_va = context_enc(val_pairs)
    y_tr = F.one_hot(train_targets, p).float()
    W = torch.linalg.solve(
        z_tr.T @ z_tr + 1e-3 * torch.eye(LATENT_DIM, device=device),
        z_tr.T @ y_tr,
    )
    val_acc = (z_va @ W).argmax(1).eq(val_targets).float().mean().item()
print(f"\nModel validation accuracy: {val_acc*100:.1f}%")

## 4. Fourier Analysis Utilities

Reusable functions for computing Fourier structure in any matrix or activation tensor.

In [ ]:
def fourier_spectrum(matrix, axis=0):
    """Compute normalized Fourier power spectrum along given axis.

    Args:
        matrix: numpy array (p, D) or (D, p)
        axis: axis along which tokens are indexed (axis with length p)

    Returns:
        power: normalized power per frequency (length p)
        top5_ratio: fraction of power in top 5 frequencies (excluding DC)
    """
    fft = np.fft.fft(matrix, axis=axis)
    power_per_freq = (np.abs(fft) ** 2).sum(axis=1 - axis)  # sum over non-token axis
    power_per_freq[0] = 0  # remove DC
    total = power_per_freq.sum()
    if total < 1e-12:
        return power_per_freq, 0.0
    normalized = power_per_freq / total
    top5 = np.sort(power_per_freq)[-5:].sum() / total
    return normalized, top5


def plot_spectrum(ax, power, title, p, highlight_top=5):
    """Plot a Fourier spectrum bar chart."""
    half = p // 2 + 1
    ax.bar(range(half), power[:half], color="steelblue", alpha=0.8, width=1.0)
    ax.axhline(y=1/(p-1), color="red", linestyle="--", alpha=0.5, label=f"Uniform ({1/(p-1):.4f})")

    # Highlight top frequencies
    top_idx = np.argsort(power)[-highlight_top:]
    top_idx = top_idx[top_idx < half]
    for idx in top_idx:
        ax.bar(idx, power[idx], color="orange", alpha=0.9, width=1.0)

    top5_ratio = np.sort(power * (power.sum() if power.sum() > 0 else 1))[-5:].sum()
    ax.set_title(f"{title}\n(top-5 = {top5_ratio:.3f})", fontsize=10, fontweight="bold")
    ax.set_xlabel("Frequency k")
    ax.set_ylabel("Normalized Power")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)


def check_circular_embedding(emb_weights, p, name=""):
    """Check if embedding weights have circular (Fourier) structure.

    Like Nanda's analysis of W_E: do tokens embed on a circle in any 2D subspace?
    """
    # PCA to find dominant structure
    pca = PCA(n_components=min(10, emb_weights.shape[1]))
    z_2d = pca.fit_transform(emb_weights)

    # Check if first two PCs form a circle (tokens ordered by index)
    angles = np.arctan2(z_2d[:, 1], z_2d[:, 0])
    # If circular, consecutive tokens should have roughly equal angular spacing
    angle_diffs = np.diff(np.sort(angles))
    angular_uniformity = angle_diffs.std() / (angle_diffs.mean() + 1e-10)

    # Fourier on the raw embeddings
    power, top5 = fourier_spectrum(emb_weights, axis=0)

    return {
        "pca_var_ratio": pca.explained_variance_ratio_,
        "z_2d": z_2d,
        "angles": angles,
        "angular_uniformity": angular_uniformity,
        "fourier_power": power,
        "fourier_top5": top5,
    }


print("Utilities defined.")

---
## Audit 1: Embedding Weights

Nanda's key finding: embedding matrix $W_E$ places tokens on a circle.
If Label-JEPA does the same, the tokens 0..96 should form circular structure
in the principal components of the embedding weights.

We check both the context encoder embedding and the target encoder (EMA) embedding.

In [ ]:
print("=" * 70)
print("AUDIT 1: EMBEDDING WEIGHTS")
print("=" * 70)

# Context encoder embedding
ctx_emb = context_enc.emb.weight.detach().cpu().numpy()  # (97, 256)
ctx_result = check_circular_embedding(ctx_emb, p, "Context Encoder")

# Target encoder embedding
tgt_emb = target_enc_ema.emb.weight.detach().cpu().numpy()  # (97, 256)
tgt_result = check_circular_embedding(tgt_emb, p, "Target Encoder")

print(f"\nContext Encoder Embedding (97 × 256):")
print(f"  Top-5 Fourier concentration: {ctx_result['fourier_top5']:.4f}")
print(f"  Uniform baseline: {5/p:.4f}")
print(f"  PCA var explained (PC1,PC2): {ctx_result['pca_var_ratio'][0]:.3f}, {ctx_result['pca_var_ratio'][1]:.3f}")
print(f"  Angular uniformity (lower=more circular): {ctx_result['angular_uniformity']:.3f}")

print(f"\nTarget Encoder (EMA) Embedding (97 × 256):")
print(f"  Top-5 Fourier concentration: {tgt_result['fourier_top5']:.4f}")
print(f"  Uniform baseline: {5/p:.4f}")
print(f"  PCA var explained (PC1,PC2): {tgt_result['pca_var_ratio'][0]:.3f}, {tgt_result['pca_var_ratio'][1]:.3f}")
print(f"  Angular uniformity (lower=more circular): {tgt_result['angular_uniformity']:.3f}")

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Context encoder: spectrum, PCA, angular distribution
plot_spectrum(axes[0, 0], ctx_result["fourier_power"], "Context Enc: Embedding Spectrum", p)

ax = axes[0, 1]
sc = ax.scatter(ctx_result["z_2d"][:, 0], ctx_result["z_2d"][:, 1],
                c=np.arange(p), cmap="hsv", s=30, edgecolors="black", linewidth=0.3)
ax.set_title(f"Context Enc: Embedding PCA\n(var: {ctx_result['pca_var_ratio'][0]:.1%}, {ctx_result['pca_var_ratio'][1]:.1%})",
             fontsize=10, fontweight="bold")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.colorbar(sc, ax=ax, label="Token index")

ax = axes[0, 2]
ax.bar(range(min(20, len(ctx_result["pca_var_ratio"]))),
       ctx_result["pca_var_ratio"][:20] * 100,
       color="steelblue", alpha=0.8)
ax.set_title("Context Enc: PCA Variance Explained", fontsize=10, fontweight="bold")
ax.set_xlabel("Component"); ax.set_ylabel("Variance (%)")
ax.grid(True, alpha=0.3)

# Target encoder: spectrum, PCA, angular distribution
plot_spectrum(axes[1, 0], tgt_result["fourier_power"], "Target Enc (EMA): Embedding Spectrum", p)

ax = axes[1, 1]
sc = ax.scatter(tgt_result["z_2d"][:, 0], tgt_result["z_2d"][:, 1],
                c=np.arange(p), cmap="hsv", s=30, edgecolors="black", linewidth=0.3)
ax.set_title(f"Target Enc (EMA): Embedding PCA\n(var: {tgt_result['pca_var_ratio'][0]:.1%}, {tgt_result['pca_var_ratio'][1]:.1%})",
             fontsize=10, fontweight="bold")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.colorbar(sc, ax=ax, label="Token index")

ax = axes[1, 2]
ax.bar(range(min(20, len(tgt_result["pca_var_ratio"]))),
       tgt_result["pca_var_ratio"][:20] * 100,
       color="steelblue", alpha=0.8)
ax.set_title("Target Enc (EMA): PCA Variance Explained", fontsize=10, fontweight="bold")
ax.set_xlabel("Component"); ax.set_ylabel("Variance (%)")
ax.grid(True, alpha=0.3)

plt.suptitle("Audit 1: Do Embeddings Have Circular/Fourier Structure?",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("audit1_embeddings.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Audit 2: Hidden Layer Activations (Context Encoder)

For each hidden layer in the context encoder MLP, we:
1. Compute activations for all $97^2$ input pairs
2. Average per residue class $c = (a+b) \bmod p$
3. Apply DFT along the class axis

If Fourier circuits exist, the class-mean activations should show spectral concentration.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 2: HIDDEN LAYER ACTIVATIONS (Context Encoder)")
print("=" * 70)

context_enc.eval()
all_pairs_device = pairs.to(device)
targets_np = targets.numpy()

# Hook into each layer to capture activations
layer_activations = {}

def make_hook(name):
    def hook_fn(module, input, output):
        layer_activations[name] = output.detach()
    return hook_fn

# Register hooks on each layer in the context encoder MLP
hooks = []
hooks.append(context_enc.net[0].register_forward_hook(make_hook("linear1_pre_gelu")))
hooks.append(context_enc.net[1].register_forward_hook(make_hook("gelu1")))
hooks.append(context_enc.net[2].register_forward_hook(make_hook("linear2_pre_gelu")))
hooks.append(context_enc.net[3].register_forward_hook(make_hook("gelu2")))
hooks.append(context_enc.net[4].register_forward_hook(make_hook("linear3_output")))

# Also capture embedding output
def emb_hook(module, input, output):
    layer_activations["embedding"] = output.detach()
hooks.append(context_enc.emb.register_forward_hook(emb_hook))

# Forward pass through all pairs
with torch.no_grad():
    _ = context_enc(all_pairs_device)

# Remove hooks
for h in hooks:
    h.remove()

# Compute per-class mean activations and Fourier structure for each layer
layer_fourier = {}

for layer_name, acts in layer_activations.items():
    acts_np = acts.cpu().numpy()
    if acts_np.ndim == 3:
        # Embedding: (9409, 2, 256) → flatten to (9409, 512)
        acts_np = acts_np.reshape(acts_np.shape[0], -1)

    # Average per residue class
    class_means = np.zeros((p, acts_np.shape[1]))
    for c in range(p):
        mask = targets_np == c
        class_means[c] = acts_np[mask].mean(axis=0)

    # Fourier analysis
    power, top5 = fourier_spectrum(class_means, axis=0)
    layer_fourier[layer_name] = {
        "class_means": class_means,
        "power": power,
        "top5": top5,
        "shape": acts_np.shape,
    }

    print(f"  {layer_name:25s} | shape: {str(acts_np.shape):15s} | Fourier top-5: {top5:.4f}")

# Visualization
layer_names = list(layer_fourier.keys())
n_layers = len(layer_names)
fig, axes = plt.subplots(2, (n_layers + 1) // 2, figsize=(5 * ((n_layers + 1) // 2), 10))
axes = axes.flatten()

for i, name in enumerate(layer_names):
    plot_spectrum(axes[i], layer_fourier[name]["power"],
                  f"Ctx {name}\n(top5={layer_fourier[name]['top5']:.3f})", p)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Audit 2: Context Encoder Hidden Layer Fourier Spectra\n(per-class-mean activations)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("audit2_ctx_hidden.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Audit 3: Hidden Layer Activations (Target Encoder EMA)

Same analysis for the target encoder. Since it only takes single tokens (0..96),
we directly pass each residue and analyze the per-layer activations.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 3: HIDDEN LAYER ACTIVATIONS (Target Encoder EMA)")
print("=" * 70)

tgt_layer_activations = {}

def make_tgt_hook(name):
    def hook_fn(module, input, output):
        tgt_layer_activations[name] = output.detach()
    return hook_fn

tgt_hooks = []
tgt_hooks.append(target_enc_ema.emb.register_forward_hook(make_tgt_hook("tgt_embedding")))
tgt_hooks.append(target_enc_ema.net[0].register_forward_hook(make_tgt_hook("tgt_linear1")))
tgt_hooks.append(target_enc_ema.net[1].register_forward_hook(make_tgt_hook("tgt_gelu")))
tgt_hooks.append(target_enc_ema.net[2].register_forward_hook(make_tgt_hook("tgt_output")))

with torch.no_grad():
    _ = target_enc_ema(torch.arange(p, device=device))

for h in tgt_hooks:
    h.remove()

tgt_layer_fourier = {}
for layer_name, acts in tgt_layer_activations.items():
    acts_np = acts.cpu().numpy()  # (97, D) — already one per residue
    power, top5 = fourier_spectrum(acts_np, axis=0)
    tgt_layer_fourier[layer_name] = {
        "activations": acts_np,
        "power": power,
        "top5": top5,
    }
    print(f"  {layer_name:20s} | shape: {str(acts_np.shape):12s} | Fourier top-5: {top5:.4f}")

# Visualization
tgt_names = list(tgt_layer_fourier.keys())
fig, axes = plt.subplots(1, len(tgt_names), figsize=(5 * len(tgt_names), 5))
if len(tgt_names) == 1:
    axes = [axes]

for i, name in enumerate(tgt_names):
    plot_spectrum(axes[i], tgt_layer_fourier[name]["power"],
                  f"TgtEMA {name}\n(top5={tgt_layer_fourier[name]['top5']:.3f})", p)

plt.suptitle("Audit 3: Target Encoder (EMA) Hidden Layer Fourier Spectra",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("audit3_tgt_hidden.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Audit 4: Predictor Activations

The predictor maps context latents to predicted target latents.
We check whether its internal bottleneck activations show Fourier structure
when averaged per residue class.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 4: PREDICTOR ACTIVATIONS")
print("=" * 70)

pred_layer_activations = {}

def make_pred_hook(name):
    def hook_fn(module, input, output):
        pred_layer_activations[name] = output.detach()
    return hook_fn

pred_hooks = []
pred_hooks.append(predictor.net[0].register_forward_hook(make_pred_hook("pred_linear1")))
pred_hooks.append(predictor.net[1].register_forward_hook(make_pred_hook("pred_gelu1")))
pred_hooks.append(predictor.net[2].register_forward_hook(make_pred_hook("pred_bottleneck")))
pred_hooks.append(predictor.net[3].register_forward_hook(make_pred_hook("pred_gelu2")))
pred_hooks.append(predictor.net[4].register_forward_hook(make_pred_hook("pred_output")))

with torch.no_grad():
    z_ctx_all = context_enc(all_pairs_device)
    _ = predictor(z_ctx_all)

for h in pred_hooks:
    h.remove()

pred_layer_fourier = {}
for layer_name, acts in pred_layer_activations.items():
    acts_np = acts.cpu().numpy()

    # Average per residue class
    class_means = np.zeros((p, acts_np.shape[1]))
    for c in range(p):
        mask = targets_np == c
        class_means[c] = acts_np[mask].mean(axis=0)

    power, top5 = fourier_spectrum(class_means, axis=0)
    pred_layer_fourier[layer_name] = {
        "class_means": class_means,
        "power": power,
        "top5": top5,
    }
    print(f"  {layer_name:20s} | shape: {str(acts_np.shape):15s} | Fourier top-5: {top5:.4f}")

# Visualization
pred_names = list(pred_layer_fourier.keys())
fig, axes = plt.subplots(1, len(pred_names), figsize=(5 * len(pred_names), 5))

for i, name in enumerate(pred_names):
    plot_spectrum(axes[i], pred_layer_fourier[name]["power"],
                  f"Pred {name}\n(top5={pred_layer_fourier[name]['top5']:.3f})", p)

plt.suptitle("Audit 4: Predictor Hidden Layer Fourier Spectra\n(per-class-mean activations)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("audit4_predictor.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Audit 5: Individual Neuron Response Patterns

Nanda found that individual MLP neurons responded as $\cos(\omega_k(a+b))$ or
$\sin(\omega_k(a+b))$ for specific key frequencies $k$. We check whether any
individual neurons in the context encoder MLP show sinusoidal response patterns.

For each neuron, we:
1. Compute its activation for all $97^2$ inputs
2. Average by residue class → get neuron's response as function of $c$
3. Compute the Fourier concentration of that single neuron
4. Check if any neuron has dominant single frequency (sinusoidal response)

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 5: INDIVIDUAL NEURON SINUSOIDAL RESPONSES")
print("=" * 70)

# Use the hidden activations already captured
# Focus on GELU outputs (post-activation) — these are what Nanda analyzed

for layer_name in ["gelu1", "gelu2"]:
    data = layer_fourier[layer_name]
    class_means = data["class_means"]  # (97, D)
    n_neurons = class_means.shape[1]

    # For each neuron, compute Fourier concentration
    neuron_top1 = []
    neuron_top1_freq = []
    neuron_top3 = []

    for d in range(n_neurons):
        neuron_signal = class_means[:, d]
        fft = np.fft.fft(neuron_signal)
        power = np.abs(fft) ** 2
        power[0] = 0  # remove DC
        total = power.sum()
        if total < 1e-12:
            neuron_top1.append(0)
            neuron_top1_freq.append(0)
            neuron_top3.append(0)
            continue

        sorted_power = np.sort(power)
        neuron_top1.append(sorted_power[-1] / total)
        neuron_top1_freq.append(np.argmax(power))
        neuron_top3.append(sorted_power[-3:].sum() / total)

    neuron_top1 = np.array(neuron_top1)
    neuron_top3 = np.array(neuron_top3)
    neuron_top1_freq = np.array(neuron_top1_freq)

    print(f"\n  {layer_name} ({n_neurons} neurons):")
    print(f"    Max single-freq concentration: {neuron_top1.max():.4f} (neuron {neuron_top1.argmax()}, freq={neuron_top1_freq[neuron_top1.argmax()]})")
    print(f"    Mean single-freq concentration: {neuron_top1.mean():.4f}")
    print(f"    Neurons with top1 > 0.3: {(neuron_top1 > 0.3).sum()}/{n_neurons}")
    print(f"    Neurons with top1 > 0.5: {(neuron_top1 > 0.5).sum()}/{n_neurons}")
    print(f"    Neurons with top1 > 0.7: {(neuron_top1 > 0.7).sum()}/{n_neurons}")

    # For comparison: in Nanda's model, key neurons have top1 > 0.8
    # If our model is non-Fourier, we expect no neurons with high concentration

    # Plot distribution and show top neurons
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Distribution of neuron Fourier concentrations
    ax = axes[0]
    ax.hist(neuron_top1, bins=50, color="steelblue", alpha=0.8, edgecolor="black", linewidth=0.5)
    ax.axvline(x=0.3, color="red", linestyle="--", alpha=0.5, label="0.3 threshold")
    ax.axvline(x=0.5, color="red", linestyle="-", alpha=0.5, label="0.5 threshold")
    ax.set_title(f"{layer_name}: Top-1 Frequency Concentration\nper Neuron", fontsize=11, fontweight="bold")
    ax.set_xlabel("Top-1 Freq / Total Power")
    ax.set_ylabel("Count")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Which frequencies are dominant across neurons
    ax = axes[1]
    freq_counts = np.bincount(neuron_top1_freq[neuron_top1 > 0.1], minlength=p)
    ax.bar(range(p // 2 + 1), freq_counts[:p // 2 + 1], color="steelblue", alpha=0.8)
    ax.set_title(f"{layer_name}: Dominant Frequency Distribution\n(neurons with top1 > 0.1)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Frequency k")
    ax.set_ylabel("# Neurons")
    ax.grid(True, alpha=0.3)

    # Show top 5 most sinusoidal neurons
    ax = axes[2]
    top_neurons = np.argsort(neuron_top1)[-5:][::-1]
    for rank, n_idx in enumerate(top_neurons):
        signal = class_means[:, n_idx]
        signal_norm = (signal - signal.mean()) / (signal.std() + 1e-10)
        ax.plot(signal_norm, label=f"Neuron {n_idx} (top1={neuron_top1[n_idx]:.2f}, k={neuron_top1_freq[n_idx]})",
                alpha=0.8)
    ax.set_title(f"{layer_name}: Top 5 Most Sinusoidal Neurons\n(class-mean response)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Residue class c")
    ax.set_ylabel("Normalized activation")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"audit5_{layer_name}_neurons.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## Audit 6: Weight Matrix Fourier Structure

Nanda analyzed the neuron-logit map $W_L$ (the last weight matrix) and found it
was well-approximated by sinusoidal functions of key frequencies. We do the
equivalent analysis on all weight matrices in the context encoder and predictor.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 6: WEIGHT MATRIX FOURIER STRUCTURE")
print("=" * 70)

# Collect all weight matrices
weight_matrices = {}

# Context encoder
weight_matrices["ctx_emb"] = context_enc.emb.weight.detach().cpu().numpy()  # (97, 256)
weight_matrices["ctx_linear1_W"] = context_enc.net[0].weight.detach().cpu().numpy()  # (256, 512)
weight_matrices["ctx_linear2_W"] = context_enc.net[2].weight.detach().cpu().numpy()  # (256, 256)
weight_matrices["ctx_linear3_W"] = context_enc.net[4].weight.detach().cpu().numpy()  # (128, 256)

# Target encoder (EMA)
weight_matrices["tgt_emb"] = target_enc_ema.emb.weight.detach().cpu().numpy()  # (97, 256)
weight_matrices["tgt_linear1_W"] = target_enc_ema.net[0].weight.detach().cpu().numpy()  # (256, 256)
weight_matrices["tgt_linear2_W"] = target_enc_ema.net[2].weight.detach().cpu().numpy()  # (128, 256)

# Predictor
weight_matrices["pred_down_W"] = predictor.net[0].weight.detach().cpu().numpy()  # (64, 128)
weight_matrices["pred_mid_W"] = predictor.net[2].weight.detach().cpu().numpy()  # (64, 64)
weight_matrices["pred_up_W"] = predictor.net[4].weight.detach().cpu().numpy()  # (128, 64)

for name, W in weight_matrices.items():
    # SVD to find dominant structure
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    effective_rank = (S ** 2).sum() ** 2 / (S ** 4).sum()
    top1_ratio = S[0] ** 2 / (S ** 2).sum()

    # For embedding matrices (97 × D), check Fourier along token axis
    if W.shape[0] == p:
        power, top5 = fourier_spectrum(W, axis=0)
        print(f"  {name:20s} | shape: {str(W.shape):12s} | eff_rank: {effective_rank:.1f} | top1_sv: {top1_ratio:.3f} | Fourier top5: {top5:.4f}")
    else:
        # For non-embedding matrices, check Fourier along both axes
        power_r, top5_r = fourier_spectrum(W, axis=0)
        power_c, top5_c = fourier_spectrum(W, axis=1)
        print(f"  {name:20s} | shape: {str(W.shape):12s} | eff_rank: {effective_rank:.1f} | top1_sv: {top1_ratio:.3f} | Fourier rows: {top5_r:.4f}, cols: {top5_c:.4f}")

---
## Audit 7: Nanda-Style Complete Circuit Analysis

Nanda's full analysis: project MLP activations onto $\cos(2\pi k \cdot / p)$ and
$\sin(2\pi k \cdot / p)$ for each frequency $k$, and check whether activations
combine as trigonometric identities like $\cos(\omega(a+b)) = \cos(\omega a)\cos(\omega b) - \sin(\omega a)\sin(\omega b)$.

If the model implements addition via rotation, this projection should yield high $R^2$.
If not, it should be near chance.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 7: TRIGONOMETRIC IDENTITY TEST (Nanda Circuit Analysis)")
print("=" * 70)

# Build Fourier features for inputs a and b
a_vals = pairs[:, 0].numpy()
b_vals = pairs[:, 1].numpy()

freqs_to_test = list(range(1, p // 2 + 1))  # all non-DC frequencies

# For each hidden layer, test: can activations be predicted by
# cos(2πk·a/p), sin(2πk·a/p), cos(2πk·b/p), sin(2πk·b/p) and their products?

from sklearn.linear_model import Ridge

for layer_name in ["gelu1", "gelu2", "linear3_output"]:
    acts = layer_activations[layer_name].cpu().numpy()
    if acts.ndim == 3:
        acts = acts.reshape(acts.shape[0], -1)

    # Test each frequency: how well do trig features of (a,b) at freq k predict activations?
    freq_r2 = []

    for k in freqs_to_test:
        omega = 2 * np.pi * k / p
        # 4 features: cos(ωa), sin(ωa), cos(ωb), sin(ωb)
        # Plus 4 product features: cos(ωa)cos(ωb), cos(ωa)sin(ωb), sin(ωa)cos(ωb), sin(ωa)sin(ωb)
        cos_a = np.cos(omega * a_vals)
        sin_a = np.sin(omega * a_vals)
        cos_b = np.cos(omega * b_vals)
        sin_b = np.sin(omega * b_vals)

        X = np.column_stack([
            cos_a, sin_a, cos_b, sin_b,
            cos_a * cos_b, cos_a * sin_b,
            sin_a * cos_b, sin_a * sin_b,
        ])

        # Fit ridge regression to predict all activations from these trig features
        ridge = Ridge(alpha=1.0)
        ridge.fit(X, acts)
        y_pred = ridge.predict(X)

        # R² per neuron, then average
        ss_res = ((acts - y_pred) ** 2).sum(axis=0)
        ss_tot = ((acts - acts.mean(axis=0)) ** 2).sum(axis=0)
        r2_per_neuron = 1 - ss_res / (ss_tot + 1e-10)
        r2_mean = r2_per_neuron.mean()
        freq_r2.append(r2_mean)

    freq_r2 = np.array(freq_r2)
    best_freq = freqs_to_test[np.argmax(freq_r2)]
    best_r2 = freq_r2.max()
    mean_r2 = freq_r2.mean()

    print(f"\n  {layer_name}:")
    print(f"    Best single-frequency R²: {best_r2:.4f} (k={best_freq})")
    print(f"    Mean R² across all frequencies: {mean_r2:.4f}")
    print(f"    Frequencies with R² > 0.3: {(freq_r2 > 0.3).sum()}/{len(freqs_to_test)}")
    print(f"    Frequencies with R² > 0.5: {(freq_r2 > 0.5).sum()}/{len(freqs_to_test)}")

    # For Nanda's model, key frequencies had R² > 0.8
    # For non-Fourier model, we expect all R² near 0

---
## Audit 7b: Multi-Frequency Combined R²

Even if no single frequency dominates, perhaps several frequencies together
explain the activations (like Nanda's 5 key frequencies). We test progressively
adding frequencies by their individual R² and check combined explanatory power.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 7b: MULTI-FREQUENCY COMBINED R²")
print("=" * 70)

for layer_name in ["gelu2", "linear3_output"]:
    acts = layer_activations[layer_name].cpu().numpy()
    if acts.ndim == 3:
        acts = acts.reshape(acts.shape[0], -1)

    # Compute single-frequency R² for ranking
    single_r2 = []
    for k in freqs_to_test:
        omega = 2 * np.pi * k / p
        cos_a = np.cos(omega * a_vals)
        sin_a = np.sin(omega * a_vals)
        cos_b = np.cos(omega * b_vals)
        sin_b = np.sin(omega * b_vals)
        X = np.column_stack([cos_a, sin_a, cos_b, sin_b,
                             cos_a*cos_b, cos_a*sin_b, sin_a*cos_b, sin_a*sin_b])
        ridge = Ridge(alpha=1.0)
        ridge.fit(X, acts)
        y_pred = ridge.predict(X)
        ss_res = ((acts - y_pred) ** 2).sum()
        ss_tot = ((acts - acts.mean(axis=0)) ** 2).sum()
        single_r2.append(1 - ss_res / (ss_tot + 1e-10))
    single_r2 = np.array(single_r2)

    # Rank frequencies by individual R²
    freq_order = np.argsort(single_r2)[::-1]

    # Incrementally add frequencies and measure combined R²
    combined_r2 = []
    for n_freqs in range(1, min(21, len(freqs_to_test) + 1)):
        selected = freq_order[:n_freqs]
        X_parts = []
        for idx in selected:
            k = freqs_to_test[idx]
            omega = 2 * np.pi * k / p
            cos_a = np.cos(omega * a_vals)
            sin_a = np.sin(omega * a_vals)
            cos_b = np.cos(omega * b_vals)
            sin_b = np.sin(omega * b_vals)
            X_parts.extend([cos_a, sin_a, cos_b, sin_b,
                            cos_a*cos_b, cos_a*sin_b, sin_a*cos_b, sin_a*sin_b])
        X = np.column_stack(X_parts)
        ridge = Ridge(alpha=1.0)
        ridge.fit(X, acts)
        y_pred = ridge.predict(X)
        ss_res = ((acts - y_pred) ** 2).sum()
        ss_tot = ((acts - acts.mean(axis=0)) ** 2).sum()
        combined_r2.append(1 - ss_res / (ss_tot + 1e-10))

    print(f"\n  {layer_name}:")
    for n in [1, 3, 5, 10, 20]:
        if n <= len(combined_r2):
            top_freqs = [freqs_to_test[freq_order[i]] for i in range(n)]
            print(f"    Top {n:2d} freqs: R² = {combined_r2[n-1]:.4f}  (freqs: {top_freqs[:5]}{'...' if n > 5 else ''})")

    # For Nanda: 5 key frequencies → R² > 0.95
    # For non-Fourier: even 20 frequencies should give low R²

---
## Audit 8: Summary Dashboard

In [ ]:
print("\n" + "=" * 70)
print("COMPLETE FOURIER AUDIT SUMMARY")
print("=" * 70)

print("\n--- Embedding Weights ---")
print(f"  Context encoder embedding: Fourier top-5 = {ctx_result['fourier_top5']:.4f} (uniform: {5/p:.4f})")
print(f"  Target encoder embedding:  Fourier top-5 = {tgt_result['fourier_top5']:.4f} (uniform: {5/p:.4f})")

print("\n--- Hidden Layer Activations (per-class mean) ---")
for name, data in layer_fourier.items():
    print(f"  Context {name:25s}: Fourier top-5 = {data['top5']:.4f}")
for name, data in tgt_layer_fourier.items():
    print(f"  Target  {name:25s}: Fourier top-5 = {data['top5']:.4f}")

print("\n--- Predictor Activations (per-class mean) ---")
for name, data in pred_layer_fourier.items():
    print(f"  {name:25s}: Fourier top-5 = {data['top5']:.4f}")

print("\n--- Individual Neuron Sinusoidal Responses ---")
for layer_name in ["gelu1", "gelu2"]:
    data = layer_fourier[layer_name]
    class_means = data["class_means"]
    n_neurons = class_means.shape[1]
    neuron_top1 = []
    for d in range(n_neurons):
        fft = np.fft.fft(class_means[:, d])
        power = np.abs(fft) ** 2
        power[0] = 0
        total = power.sum()
        neuron_top1.append(power.max() / total if total > 1e-12 else 0)
    neuron_top1 = np.array(neuron_top1)
    print(f"  {layer_name}: max neuron top-1 = {neuron_top1.max():.4f}, "
          f"neurons > 0.5 = {(neuron_top1 > 0.5).sum()}/{n_neurons}")

# Verdict
print("\n" + "=" * 70)

all_top5 = []
for data in layer_fourier.values():
    all_top5.append(data["top5"])
for data in tgt_layer_fourier.values():
    all_top5.append(data["top5"])
for data in pred_layer_fourier.values():
    all_top5.append(data["top5"])
all_top5.extend([ctx_result["fourier_top5"], tgt_result["fourier_top5"]])

max_fourier = max(all_top5)
uniform = 5 / p

if max_fourier > 0.5:
    print(f"⚠  FOURIER STRUCTURE DETECTED (max top-5 = {max_fourier:.4f})")
    print(f"   The 'no Fourier circuits' claim needs qualification.")
elif max_fourier > 0.2:
    print(f"⚡ MILD FOURIER CONCENTRATION (max top-5 = {max_fourier:.4f})")
    print(f"   Some spectral structure exists but is not dominant.")
    print(f"   Compare: uniform baseline = {uniform:.4f}")
else:
    print(f"✓  NO FOURIER STRUCTURE FOUND (max top-5 = {max_fourier:.4f})")
    print(f"   Across all layers, embeddings, and individual neurons.")
    print(f"   Uniform baseline = {uniform:.4f}")

print("=" * 70)